<a href="https://colab.research.google.com/github/revak18/neural-dde-echo-project/blob/main/Neural_DDE_Echo_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**After any runtime disconnect**

Rerun:
1.   List item Section 1   Environment, paths, seed
2.   List item Section 2   Data pipeline class definitions


1.   List item Section 3   Dataset construction
2.   List item Section 4   ConvNeXt class definitions








# EF Prediction from EchoNet-Dynamic

This notebook predicts ejection fraction (EF) from heart ultrasound videos. It uses motion to locate the heart, ConvNeXt-Tiny to extract frame features, and a Temporal Transformer to learn changes over time. Models are selected using validation results.

**Imports, device, and reproducibility**

In [ ]:
# ============================================================
# SECTION 1.1 — IMPORTS AND REPRODUCIBILITY
# ============================================================

from pathlib import Path
import os
import sys
import math
import random
import platform

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
)

from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights,
)

import torchvision.transforms.functional as TF

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

def set_seed(seed: int = SEED):
    # Set random seeds used by Python, NumPy, and PyTorch.

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ------------------------------------------------------------
# Compute device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Automatic mixed precision.
USE_AMP = device.type == "cuda"
print("AMP enabled:", USE_AMP)

# ------------------------------------------------------------
# Environment report
# ------------------------------------------------------------
print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("Python executable:", sys.executable)
print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("OpenCV version:", cv2.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    gpu_memory_gb = (torch.cuda.get_device_properties(0).total_memory/ 1024**3)
    print("GPU memory:", f"{gpu_memory_gb:.2f} GB")
else:
    print("CUDA GPU: not available")
print("Random seed:", SEED)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 1.2 — PROJECT AND DATASET PATHS
# ============================================================

# ------------------------------------------------------------
# USER CONFIGURATION
## Only these paths should need to be changed when running the
# notebook on another computer.
# ------------------------------------------------------------

PROJECT_ROOT = Path(
   r"G:\My Drive\IEEE_Projects"
   r"\Reliability_Aware_Neural_DDE_for_EchoNet_Dynamic"
   r"\neural_dde_echo_project"
)

ECHONET_ROOT = Path(
   r"G:\My Drive\IEEE_Projects"
   r"\Reliability_Aware_Neural_DDE_for_EchoNet_Dynamic"
   r"\EchoNet-Dynamic"
)

# from google.colab import drive
# drive.mount("/content/drive")


# PROJECT_ROOT = Path(
#     "/content/drive/MyDrive/IEEE_Projects"
#     "/Reliability_Aware_Neural_DDE_for_EchoNet_Dynamic"
#     "/neural_dde_echo_project"
# )

# ECHONET_ROOT = Path(
#     "/content/drive/MyDrive/IEEE_Projects"
#     "/Reliability_Aware_Neural_DDE_for_EchoNet_Dynamic"
#     "/EchoNet-Dynamic"
# )


# ------------------------------------------------------------
# Derived paths
# Do not manually edit these.
# ------------------------------------------------------------
CSV_FILE = (ECHONET_ROOT/ "FileList.csv")
TRACING_FILE = (ECHONET_ROOT/ "VolumeTracings.csv")
VIDEO_DIR = (ECHONET_ROOT/ "Videos")
CHECKPOINT_DIR = (PROJECT_ROOT/ "checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True,)

# ------------------------------------------------------------
# Experiment checkpoint paths
# ------------------------------------------------------------
FROZEN_BEST_PATH = (CHECKPOINT_DIR/ "convnext_ef_baseline_frozen_best.pth")
FROZEN_LATEST_PATH = (CHECKPOINT_DIR/ "convnext_ef_baseline_frozen_latest.pth")
FINETUNE_BEST_PATH = (CHECKPOINT_DIR/ "convnext_ef_partial_finetune_best.pth")
FINETUNE_LATEST_PATH = (CHECKPOINT_DIR/ "convnext_ef_partial_finetune_latest.pth")
TRANSFORMER_BEST_PATH = (CHECKPOINT_DIR/ "convnext_transformer_ef_best.pth")
TRANSFORMER_LATEST_PATH = (CHECKPOINT_DIR/ "convnext_transformer_ef_latest.pth")

# ------------------------------------------------------------
# Verify required dataset structure
# ------------------------------------------------------------
print("=" * 60)
print("PATH CONFIGURATION")
print("=" * 60)

print("Project root: ", PROJECT_ROOT)
print("EchoNet root: ", ECHONET_ROOT)
print("Required files: ")
print("FileList.csv: ", CSV_FILE.exists())
print("VolumeTracings.csv: ", TRACING_FILE.exists())
print("Videos directory: ", VIDEO_DIR.exists())

# ------------------------------------------------------------
# Fail immediately if required data is missing
# ------------------------------------------------------------

if not CSV_FILE.exists():
    raise FileNotFoundError(f"Could not find FileList.csv:\n{CSV_FILE}")
if not VIDEO_DIR.exists():
    raise FileNotFoundError(f"Could not find Videos directory:\n{VIDEO_DIR}")

# ------------------------------------------------------------
# Count AVI videos
# ------------------------------------------------------------
avi_count = sum(1 for _ in VIDEO_DIR.glob("*.avi"))
print()
print("AVI videos found: ", avi_count)
if avi_count == 0:
    raise RuntimeError(f"No AVI videos found in:\n{VIDEO_DIR}")
print()
print("Checkpoint directory: ", CHECKPOINT_DIR)
print("=" * 60)
print("Configuration verified successfully.")
print("=" * 60)

# 2. Data Preprocessing

Each video is read with OpenCV, then 32 frames are sampled in order. Motion is used to find the heart region, and the same crop is applied to every frame. Training videos also get the same augmentation across frames.

Frames are resized to 224 × 224 and normalized using ImageNet values. The output **tensor** has shape `[32, 3, 224, 224]`.

The heart region is found without using the VolumeTracings annotations.

In [ ]:
# ============================================================
# SECTION 2.1 — UNIFORM FRAME SAMPLING
# ============================================================

class UniformFrameSampler:

    # Uniformly sample frames in chronological order.
    # frames: [N, H, W, C]
    # Returns sampled_frames: [num_frames, H, W, C] and original frame_indices.

    def __init__(self, num_frames=32):
        if num_frames <= 0: raise ValueError("num_frames must be positive.")
        self.num_frames = num_frames

    def __call__(self, frames):
        if not isinstance(frames, np.ndarray): raise TypeError("frames must be a NumPy array.")
        if frames.ndim != 4: raise ValueError("frames must have shape [N, H, W, C].")
        total_frames = len(frames)
        if total_frames == 0: raise ValueError("Video contains no frames.")
        frame_indices = np.linspace(0, total_frames - 1, self.num_frames).round().astype(int)
        sampled_frames = frames[frame_indices]

        return sampled_frames, frame_indices

In [ ]:
# ============================================================
# SECTION 2.2 — FRAME PREPROCESSING
# ============================================================

class EchoPreprocessor:
    # Resize and normalize sampled echocardiography frames.
    # Input frames: [T, H, W, C] RGB uint8
    # Output video: torch.Tensor [T, 3, image_size, image_size]

    def __init__(self, image_size=224):
        self.image_size = image_size
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __call__(self, frames):
        if not isinstance(frames, np.ndarray): raise TypeError("frames must be a NumPy array.")
        if frames.ndim != 4: raise ValueError("frames must have shape [T, H, W, C].")

        processed_frames = []
        for frame in frames:
            frame = cv2.resize(frame, (self.image_size, self.image_size))
            frame = frame.astype(np.float32) / 255.0
            frame = (frame - self.mean) / self.std
            processed_frames.append(frame)

        video = np.stack(processed_frames)
        video = torch.from_numpy(video)
        video = video.permute(0, 3, 1, 2)  # [T,H,W,C] -> [T,C,H,W]
        return video.float()

In [ ]:
# ============================================================
# SECTION 2.3 — VIDEO AUGMENTATION
# ============================================================

class EchoVideoAugmenter:
    # Apply identical random transforms to all frames for temporal consistency.
    # Input/output frames: [T, H, W, C], RGB uint8

    def __init__(self, horizontal_flip_prob=0.5, max_rotation=5.0,
                 brightness_range=(0.9, 1.1), contrast_range=(0.9, 1.1)):
        self.horizontal_flip_prob = horizontal_flip_prob
        self.max_rotation = max_rotation
        self.brightness_range = brightness_range
        self.contrast_range = contrast_range

    def __call__(self, frames):
        if not isinstance(frames, np.ndarray): raise TypeError("frames must be a NumPy array.")
        if frames.ndim != 4: raise ValueError("frames must have shape [T, H, W, C].")

        # Sample one augmentation configuration for the full video.
        do_flip = random.random() < self.horizontal_flip_prob
        angle = random.uniform(-self.max_rotation, self.max_rotation)
        brightness = random.uniform(*self.brightness_range)
        contrast = random.uniform(*self.contrast_range)

        augmented_frames = []
        for frame in frames:
            frame_tensor = torch.from_numpy(frame).permute(2, 0, 1)  # [H,W,C] -> [C,H,W]
            if do_flip: frame_tensor = TF.hflip(frame_tensor)
            frame_tensor = TF.rotate(frame_tensor, angle)
            frame_tensor = TF.adjust_brightness(frame_tensor, brightness)
            frame_tensor = TF.adjust_contrast(frame_tensor, contrast)
            frame_tensor = frame_tensor.clamp(0, 255).to(torch.uint8).permute(1, 2, 0)  # [C,H,W] -> [H,W,C]
            augmented_frames.append(frame_tensor.numpy())

        return np.stack(augmented_frames)

In [ ]:
# ============================================================
# SECTION 2.4 — MOTION-BASED CARDIAC ROI
# ============================================================

class MotionBasedCardiacROI:
    # Detect dominant motion and apply one padded ROI to every frame.
    # Input: frames [T, H, W, 3] RGB. Output: cropped_frames and bbox (x1, y1, x2, y2).

    def __init__(self, padding=0.15, motion_percentile=75):
        self.padding = padding
        self.motion_percentile = motion_percentile

    def __call__(self, frames):
        if not isinstance(frames, np.ndarray): raise TypeError("frames must be a NumPy array.")
        if frames.ndim != 4: raise ValueError("frames must have shape [T, H, W, C].")

        h, w = frames.shape[1:3]
        if len(frames) < 2: return frames, (0, 0, w, h)

        gray_frames = np.array([cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY) for frame in frames])
        differences = [cv2.absdiff(gray_frames[i], gray_frames[i + 1]) for i in range(len(gray_frames) - 1)]
        motion_map = np.mean(differences, axis=0).astype(np.uint8)
        motion_map = cv2.GaussianBlur(motion_map, (5, 5), 0)

        threshold_value = np.percentile(motion_map, self.motion_percentile)
        binary = (motion_map > threshold_value).astype(np.uint8) * 255
        kernel = np.ones((5, 5), dtype=np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours) == 0: return frames, (0, 0, w, h)

        largest = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(largest)
        pad_x, pad_y = int(bw * self.padding), int(bh * self.padding)
        x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
        x2, y2 = min(w, x + bw + pad_x), min(h, y + bh + pad_y)

        if x2 <= x1 or y2 <= y1: return frames, (0, 0, w, h)
        cropped_frames = frames[:, y1:y2, x1:x2, :]
        return cropped_frames, (x1, y1, x2, y2)

In [ ]:
# ============================================================
# SECTION 2.5 — ECHONET-DYNAMIC DATASET
# ============================================================

class EchoNetDynamicDataset(Dataset):
    # Processing: AVI -> frame sampling -> motion ROI -> augmentation -> preprocessing -> tensor.

    def __init__(self, csv_file, video_dir, split, frame_sampler, preprocessor, augmenter=None, roi_processor=None):
        self.video_dir = Path(video_dir)
        self.frame_sampler = frame_sampler
        self.preprocessor = preprocessor
        self.augmenter = augmenter
        self.roi_processor = roi_processor

        metadata = pd.read_csv(csv_file)
        required_columns = {"FileName", "EF", "Split"}
        missing_columns = required_columns - set(metadata.columns)
        if missing_columns: raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

        split = split.upper()
        metadata["Split"] = metadata["Split"].astype(str).str.upper()
        self.metadata = metadata[metadata["Split"] == split].reset_index(drop=True)
        if self.metadata.empty: raise ValueError(f"No samples found for split: {split}")
        self.split = split

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        filename, ef = str(row["FileName"]), float(row["EF"])
        if not np.isfinite(ef) or not 0 <= ef <= 100: raise ValueError(f"Invalid EF value for {filename}: {ef}")

        video_path = self._get_video_path(filename)
        frames = self._read_video(video_path)
        original_num_frames = len(frames)
        sampled_frames, frame_indices = self.frame_sampler(frames)

        if self.roi_processor is not None:
            sampled_frames, roi_bbox = self.roi_processor(sampled_frames)
        else:
            h, w = sampled_frames.shape[1:3]
            roi_bbox = (0, 0, w, h)

        if self.augmenter is not None: sampled_frames = self.augmenter(sampled_frames)
        video = self.preprocessor(sampled_frames)
        if not isinstance(video, torch.Tensor): raise TypeError("Preprocessor must return a torch.Tensor.")

        return {
            "video": video,
            "ef": torch.tensor([ef], dtype=torch.float32),
            "filename": filename,
            "frame_indices": torch.as_tensor(frame_indices, dtype=torch.long),
            "original_num_frames": original_num_frames,
            "roi_bbox": torch.tensor(roi_bbox, dtype=torch.long),
        }

    def _get_video_path(self, filename):
        path = Path(filename)
        video_path = self.video_dir / path.name if path.suffix else self.video_dir / f"{filename}.avi"
        return video_path

    @staticmethod
    def _read_video(video_path):
        capture = cv2.VideoCapture(str(video_path))
        if not capture.isOpened(): raise RuntimeError(f"Could not open video: {video_path}")

        frames = []
        try:
            while True:
                success, frame = capture.read()
                if not success: break
                frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        finally:
            capture.release()

        if not frames: raise RuntimeError(f"No frames were read from: {video_path}")
        return np.stack(frames)

In [ ]:
# ============================================================
# SECTION 2.6 — COMPONENT DEFINITION CHECK
# ============================================================

sampler_test = UniformFrameSampler(num_frames=32)
preprocessor_test = EchoPreprocessor(image_size=224)
augmenter_test = EchoVideoAugmenter()
roi_test = MotionBasedCardiacROI(padding=0.15)

# Synthetic video: 60 RGB frames of size 112 x 112.
synthetic_video = np.random.randint(0, 256, size=(60, 112, 112, 3), dtype=np.uint8)
sampled_frames, frame_indices = sampler_test(synthetic_video)
cropped_frames, bbox = roi_test(sampled_frames)
processed_video = preprocessor_test(cropped_frames)

print("=" * 60)
print("DATA PIPELINE DEFINITION CHECK")
print("=" * 60)
print("Original video:", synthetic_video.shape)
print("Sampled video:", sampled_frames.shape)
print("Number of frame indices:", len(frame_indices))
print("Frame indices ordered:", bool(np.all(np.diff(frame_indices) >= 0)))
print("ROI crop:", cropped_frames.shape)
print("ROI bounding box:", bbox)
print("Processed tensor:", processed_video.shape)
print("Tensor dtype:", processed_video.dtype)

assert sampled_frames.shape[0] == 32
assert len(frame_indices) == 32
assert np.all(np.diff(frame_indices) >= 0)
assert processed_video.shape == (32, 3, 224, 224)
assert processed_video.dtype == torch.float32

print()
print("Section 2 component check passed.")
print("=" * 60)

# 3. Preparing the Datasets

The TRAIN and VAL datasets use the preprocessing defined above. Only TRAIN videos get random augmentation.

The official TEST split is checked only for metadata counts. No TEST dataset is created or used to develop the models.

All experiments use the same **preprocessing** settings.

In [ ]:
# ============================================================
# SECTION 3.1 — DATASET METADATA AND PIPELINE CONFIGURATION
# ============================================================

NUM_FRAMES = 32
IMAGE_SIZE = 224
ROI_PADDING = 0.15

metadata = pd.read_csv(CSV_FILE)
print("=" * 60)
print("ECHONET-DYNAMIC METADATA")
print("=" * 60)
print("Metadata shape:", metadata.shape)
print("\nColumns:")
print(metadata.columns.tolist())

required_columns = {"FileName", "EF", "Split"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns: raise ValueError(f"FileList.csv is missing required columns: {sorted(missing_columns)}")

metadata["Split"] = metadata["Split"].astype(str).str.upper()
split_counts = metadata["Split"].value_counts().sort_index()
print("\nOfficial split counts:")
print(split_counts)

if metadata["EF"].isna().any(): raise ValueError("Missing EF values detected.")
if not metadata["EF"].between(0, 100).all(): raise ValueError("EF values outside [0, 100] detected.")
print("\nEF range:", f"{metadata['EF'].min():.2f}", "to", f"{metadata['EF'].max():.2f}")

expected_split_counts = {"TRAIN": 7465, "VAL": 1288, "TEST": 1277}
for split_name, expected_count in expected_split_counts.items():
    actual_count = int(split_counts.get(split_name, 0))
    if actual_count != expected_count:
        raise RuntimeError(f"{split_name} split contains {actual_count} videos; expected {expected_count}.")

print("\nTotal videos:", len(metadata))
assert len(metadata) == 10030

sampler = UniformFrameSampler(num_frames=NUM_FRAMES)
preprocessor = EchoPreprocessor(image_size=IMAGE_SIZE)
roi_detector = MotionBasedCardiacROI(padding=ROI_PADDING)
train_augmenter = EchoVideoAugmenter()

print("\nPipeline configuration:")
print("Frames per video:", NUM_FRAMES)
print("Output image size:", IMAGE_SIZE)
print("ROI padding:", ROI_PADDING)
print("\nMetadata verification passed.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 3.2 — TRAIN AND VALIDATION DATASETS
# ============================================================

train_dataset = EchoNetDynamicDataset(csv_file=CSV_FILE, video_dir=VIDEO_DIR, split="TRAIN",
                                      frame_sampler=sampler, preprocessor=preprocessor,
                                      augmenter=train_augmenter, roi_processor=roi_detector)
val_dataset = EchoNetDynamicDataset(csv_file=CSV_FILE, video_dir=VIDEO_DIR, split="VAL",
                                    frame_sampler=sampler, preprocessor=preprocessor,
                                    augmenter=None, roi_processor=roi_detector)

print("=" * 60)
print("DATASET CONSTRUCTION")
print("=" * 60)
print("TRAIN videos:", len(train_dataset))
print("VAL videos:", len(val_dataset))

assert len(train_dataset) == 7465
assert len(val_dataset) == 1288

print("\nTRAIN augmentation:", train_dataset.augmenter is not None)
print("VAL augmentation:", val_dataset.augmenter is not None)

assert train_dataset.augmenter is not None
assert val_dataset.augmenter is None

print("\nTRAIN and VAL datasets created successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 3.3 — REAL-VIDEO PIPELINE CHECK
# ============================================================

sample = val_dataset[0]
video, ef = sample["video"], sample["ef"]
frame_indices, roi_bbox = sample["frame_indices"], sample["roi_bbox"]

print("=" * 60)
print("REAL ECHONET VIDEO CHECK")
print("=" * 60)
print("Filename:", sample["filename"])
print("EF:", ef.item())
print("Original number of frames:", sample["original_num_frames"])
print("Sampled frame indices:")
print(frame_indices.tolist())
print("Frame indices ordered:", bool(torch.all(frame_indices[1:] >= frame_indices[:-1])))
print("ROI bounding box:", roi_bbox.tolist())
print("Final video tensor:", video.shape)
print("Tensor dtype:", video.dtype)
print("Contains NaN:", torch.isnan(video).any().item())
print("Contains Inf:", torch.isinf(video).any().item())

assert video.shape == (NUM_FRAMES, 3, IMAGE_SIZE, IMAGE_SIZE)
assert video.dtype == torch.float32
assert len(frame_indices) == NUM_FRAMES
assert torch.all(frame_indices[1:] >= frame_indices[:-1])
assert sample["original_num_frames"] > 0
assert torch.isfinite(video).all()
assert 0 <= ef.item() <= 100

print("\nReal-video pipeline check passed.")
print("=" * 60)

# 4. ConvNeXt Frame Features

A pretrained **ConvNeXt-Tiny** model processes each of the 32 frames separately, producing 768 features per frame.

The batch shape changes from `[B, 32, 3, 224, 224]` to `[B, 32, 768]`, where `B` is the batch size.

In [ ]:
# ============================================================
# SECTION 4.1 — CONVNEXT SPATIAL ENCODER
# ============================================================

class ConvNeXtSpatialEncoder(nn.Module):
    # Convert video [B, T, 3, H, W] to per-frame features [B, T, 768].

    def __init__(self):
        super().__init__()
        backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.norm = backbone.classifier[0]  # Retain LayerNorm; remove classifier.
        self.feature_dim = 768

    def forward(self, video):
        if video.ndim != 5: raise ValueError("Expected video with shape [B, T, 3, H, W].")

        B, T, C, H, W = video.shape
        if C != 3: raise ValueError(f"Expected 3 input channels, got {C}.")

        x = video.reshape(B * T, C, H, W)  # Process frames as independent images.
        x = self.features(x)
        x = self.avgpool(x)
        x = self.norm(x)
        x = torch.flatten(x, 1)
        return x.reshape(B, T, self.feature_dim)

In [ ]:
# ============================================================
# SECTION 4.2 — CONVNEXT EF BASELINE
# ============================================================

class ConvNeXtEFBaseline(nn.Module):
    # Video [B, T, 3, H, W] -> frame features -> temporal mean -> EF [B, 1].

    def __init__(self):
        super().__init__()
        self.spatial_encoder = ConvNeXtSpatialEncoder()
        self.ef_head = nn.Linear(self.spatial_encoder.feature_dim, 1)

    def forward(self, video):
        frame_features = self.spatial_encoder(video)  # [B,T,3,H,W] -> [B,T,768]
        video_features = frame_features.mean(dim=1)  # [B,T,768] -> [B,768]
        return self.ef_head(video_features)  # [B,768] -> [B,1]

In [ ]:
# ============================================================
# SECTION 4.3 — SPATIAL ENCODER CHECK
# ============================================================

spatial_encoder = ConvNeXtSpatialEncoder().to(device)
spatial_encoder.eval()

# Use deterministic VAL sample created in Section 3.
sample = val_dataset[0]
video = sample["video"].unsqueeze(0).to(device)

print("=" * 60)
print("CONVNEXT SPATIAL ENCODER CHECK")
print("=" * 60)
print("Input video:", video.shape)

with torch.no_grad():
    frame_features = spatial_encoder(video)

print("Output features:", frame_features.shape)
print("Feature dtype:", frame_features.dtype)
print("Contains NaN:", torch.isnan(frame_features).any().item())
print("Contains Inf:", torch.isinf(frame_features).any().item())

assert video.shape == (1, NUM_FRAMES, 3, IMAGE_SIZE, IMAGE_SIZE)
assert frame_features.shape == (1, NUM_FRAMES, 768)
assert torch.isfinite(frame_features).all()

print("\nConvNeXt spatial encoder check passed.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 4.4 — EF BASELINE FORWARD CHECK
# ============================================================

baseline_model = ConvNeXtEFBaseline().to(device)
baseline_model.eval()

sample = val_dataset[0]
video = sample["video"].unsqueeze(0).to(device)
true_ef = sample["ef"].unsqueeze(0).to(device)

with torch.no_grad():
    predicted_ef = baseline_model(video)

print("=" * 60)
print("CONVNEXT EF BASELINE CHECK")
print("=" * 60)
print("Input video:", video.shape)
print("True EF shape:", true_ef.shape)
print("Prediction shape:", predicted_ef.shape)
print("True EF:", f"{true_ef.item():.3f}")
print("Untrained predicted EF:", f"{predicted_ef.item():.3f}")
print("Prediction finite:", torch.isfinite(predicted_ef).all().item())

assert predicted_ef.shape == (1, 1)
assert torch.isfinite(predicted_ef).all()

print("\nConvNeXt EF baseline forward check passed.")
print("=" * 60)

# 5. Experiment 1: Frozen ConvNeXt Baseline

The pretrained ConvNeXt-Tiny model is kept **frozen**. It extracts 768 features from each of the 32 frames. These features are averaged across frames, and only a linear layer is trained to predict EF.

Each video gets one EF prediction, giving an output shape of `[B, 1]`.

In [ ]:
# ============================================================
# SECTION 5.1 — EXPERIMENT CONFIGURATION AND DATALOADERS
# ============================================================

# Separate directory for a clean reproducibility run.
E2E_RUN_DIR = CHECKPOINT_DIR / "e2e_reproducibility_run"
E2E_RUN_DIR.mkdir(parents=True, exist_ok=True)
FROZEN_E2E_BEST_PATH = E2E_RUN_DIR / "01_frozen_convnext_best.pth"
FROZEN_E2E_LATEST_PATH = E2E_RUN_DIR / "01_frozen_convnext_latest.pth"

FROZEN_BATCH_SIZE = 16
FROZEN_MAX_EPOCHS = 10
FROZEN_PATIENCE = 3
FROZEN_HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP = device.type == "cuda"

# Dedicated deterministic TRAIN-shuffle generator.
frozen_train_generator = torch.Generator()
frozen_train_generator.manual_seed(SEED)

# num_workers=0 supports Windows, Colab local runtime, and notebooks.
frozen_train_loader = DataLoader(train_dataset, batch_size=FROZEN_BATCH_SIZE, shuffle=True,
                                 num_workers=0, pin_memory=USE_AMP, generator=frozen_train_generator)
frozen_val_loader = DataLoader(val_dataset, batch_size=FROZEN_BATCH_SIZE, shuffle=False,
                               num_workers=0, pin_memory=USE_AMP)

print("=" * 60)
print("FROZEN BASELINE CONFIGURATION")
print("=" * 60)
print("TRAIN videos:", len(train_dataset))
print("VAL videos:", len(val_dataset))
print("Batch size:", FROZEN_BATCH_SIZE)
print("TRAIN batches:", len(frozen_train_loader))
print("VAL batches:", len(frozen_val_loader))
print("Maximum epochs:", FROZEN_MAX_EPOCHS)
print("Early-stopping patience:", FROZEN_PATIENCE)
print("EF-head learning rate:", FROZEN_HEAD_LR)
print("AMP enabled:", USE_AMP)
print("\nCheckpoint directory:")
print(E2E_RUN_DIR)
print("=" * 60)

assert len(frozen_train_loader) == 467
assert len(frozen_val_loader) == 81

In [ ]:
# ============================================================
# SECTION 5.2 — FROZEN BASELINE MODEL
# ============================================================

# Reset random state before trainable-weight initialization.
set_seed(SEED)
frozen_model = ConvNeXtEFBaseline().to(device)

# Freeze ConvNeXt encoder; train only the EF regression head.
for parameter in frozen_model.spatial_encoder.parameters(): parameter.requires_grad = False
for parameter in frozen_model.ef_head.parameters(): parameter.requires_grad = True

total_parameters = sum(p.numel() for p in frozen_model.parameters())
trainable_parameters = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)

print("=" * 60)
print("FROZEN BASELINE MODEL")
print("=" * 60)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Trainable percentage:", f"{100 * trainable_parameters / total_parameters:.4f}%")
print("\nSpatial encoder trainable:", any(p.requires_grad for p in frozen_model.spatial_encoder.parameters()))
print("EF head trainable:", any(p.requires_grad for p in frozen_model.ef_head.parameters()))

assert not any(p.requires_grad for p in frozen_model.spatial_encoder.parameters())
assert all(p.requires_grad for p in frozen_model.ef_head.parameters())

print("\nFrozen baseline model configured successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 5.3 — OPTIMIZER AND AUTO-RESUME
# ============================================================

frozen_criterion = nn.MSELoss()
frozen_optimizer = torch.optim.AdamW(frozen_model.ef_head.parameters(), lr=FROZEN_HEAD_LR, weight_decay=WEIGHT_DECAY)
frozen_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# Default training state.
frozen_start_epoch = 0
frozen_best_val_mae = float("inf")
frozen_epochs_without_improvement = 0
frozen_history = []

# Resume from the latest completed-epoch checkpoint when available.
if FROZEN_E2E_LATEST_PATH.exists():
    print("Existing E2E frozen-baseline checkpoint found.")
    frozen_checkpoint = torch.load(FROZEN_E2E_LATEST_PATH, map_location=device, weights_only=False)
    frozen_model.load_state_dict(frozen_checkpoint["model_state_dict"])
    frozen_optimizer.load_state_dict(frozen_checkpoint["optimizer_state_dict"])
    frozen_scaler.load_state_dict(frozen_checkpoint["scaler_state_dict"])
    frozen_start_epoch = frozen_checkpoint["epoch"]
    frozen_best_val_mae = frozen_checkpoint["best_val_mae"]
    frozen_epochs_without_improvement = frozen_checkpoint["epochs_without_improvement"]
    frozen_history = frozen_checkpoint.get("history", [])

    # Restore RNG states for reproducible continuation when present.
    if "python_rng_state" in frozen_checkpoint:
        random.setstate(frozen_checkpoint["python_rng_state"])
        np.random.set_state(frozen_checkpoint["numpy_rng_state"])
        torch.set_rng_state(frozen_checkpoint["torch_rng_state"].cpu())
        if torch.cuda.is_available() and frozen_checkpoint["cuda_rng_state_all"] is not None:
            torch.cuda.set_rng_state_all([state.cpu() for state in frozen_checkpoint["cuda_rng_state_all"]])
        frozen_train_generator.set_state(frozen_checkpoint["train_generator_state"].cpu())

    print("\nResuming after epoch:", frozen_start_epoch)
    print("Next epoch:", frozen_start_epoch + 1)
    print("Best validation MAE:", frozen_best_val_mae)
else:
    print("No E2E frozen-baseline checkpoint found.")
    print("Training will begin from epoch 1.")

print("\nOptimizer learning rate:", frozen_optimizer.param_groups[0]["lr"])

In [ ]:
# ============================================================
# SECTION 5.4 — BATCH-SIZE FORWARD CHECK
# ============================================================

torch.cuda.empty_cache()
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

frozen_model.eval()
validation_batch = next(iter(frozen_val_loader))
validation_video = validation_batch["video"].to(device, non_blocking=True)
validation_ef = validation_batch["ef"].to(device, non_blocking=True)

with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
        validation_prediction = frozen_model(validation_video)

print("=" * 60)
print("FROZEN BASELINE BATCH CHECK")
print("=" * 60)
print("Video batch:", validation_video.shape)
print("EF batch:", validation_ef.shape)
print("Prediction:", validation_prediction.shape)
print("Predictions finite:", torch.isfinite(validation_prediction).all().item())
if torch.cuda.is_available(): print("Peak VRAM:", f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

assert validation_video.shape[0] <= FROZEN_BATCH_SIZE
assert validation_prediction.shape == validation_ef.shape
assert torch.isfinite(validation_prediction).all()

print("\nFrozen baseline batch check passed.")
print("=" * 60)

del validation_batch, validation_video, validation_ef, validation_prediction
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# SECTION 5.5 — FROZEN CONVNEXT BASELINE TRAINING
# ============================================================

print(f"\n{'=' * 60}\nFROZEN CONVNEXT BASELINE TRAINING\nStarting at epoch {frozen_start_epoch + 1}\n{'=' * 60}")

for epoch in range(frozen_start_epoch, FROZEN_MAX_EPOCHS):
    # Train only EF head; keep frozen ConvNeXt deterministic.
    frozen_model.train()
    frozen_model.spatial_encoder.eval()
    frozen_model.ef_head.train()
    train_squared_error, train_samples = 0.0, 0

    for batch_idx, batch in enumerate(frozen_train_loader):
        video = batch["video"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)
        frozen_optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                frame_features = frozen_model.spatial_encoder(video)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            video_features = frame_features.mean(dim=1)
            predicted_ef = frozen_model.ef_head(video_features)
            loss = frozen_criterion(predicted_ef, true_ef)

        frozen_scaler.scale(loss).backward()
        frozen_scaler.step(frozen_optimizer)
        frozen_scaler.update()

        error = predicted_ef.detach().float() - true_ef.float()
        train_squared_error += error.pow(2).sum().item()
        train_samples += true_ef.numel()

        if (batch_idx + 1) % 200 == 0:
            running_rmse = math.sqrt(train_squared_error / train_samples)
            print(f"Frozen Epoch {epoch + 1} | Batch {batch_idx + 1}/{len(frozen_train_loader)} | Train RMSE: {running_rmse:.3f}")

    train_rmse = math.sqrt(train_squared_error / train_samples)

    # Validation.
    frozen_model.eval()
    val_absolute_error, val_squared_error, val_samples = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in frozen_val_loader:
            video = batch["video"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                predicted_ef = frozen_model(video)

            error = predicted_ef.float() - true_ef.float()
            val_absolute_error += error.abs().sum().item()
            val_squared_error += error.pow(2).sum().item()
            val_samples += true_ef.numel()

    val_mae = val_absolute_error / val_samples
    val_rmse = math.sqrt(val_squared_error / val_samples)
    print(f"\nFrozen Epoch {epoch + 1}/{FROZEN_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")

    improved = val_mae < frozen_best_val_mae
    if improved:
        frozen_best_val_mae = val_mae
        frozen_epochs_without_improvement = 0
    else:
        frozen_epochs_without_improvement += 1

    frozen_history.append({"epoch": epoch + 1, "train_rmse": train_rmse, "val_mae": val_mae, "val_rmse": val_rmse})

    frozen_checkpoint = {
        "epoch": epoch + 1, "stage": "frozen_convnext_baseline", "seed": SEED, "batch_size": FROZEN_BATCH_SIZE,
        "model_state_dict": frozen_model.state_dict(), "optimizer_state_dict": frozen_optimizer.state_dict(),
        "scaler_state_dict": frozen_scaler.state_dict(), "train_rmse": train_rmse, "val_mae": val_mae,
        "val_rmse": val_rmse, "best_val_mae": frozen_best_val_mae,
        "epochs_without_improvement": frozen_epochs_without_improvement, "history": frozen_history,
        "python_rng_state": random.getstate(), "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "train_generator_state": frozen_train_generator.get_state(),
    }

    torch.save(frozen_checkpoint, FROZEN_E2E_LATEST_PATH)
    if improved:
        torch.save(frozen_checkpoint, FROZEN_E2E_BEST_PATH)
        print("Saved new best frozen baseline.")

    print("Best frozen VAL MAE:", f"{frozen_best_val_mae:.3f}")
    print("No-improvement epochs:", frozen_epochs_without_improvement)
    print("-" * 60)
    frozen_start_epoch = epoch + 1

    if frozen_epochs_without_improvement >= FROZEN_PATIENCE:
        print("Early stopping triggered.")
        break

print(f"\n{'=' * 60}\nFROZEN BASELINE TRAINING FINISHED")
print("Completed epochs:", frozen_start_epoch)
print("Best validation MAE:", f"{frozen_best_val_mae:.3f}")
print("=" * 60)

# 6. Experiment 2: Partial ConvNeXt Fine-Tuning

This experiment starts from the best frozen-baseline checkpoint. The earlier ConvNeXt layers stay **frozen**, while the final feature stage, final normalization layer, and EF prediction layer are trained.

Frame features are still averaged before predicting EF.

In [ ]:
# ============================================================
# SECTION 6.1 — PARTIAL FINE-TUNING CONFIGURATION
# ============================================================

FINETUNE_E2E_BEST_PATH = E2E_RUN_DIR / "02_partial_finetune_best.pth"
FINETUNE_E2E_LATEST_PATH = E2E_RUN_DIR / "02_partial_finetune_latest.pth"

# Batch size 2: gradients through ConvNeXt substantially increase VRAM usage.
FINETUNE_BATCH_SIZE = 2
FINETUNE_MAX_EPOCHS = 3
FINETUNE_BACKBONE_LR = 1e-5
FINETUNE_HEAD_LR = 1e-4
FINETUNE_WEIGHT_DECAY = 1e-4

# Dedicated deterministic TRAIN-shuffle generator.
finetune_train_generator = torch.Generator()
finetune_train_generator.manual_seed(SEED)

finetune_train_loader = DataLoader(train_dataset, batch_size=FINETUNE_BATCH_SIZE, shuffle=True,
                                   num_workers=0, pin_memory=USE_AMP, generator=finetune_train_generator)
finetune_val_loader = DataLoader(val_dataset, batch_size=FINETUNE_BATCH_SIZE, shuffle=False,
                                 num_workers=0, pin_memory=USE_AMP)

print("=" * 60)
print("PARTIAL FINE-TUNING CONFIGURATION")
print("=" * 60)
print("Batch size:", FINETUNE_BATCH_SIZE)
print("TRAIN batches:", len(finetune_train_loader))
print("VAL batches:", len(finetune_val_loader))
print("Maximum epochs:", FINETUNE_MAX_EPOCHS)
print("ConvNeXt learning rate:", FINETUNE_BACKBONE_LR)
print("EF-head learning rate:", FINETUNE_HEAD_LR)
print("Frozen baseline checkpoint:", FROZEN_E2E_BEST_PATH.exists())
print("=" * 60)

assert FROZEN_E2E_BEST_PATH.exists()
assert len(finetune_train_loader) == 3733
assert len(finetune_val_loader) == 644

In [ ]:
# ============================================================
# SECTION 6.2 — BUILD PARTIALLY TRAINABLE MODEL
# ============================================================

set_seed(SEED)
finetune_model = ConvNeXtEFBaseline().to(device)

# Load the best frozen-baseline checkpoint.
frozen_best_checkpoint = torch.load(FROZEN_E2E_BEST_PATH, map_location=device, weights_only=False)
finetune_model.load_state_dict(frozen_best_checkpoint["model_state_dict"])
FROZEN_BASELINE_MAE = frozen_best_checkpoint["val_mae"]
FROZEN_BASELINE_RMSE = frozen_best_checkpoint["val_rmse"]

print("Loaded frozen baseline epoch:", frozen_best_checkpoint["epoch"])
print("Frozen baseline VAL MAE:", f"{FROZEN_BASELINE_MAE:.3f}")
print("Frozen baseline VAL RMSE:", f"{FROZEN_BASELINE_RMSE:.3f}")

# Freeze all parameters, then unfreeze the last feature stage, final normalization, and EF head.
for parameter in finetune_model.parameters(): parameter.requires_grad = False
for parameter in finetune_model.spatial_encoder.features[-1].parameters(): parameter.requires_grad = True
for parameter in finetune_model.spatial_encoder.norm.parameters(): parameter.requires_grad = True
for parameter in finetune_model.ef_head.parameters(): parameter.requires_grad = True

total_parameters = sum(p.numel() for p in finetune_model.parameters())
trainable_parameters = sum(p.numel() for p in finetune_model.parameters() if p.requires_grad)

print(f"\n{'=' * 60}\nPARTIAL FINE-TUNING MODEL\n{'=' * 60}")
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Trainable percentage:", f"{100 * trainable_parameters / total_parameters:.2f}%")
print("\nFinal ConvNeXt stage trainable:", any(p.requires_grad for p in finetune_model.spatial_encoder.features[-1].parameters()))
print("Final normalization trainable:", any(p.requires_grad for p in finetune_model.spatial_encoder.norm.parameters()))
print("EF head trainable:", any(p.requires_grad for p in finetune_model.ef_head.parameters()))
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 6.3 — OPTIMIZER AND AUTO-RESUME
# ============================================================

finetune_criterion = nn.MSELoss()
finetune_optimizer = torch.optim.AdamW([
    {"params": finetune_model.spatial_encoder.features[-1].parameters(), "lr": FINETUNE_BACKBONE_LR},
    {"params": finetune_model.spatial_encoder.norm.parameters(), "lr": FINETUNE_BACKBONE_LR},
    {"params": finetune_model.ef_head.parameters(), "lr": FINETUNE_HEAD_LR},
], weight_decay=FINETUNE_WEIGHT_DECAY)
finetune_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# Default experiment state.
finetune_start_epoch = 0
finetune_best_val_mae = float("inf")
finetune_history = []

# Resume only from this E2E fine-tuning experiment.
if FINETUNE_E2E_LATEST_PATH.exists():
    print("Existing E2E fine-tuning checkpoint found.")
    checkpoint = torch.load(FINETUNE_E2E_LATEST_PATH, map_location=device, weights_only=False)
    finetune_model.load_state_dict(checkpoint["model_state_dict"])
    finetune_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    finetune_scaler.load_state_dict(checkpoint["scaler_state_dict"])
    finetune_start_epoch = checkpoint["epoch"]
    finetune_best_val_mae = checkpoint["best_val_mae"]
    finetune_history = checkpoint.get("history", [])

    # Restore RNG states for reproducible continuation.
    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if torch.cuda.is_available():
        torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng_state_all"]])
    finetune_train_generator.set_state(checkpoint["train_generator_state"].cpu())

    print("\nResuming after epoch:", finetune_start_epoch)
    print("Next epoch:", finetune_start_epoch + 1)
    print("Best fine-tune VAL MAE:", f"{finetune_best_val_mae:.3f}")
else:
    print("No E2E fine-tuning checkpoint found.")
    print("Fine-tuning will begin from epoch 1.")

print("\nConvNeXt LR:", finetune_optimizer.param_groups[0]["lr"])
print("Normalization LR:", finetune_optimizer.param_groups[1]["lr"])
print("EF-head LR:", finetune_optimizer.param_groups[2]["lr"])

In [ ]:
# ============================================================
# SECTION 6.4 — PARTIAL CONVNEXT FINE-TUNING
# ============================================================

torch.cuda.empty_cache()
print(f"\n{'=' * 60}\nPARTIAL CONVNEXT FINE-TUNING")
print(f"Starting at epoch {finetune_start_epoch + 1}")
print("Frozen baseline VAL MAE:", f"{FROZEN_BASELINE_MAE:.3f}")
print("=" * 60)

for epoch in range(finetune_start_epoch, FINETUNE_MAX_EPOCHS):
    # Keep frozen encoder sections deterministic; train only final stage, norm, and EF head.
    finetune_model.train()
    finetune_model.spatial_encoder.eval()
    finetune_model.spatial_encoder.features[-1].train()
    finetune_model.spatial_encoder.norm.train()
    finetune_model.ef_head.train()
    train_squared_error, train_samples = 0.0, 0

    for batch_idx, batch in enumerate(finetune_train_loader):
        video = batch["video"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)
        finetune_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            predicted_ef = finetune_model(video)
            loss = finetune_criterion(predicted_ef, true_ef)

        finetune_scaler.scale(loss).backward()
        finetune_scaler.step(finetune_optimizer)
        finetune_scaler.update()

        error = predicted_ef.detach().float() - true_ef.float()
        train_squared_error += error.pow(2).sum().item()
        train_samples += true_ef.numel()

        if (batch_idx + 1) % 250 == 0:
            running_rmse = math.sqrt(train_squared_error / train_samples)
            print(f"Fine-tune Epoch {epoch + 1} | Batch {batch_idx + 1}/{len(finetune_train_loader)} | Train RMSE: {running_rmse:.3f}")

    train_rmse = math.sqrt(train_squared_error / train_samples)

    # Validation.
    finetune_model.eval()
    val_absolute_error, val_squared_error, val_samples = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in finetune_val_loader:
            video = batch["video"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                predicted_ef = finetune_model(video)

            error = predicted_ef.float() - true_ef.float()
            val_absolute_error += error.abs().sum().item()
            val_squared_error += error.pow(2).sum().item()
            val_samples += true_ef.numel()

    val_mae = val_absolute_error / val_samples
    val_rmse = math.sqrt(val_squared_error / val_samples)
    print(f"\nFine-tune Epoch {epoch + 1}/{FINETUNE_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")
    print(f"Frozen baseline MAE: {FROZEN_BASELINE_MAE:.3f}")

    improved = val_mae < finetune_best_val_mae
    if improved: finetune_best_val_mae = val_mae
    finetune_history.append({"epoch": epoch + 1, "train_rmse": train_rmse, "val_mae": val_mae, "val_rmse": val_rmse})

    checkpoint = {
        "epoch": epoch + 1, "stage": "partial_convnext_finetune", "seed": SEED, "batch_size": FINETUNE_BATCH_SIZE,
        "model_state_dict": finetune_model.state_dict(), "optimizer_state_dict": finetune_optimizer.state_dict(),
        "scaler_state_dict": finetune_scaler.state_dict(), "train_rmse": train_rmse, "val_mae": val_mae,
        "val_rmse": val_rmse, "best_val_mae": finetune_best_val_mae,
        "frozen_baseline_mae": FROZEN_BASELINE_MAE, "frozen_baseline_rmse": FROZEN_BASELINE_RMSE,
        "history": finetune_history, "python_rng_state": random.getstate(), "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "train_generator_state": finetune_train_generator.get_state(),
    }

    torch.save(checkpoint, FINETUNE_E2E_LATEST_PATH)
    if improved:
        torch.save(checkpoint, FINETUNE_E2E_BEST_PATH)
        print("Saved new best fine-tuned model.")

    print("Best fine-tune VAL MAE:", f"{finetune_best_val_mae:.3f}")
    print("-" * 60)
    finetune_start_epoch = epoch + 1

print(f"\n{'=' * 60}\nPARTIAL FINE-TUNING FINISHED")
print("Completed epochs:", finetune_start_epoch)
print("Frozen baseline VAL MAE:", f"{FROZEN_BASELINE_MAE:.3f}")
print("Best fine-tuned VAL MAE:", f"{finetune_best_val_mae:.3f}")
print("Absolute MAE improvement:", f"{FROZEN_BASELINE_MAE - finetune_best_val_mae:.3f}")
print("=" * 60)

# 7. Experiment 3: Temporal Transformer

The best ConvNeXt model from Experiment 2 is kept frozen. Instead of averaging its frame features right away, a 2-layer **Transformer** processes them using learned frame-position embeddings.

The Transformer outputs are then averaged and passed to a linear layer to predict EF. Only the position embeddings, Transformer, and EF prediction layer are trained.

In [ ]:
# ============================================================
# SECTION 7.1 — TEMPORAL TRANSFORMER
# ============================================================

class TemporalTransformer(nn.Module):
    # Model frame features [B, T, D] -> contextualized sequence [B, T, D] and video feature [B, D].

    def __init__(self, feature_dim=768, num_frames=32, num_heads=8, num_layers=2,
                 feedforward_dim=1536, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.num_frames = num_frames

        # Learnable temporal positional embeddings.
        self.positional_embedding = nn.Parameter(torch.zeros(1, num_frames, feature_dim))
        nn.init.trunc_normal_(self.positional_embedding, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_dim, nhead=num_heads, dim_feedforward=feedforward_dim,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.final_norm = nn.LayerNorm(feature_dim)

    def forward(self, frame_features):
        if frame_features.ndim != 3: raise ValueError("Expected frame features with shape [B, T, D].")

        B, T, D = frame_features.shape
        if D != self.feature_dim: raise ValueError(f"Expected feature dimension {self.feature_dim}, got {D}.")
        if T > self.num_frames: raise ValueError(f"Received {T} frames but Transformer supports at most {self.num_frames}.")

        x = frame_features + self.positional_embedding[:, :T, :]  # Add temporal positions.
        x = self.transformer(x)  # Temporal self-attention.
        x = self.final_norm(x)
        return x, x.mean(dim=1)  # [B,T,D], [B,D]

In [ ]:
# ============================================================
# SECTION 7.2 — CONVNEXT + TRANSFORMER EF MODEL
# ============================================================

class ConvNeXtTransformerEF(nn.Module):
    # Video [B,T,3,H,W] -> ConvNeXt features -> Transformer -> EF prediction [B,1].

    def __init__(self, spatial_encoder, temporal_transformer, feature_dim=768):
        super().__init__()
        self.spatial_encoder = spatial_encoder
        self.temporal_transformer = temporal_transformer
        self.ef_head = nn.Linear(feature_dim, 1)

    def forward(self, video):
        frame_features = self.spatial_encoder(video)  # [B,T,3,H,W] -> [B,T,768]
        _, video_feature = self.temporal_transformer(frame_features)  # [B,T,768] -> [B,768]
        return self.ef_head(video_feature)  # [B,768] -> [B,1]

In [ ]:
# ============================================================
# SECTION 7.3 — TRANSFORMER EXPERIMENT CONFIGURATION
# ============================================================

TRANSFORMER_E2E_BEST_PATH = E2E_RUN_DIR / "03_transformer_best.pth"
TRANSFORMER_E2E_LATEST_PATH = E2E_RUN_DIR / "03_transformer_latest.pth"

# Transformer architecture and training hyperparameters.
TRANSFORMER_FEATURE_DIM = 768
TRANSFORMER_NUM_HEADS = 8
TRANSFORMER_NUM_LAYERS = 2
TRANSFORMER_FEEDFORWARD_DIM = 1536
TRANSFORMER_DROPOUT = 0.1
TRANSFORMER_BATCH_SIZE = 16
TRANSFORMER_MAX_EPOCHS = 10
TRANSFORMER_PATIENCE = 3
TRANSFORMER_LR = 1e-4
TRANSFORMER_HEAD_LR = 1e-3
TRANSFORMER_WEIGHT_DECAY = 1e-4

# Dedicated deterministic TRAIN-shuffle generator.
transformer_train_generator = torch.Generator()
transformer_train_generator.manual_seed(SEED)

transformer_train_loader = DataLoader(train_dataset, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=True,
                                      num_workers=0, pin_memory=USE_AMP, generator=transformer_train_generator)
transformer_val_loader = DataLoader(val_dataset, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=False,
                                    num_workers=0, pin_memory=USE_AMP)

print("=" * 60)
print("TRANSFORMER CONFIGURATION")
print("=" * 60)
print("Batch size:", TRANSFORMER_BATCH_SIZE)
print("TRAIN batches:", len(transformer_train_loader))
print("VAL batches:", len(transformer_val_loader))
print("Transformer layers:", TRANSFORMER_NUM_LAYERS)
print("Attention heads:", TRANSFORMER_NUM_HEADS)
print("Feed-forward dimension:", TRANSFORMER_FEEDFORWARD_DIM)
print("Maximum epochs:", TRANSFORMER_MAX_EPOCHS)
print("Early-stopping patience:", TRANSFORMER_PATIENCE)
print("Fine-tuned ConvNeXt checkpoint:", FINETUNE_E2E_BEST_PATH.exists())
print("=" * 60)

assert FINETUNE_E2E_BEST_PATH.exists()
assert len(transformer_train_loader) == 467
assert len(transformer_val_loader) == 81

In [ ]:
# ============================================================
# SECTION 7.4 — BUILD TRANSFORMER MODEL
# ============================================================

set_seed(SEED)

# Rebuild and load the best partially fine-tuned ConvNeXt baseline.
best_finetuned_model = ConvNeXtEFBaseline().to(device)
finetune_best_checkpoint = torch.load(FINETUNE_E2E_BEST_PATH, map_location=device, weights_only=False)
best_finetuned_model.load_state_dict(finetune_best_checkpoint["model_state_dict"])
FINETUNED_BASELINE_MAE = finetune_best_checkpoint["val_mae"]
FINETUNED_BASELINE_RMSE = finetune_best_checkpoint["val_rmse"]

print("Loaded fine-tuned epoch:", finetune_best_checkpoint["epoch"])
print("Fine-tuned baseline VAL MAE:", f"{FINETUNED_BASELINE_MAE:.3f}")
print("Fine-tuned baseline VAL RMSE:", f"{FINETUNED_BASELINE_RMSE:.3f}")

# Create a fresh Transformer on top of the best fine-tuned spatial encoder.
temporal_transformer = TemporalTransformer(
    feature_dim=TRANSFORMER_FEATURE_DIM, num_frames=NUM_FRAMES, num_heads=TRANSFORMER_NUM_HEADS,
    num_layers=TRANSFORMER_NUM_LAYERS, feedforward_dim=TRANSFORMER_FEEDFORWARD_DIM, dropout=TRANSFORMER_DROPOUT,
)
transformer_model = ConvNeXtTransformerEF(
    spatial_encoder=best_finetuned_model.spatial_encoder, temporal_transformer=temporal_transformer,
    feature_dim=TRANSFORMER_FEATURE_DIM,
).to(device)

# Freeze ConvNeXt; train Transformer and EF head.
for parameter in transformer_model.spatial_encoder.parameters(): parameter.requires_grad = False
for parameter in transformer_model.temporal_transformer.parameters(): parameter.requires_grad = True
for parameter in transformer_model.ef_head.parameters(): parameter.requires_grad = True

total_parameters = sum(p.numel() for p in transformer_model.parameters())
trainable_parameters = sum(p.numel() for p in transformer_model.parameters() if p.requires_grad)

print(f"\n{'=' * 60}\nTRANSFORMER MODEL\n{'=' * 60}")
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Trainable percentage:", f"{100 * trainable_parameters / total_parameters:.2f}%")
print("\nConvNeXt trainable:", any(p.requires_grad for p in transformer_model.spatial_encoder.parameters()))
print("Transformer trainable:", any(p.requires_grad for p in transformer_model.temporal_transformer.parameters()))
print("EF head trainable:", any(p.requires_grad for p in transformer_model.ef_head.parameters()))

assert not any(p.requires_grad for p in transformer_model.spatial_encoder.parameters())
assert all(p.requires_grad for p in transformer_model.ef_head.parameters())

print("\nTransformer model configured successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 7.5 — TRANSFORMER FORWARD CHECK
# ============================================================

transformer_model.eval()
sample = val_dataset[0]
video = sample["video"].unsqueeze(0).to(device)
true_ef = sample["ef"].unsqueeze(0).to(device)

with torch.no_grad():
    frame_features = transformer_model.spatial_encoder(video)
    sequence_features, video_feature = transformer_model.temporal_transformer(frame_features)
    predicted_ef = transformer_model.ef_head(video_feature)

print("=" * 60)
print("TRANSFORMER FORWARD CHECK")
print("=" * 60)
print("Input video:", video.shape)
print("ConvNeXt features:", frame_features.shape)
print("Transformer sequence:", sequence_features.shape)
print("Video feature:", video_feature.shape)
print("EF prediction:", predicted_ef.shape)
print("Prediction finite:", torch.isfinite(predicted_ef).all().item())
print("Features finite:", torch.isfinite(sequence_features).all().item())

assert frame_features.shape == (1, NUM_FRAMES, 768)
assert sequence_features.shape == (1, NUM_FRAMES, 768)
assert video_feature.shape == (1, 768)
assert predicted_ef.shape == (1, 1)
assert torch.isfinite(predicted_ef).all()

print("\nTransformer forward check passed.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 7.6 — OPTIMIZER AND AUTO-RESUME
# ============================================================

transformer_criterion = nn.MSELoss()
transformer_optimizer = torch.optim.AdamW([
    {"params": transformer_model.temporal_transformer.parameters(), "lr": TRANSFORMER_LR},
    {"params": transformer_model.ef_head.parameters(), "lr": TRANSFORMER_HEAD_LR},
], weight_decay=TRANSFORMER_WEIGHT_DECAY)
transformer_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# Default state.
transformer_start_epoch = 0
transformer_best_val_mae = float("inf")
transformer_epochs_without_improvement = 0
transformer_history = []

# Resume from the latest completed Transformer epoch when available.
if TRANSFORMER_E2E_LATEST_PATH.exists():
    print("Existing E2E Transformer checkpoint found.")
    checkpoint = torch.load(TRANSFORMER_E2E_LATEST_PATH, map_location=device, weights_only=False)
    transformer_model.load_state_dict(checkpoint["model_state_dict"])
    transformer_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    transformer_scaler.load_state_dict(checkpoint["scaler_state_dict"])
    transformer_start_epoch = checkpoint["epoch"]
    transformer_best_val_mae = checkpoint["best_val_mae"]
    transformer_epochs_without_improvement = checkpoint["epochs_without_improvement"]
    transformer_history = checkpoint.get("history", [])

    # Restore RNG states for reproducible continuation.
    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if torch.cuda.is_available():
        torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng_state_all"]])
    transformer_train_generator.set_state(checkpoint["train_generator_state"].cpu())

    print("\nResuming after epoch:", transformer_start_epoch)
    print("Next epoch:", transformer_start_epoch + 1)
    print("Best Transformer VAL MAE:", f"{transformer_best_val_mae:.3f}")
    print("No-improvement epochs:", transformer_epochs_without_improvement)
else:
    print("No E2E Transformer checkpoint found.")
    print("Transformer training will begin from epoch 1.")

print("\nTransformer LR:", transformer_optimizer.param_groups[0]["lr"])
print("EF-head LR:", transformer_optimizer.param_groups[1]["lr"])

In [ ]:
# ============================================================
# SECTION 7.7 — TRANSFORMER TRAINING
# ============================================================

torch.cuda.empty_cache()
print(f"\n{'=' * 60}\nTEMPORAL TRANSFORMER TRAINING")
print(f"Starting at epoch {transformer_start_epoch + 1}")
print("Fine-tuned ConvNeXt baseline MAE:", f"{FINETUNED_BASELINE_MAE:.3f}")
print("=" * 60)

for epoch in range(transformer_start_epoch, TRANSFORMER_MAX_EPOCHS):
    # Keep ConvNeXt frozen; train Transformer and EF head.
    transformer_model.train()
    transformer_model.spatial_encoder.eval()
    transformer_model.temporal_transformer.train()
    transformer_model.ef_head.train()
    train_squared_error, train_samples = 0.0, 0

    for batch_idx, batch in enumerate(transformer_train_loader):
        video = batch["video"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)
        transformer_optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                frame_features = transformer_model.spatial_encoder(video)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            _, video_feature = transformer_model.temporal_transformer(frame_features)
            predicted_ef = transformer_model.ef_head(video_feature)
            loss = transformer_criterion(predicted_ef, true_ef)

        transformer_scaler.scale(loss).backward()
        transformer_scaler.step(transformer_optimizer)
        transformer_scaler.update()

        error = predicted_ef.detach().float() - true_ef.float()
        train_squared_error += error.pow(2).sum().item()
        train_samples += true_ef.numel()

        if (batch_idx + 1) % 200 == 0:
            running_rmse = math.sqrt(train_squared_error / train_samples)
            print(f"Transformer Epoch {epoch + 1} | Batch {batch_idx + 1}/{len(transformer_train_loader)} | Train RMSE: {running_rmse:.3f}")

    train_rmse = math.sqrt(train_squared_error / train_samples)

    # Validation.
    transformer_model.eval()
    val_absolute_error, val_squared_error, val_samples = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in transformer_val_loader:
            video = batch["video"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)

            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                predicted_ef = transformer_model(video)

            error = predicted_ef.float() - true_ef.float()
            val_absolute_error += error.abs().sum().item()
            val_squared_error += error.pow(2).sum().item()
            val_samples += true_ef.numel()

    val_mae = val_absolute_error / val_samples
    val_rmse = math.sqrt(val_squared_error / val_samples)
    print(f"\nTransformer Epoch {epoch + 1}/{TRANSFORMER_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")
    print("Fine-tuned ConvNeXt baseline MAE:", f"{FINETUNED_BASELINE_MAE:.3f}")

    improved = val_mae < transformer_best_val_mae
    if improved:
        transformer_best_val_mae = val_mae
        transformer_epochs_without_improvement = 0
    else:
        transformer_epochs_without_improvement += 1

    transformer_history.append({"epoch": epoch + 1, "train_rmse": train_rmse, "val_mae": val_mae, "val_rmse": val_rmse})
    checkpoint = {
        "epoch": epoch + 1, "stage": "convnext_temporal_transformer", "seed": SEED, "batch_size": TRANSFORMER_BATCH_SIZE,
        "model_state_dict": transformer_model.state_dict(), "optimizer_state_dict": transformer_optimizer.state_dict(),
        "scaler_state_dict": transformer_scaler.state_dict(), "train_rmse": train_rmse, "val_mae": val_mae,
        "val_rmse": val_rmse, "best_val_mae": transformer_best_val_mae,
        "epochs_without_improvement": transformer_epochs_without_improvement,
        "finetuned_baseline_mae": FINETUNED_BASELINE_MAE, "finetuned_baseline_rmse": FINETUNED_BASELINE_RMSE,
        "history": transformer_history, "python_rng_state": random.getstate(), "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "train_generator_state": transformer_train_generator.get_state(),
    }

    torch.save(checkpoint, TRANSFORMER_E2E_LATEST_PATH)
    if improved:
        torch.save(checkpoint, TRANSFORMER_E2E_BEST_PATH)
        print("Saved new best Transformer model.")

    print("Best Transformer VAL MAE:", f"{transformer_best_val_mae:.3f}")
    print("No-improvement epochs:", transformer_epochs_without_improvement)
    print("-" * 60)
    transformer_start_epoch = epoch + 1

    if transformer_epochs_without_improvement >= TRANSFORMER_PATIENCE:
        print("Early stopping triggered.")
        break

print(f"\n{'=' * 60}\nTRANSFORMER TRAINING FINISHED")
print("Completed epochs:", transformer_start_epoch)
print("Fine-tuned baseline VAL MAE:", f"{FINETUNED_BASELINE_MAE:.3f}")
print("Best Transformer VAL MAE:", f"{transformer_best_val_mae:.3f}")
print("Absolute MAE improvement:", f"{FINETUNED_BASELINE_MAE - transformer_best_val_mae:.3f}")
print("=" * 60)

# 8. Results So Far

All three experiments use the same TRAIN and VAL splits and preprocessing.

| Model | Validation MAE |
|---|---:|
| Frozen ConvNeXt + mean pooling | 8.495 |
| Partially fine-tuned ConvNeXt + mean pooling | 7.386 |
| Frozen fine-tuned ConvNeXt + 2-layer Transformer | 6.467 |

Fine-tuning reduced validation error, and adding the **Transformer** reduced it further. The best checkpoint was selected by validation MAE, not the final training epoch.

TEST videos have not been used for model development. The best Transformer model will be used for the next stage, which adds reliability features.

In [ ]:
# ============================================================
# SECTION 8.0 — RESTORE COMPLETED EXPERIMENT STATE
# ============================================================

# Restore checkpoint paths and results after a fresh runtime; no training occurs.
E2E_RUN_DIR = CHECKPOINT_DIR / "e2e_reproducibility_run"

FROZEN_E2E_BEST_PATH = E2E_RUN_DIR / "01_frozen_convnext_best.pth"
FROZEN_E2E_LATEST_PATH = E2E_RUN_DIR / "01_frozen_convnext_latest.pth"
FINETUNE_E2E_BEST_PATH = E2E_RUN_DIR / "02_partial_finetune_best.pth"
FINETUNE_E2E_LATEST_PATH = E2E_RUN_DIR / "02_partial_finetune_latest.pth"
TRANSFORMER_E2E_BEST_PATH = E2E_RUN_DIR / "03_transformer_best.pth"
TRANSFORMER_E2E_LATEST_PATH = E2E_RUN_DIR / "03_transformer_latest.pth"

# Must match the Transformer architecture used during training.
TRANSFORMER_FEATURE_DIM = 768
TRANSFORMER_NUM_HEADS = 8
TRANSFORMER_NUM_LAYERS = 2
TRANSFORMER_FEEDFORWARD_DIM = 1536
TRANSFORMER_DROPOUT = 0.1

print("=" * 60)
print("COMPLETED EXPERIMENT CHECKPOINTS")
print("=" * 60)
print("Frozen ConvNeXt BEST:", FROZEN_E2E_BEST_PATH.exists())
print("Partial fine-tune BEST:", FINETUNE_E2E_BEST_PATH.exists())
print("Transformer BEST:", TRANSFORMER_E2E_BEST_PATH.exists())
print("Transformer LATEST:", TRANSFORMER_E2E_LATEST_PATH.exists())

assert FROZEN_E2E_BEST_PATH.exists(), f"Missing:\n{FROZEN_E2E_BEST_PATH}"
assert FINETUNE_E2E_BEST_PATH.exists(), f"Missing:\n{FINETUNE_E2E_BEST_PATH}"
assert TRANSFORMER_E2E_BEST_PATH.exists(), f"Missing:\n{TRANSFORMER_E2E_BEST_PATH}"

frozen_result = torch.load(FROZEN_E2E_BEST_PATH, map_location="cpu", weights_only=False)
finetune_result = torch.load(FINETUNE_E2E_BEST_PATH, map_location="cpu", weights_only=False)
transformer_result = torch.load(TRANSFORMER_E2E_BEST_PATH, map_location="cpu", weights_only=False)

print("\nCompleted experiment results:")
print("Frozen ConvNeXt MAE:", f"{frozen_result['val_mae']:.3f}")
print("Partial fine-tune MAE:", f"{finetune_result['val_mae']:.3f}")
print("Transformer MAE:", f"{transformer_result['val_mae']:.3f}")
print("Best Transformer epoch:", transformer_result["epoch"])
print("\nCompleted experiment state restored successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 8.1 — VERIFY BEST TRANSFORMER CHECKPOINT
# ============================================================

assert TRANSFORMER_E2E_BEST_PATH.exists(), "Best Transformer checkpoint was not found."
best_transformer_checkpoint = torch.load(TRANSFORMER_E2E_BEST_PATH, map_location=device, weights_only=False)

print("=" * 60)
print("BEST TRANSFORMER CHECKPOINT")
print("=" * 60)
print("Best epoch:", best_transformer_checkpoint["epoch"])
print("Validation MAE:", f"{best_transformer_checkpoint['val_mae']:.3f}")
print("Validation RMSE:", f"{best_transformer_checkpoint['val_rmse']:.3f}")
print("Training RMSE:", f"{best_transformer_checkpoint['train_rmse']:.3f}")
print("Stage:", best_transformer_checkpoint["stage"])
print("Batch size:", best_transformer_checkpoint["batch_size"])
print("=" * 60)



---



---



In [ ]:
# ============================================================
# RECOVER BEST TRANSFORMER CHECKPOINT
# ============================================================

best_transformer_checkpoint = torch.load(
    TRANSFORMER_BEST_PATH,
    map_location=device,
    weights_only=False,
)

print("Best Transformer checkpoint loaded.")
print("Epoch:", best_transformer_checkpoint["epoch"])
print("VAL MAE:", best_transformer_checkpoint["val_mae"])
print("VAL RMSE:", best_transformer_checkpoint["val_rmse"])

In [ ]:
# ============================================================
# CHECK BOTH TRANSFORMER CHECKPOINTS
# ============================================================

for filename in [
    "03_transformer_best.pth",
    "03_transformer_best (1).pth",
]:
    path = (
        PROJECT_ROOT
        / "checkpoints"
        / "e2e_reproducibility_run"
        / filename
    )

    ckpt = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    print("=" * 60)
    print(filename)
    print("Modified path:", path)
    print("Epoch:", ckpt.get("epoch"))
    print("VAL MAE:", ckpt.get("val_mae", ckpt.get("best_val_mae")))
    print("VAL RMSE:", ckpt.get("val_rmse"))

In [ ]:
# ============================================================
# RECOVER CORRECT TRANSFORMER CHECKPOINT
# ============================================================

CURRENT_TRANSFORMER_BEST_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "e2e_reproducibility_run"
    / "03_transformer_best.pth"
)

best_transformer_checkpoint = torch.load(
    CURRENT_TRANSFORMER_BEST_PATH,
    map_location=device,
    weights_only=False,
)

print("Correct Transformer checkpoint loaded.")
print("Epoch:", best_transformer_checkpoint["epoch"])
print("VAL MAE:", best_transformer_checkpoint["val_mae"])
print("VAL RMSE:", best_transformer_checkpoint["val_rmse"])



---



---



In [ ]:
# ============================================================
# SECTION 8.2 — LOAD BEST TRANSFORMER MODEL
# ============================================================

# Reconstruct architecture and restore the best complete model state.
best_spatial_encoder = ConvNeXtSpatialEncoder()
best_temporal_transformer = TemporalTransformer(
    feature_dim=TRANSFORMER_FEATURE_DIM, num_frames=NUM_FRAMES, num_heads=TRANSFORMER_NUM_HEADS,
    num_layers=TRANSFORMER_NUM_LAYERS, feedforward_dim=TRANSFORMER_FEEDFORWARD_DIM, dropout=TRANSFORMER_DROPOUT,
)
best_transformer_model = ConvNeXtTransformerEF(
    spatial_encoder=best_spatial_encoder, temporal_transformer=best_temporal_transformer,
    feature_dim=TRANSFORMER_FEATURE_DIM,
).to(device)

best_transformer_model.load_state_dict(best_transformer_checkpoint["model_state_dict"])
best_transformer_model.eval()
print("Best Transformer model loaded successfully.")

In [ ]:
# ============================================================
# SECTION 8.3 — BEST MODEL FORWARD VERIFICATION
# ============================================================

sample = val_dataset[0]
video = sample["video"].unsqueeze(0).to(device)
true_ef = sample["ef"].item()

with torch.no_grad():
    predicted_ef = best_transformer_model(video)

print("=" * 60)
print("BEST TRANSFORMER FORWARD CHECK")
print("=" * 60)
print("Filename:", sample["filename"])
print("True EF:", f"{true_ef:.3f}")
print("Predicted EF:", f"{predicted_ef.item():.3f}")
print("Absolute error:", f"{abs(predicted_ef.item() - true_ef):.3f}")
print("Prediction shape:", predicted_ef.shape)
print("Prediction finite:", torch.isfinite(predicted_ef).all().item())

assert predicted_ef.shape == (1, 1)
assert torch.isfinite(predicted_ef).all()

print("\nBest Transformer verification passed.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 8.4 — ABLATION SUMMARY
# ============================================================

frozen_result = torch.load(FROZEN_E2E_BEST_PATH, map_location="cpu", weights_only=False)
finetune_result = torch.load(FINETUNE_E2E_BEST_PATH, map_location="cpu", weights_only=False)
transformer_result = torch.load(TRANSFORMER_E2E_BEST_PATH, map_location="cpu", weights_only=False)

ablation_results = pd.DataFrame([
    {"Experiment": "Frozen ConvNeXt", "VAL MAE": frozen_result["val_mae"], "VAL RMSE": frozen_result["val_rmse"]},
    {"Experiment": "Partial ConvNeXt fine-tuning", "VAL MAE": finetune_result["val_mae"], "VAL RMSE": finetune_result["val_rmse"]},
    {"Experiment": "ConvNeXt + Temporal Transformer", "VAL MAE": transformer_result["val_mae"], "VAL RMSE": transformer_result["val_rmse"]},
])

ablation_results["MAE Improvement vs Previous"] = [
    np.nan, frozen_result["val_mae"] - finetune_result["val_mae"],
    finetune_result["val_mae"] - transformer_result["val_mae"],
]

display(ablation_results.round(3))

# 9. Reliability Features

Each sampled frame gets four **reliability** scores:

- Q(t): Image quality, based on contrast and sharpness.
- M(t): How smoothly the learned frame features change over time.
- B(t): How stable the image edges are within the heart region. This is a proxy, not a measurement of the actual left-ventricle boundary.
- C(t): How similar the Transformer features are at points one estimated heartbeat apart.

Together, these form the vector `R(t) = [Q(t), M(t), B(t), C(t)]`.

The scores use no EF labels, TEST data, or VolumeTracings annotations. This module has no trainable parameters and is checked separately before being added to the Neural DDE.

In [ ]:
# ============================================================
# SECTION 9.1 — RELIABILITY FEATURE EXTRACTOR
# ============================================================

class ReliabilityFeatureExtractor(nn.Module):
    # Per-frame reliability [B,T,4]: image quality Q, motion M, boundary B, and cycle consistency C.

    def __init__(self, min_cycle_lag=3, max_cycle_lag=16, eps=1e-6):
        super().__init__()
        self.min_cycle_lag, self.max_cycle_lag, self.eps = min_cycle_lag, max_cycle_lag, eps
        self.register_buffer("imagenet_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1))

        sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
        sobel_y = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]])
        self.register_buffer("sobel_x", sobel_x.view(1, 1, 3, 3))
        self.register_buffer("sobel_y", sobel_y.view(1, 1, 3, 3))

    def _temporal_minmax(self, x):
        x_min, x_max = x.min(dim=1, keepdim=True).values, x.max(dim=1, keepdim=True).values
        return (x - x_min) / (x_max - x_min + self.eps)

    def _to_grayscale(self, video):
        video = (video * self.imagenet_std + self.imagenet_mean).clamp(0.0, 1.0)
        return 0.2989 * video[:, :, 0] + 0.5870 * video[:, :, 1] + 0.1140 * video[:, :, 2]

    def _image_quality(self, gray):
        B, T, H, W = gray.shape
        contrast = gray.flatten(2).std(dim=2)
        frames = gray.reshape(B * T, 1, H, W)
        gx, gy = torch.nn.functional.conv2d(frames, self.sobel_x, padding=1), torch.nn.functional.conv2d(frames, self.sobel_y, padding=1)
        gradient_energy = torch.sqrt(gx.pow(2) + gy.pow(2) + self.eps).mean(dim=(1, 2, 3)).reshape(B, T)
        return 0.5 * self._temporal_minmax(contrast) + 0.5 * self._temporal_minmax(gradient_energy)

    def _motion_consistency(self, features):
        B, T, D = features.shape
        velocity = torch.linalg.vector_norm(features[:, 1:] - features[:, :-1], dim=2) / math.sqrt(D)
        acceleration = torch.zeros(B, T, device=features.device, dtype=features.dtype)
        if T > 2:
            acceleration[:, 2:] = (velocity[:, 1:] - velocity[:, :-1]).abs()
            acceleration[:, 1] = acceleration[:, 2]
        return 1.0 - self._temporal_minmax(acceleration)

    def _boundary_consistency(self, gray):
        B, T, H, W = gray.shape
        frames = gray.reshape(B * T, 1, H, W)
        gx, gy = torch.nn.functional.conv2d(frames, self.sobel_x, padding=1), torch.nn.functional.conv2d(frames, self.sobel_y, padding=1)
        edges = torch.sqrt(gx.pow(2) + gy.pow(2) + self.eps).reshape(B, T, H, W)
        edge_change = torch.zeros(B, T, device=gray.device, dtype=gray.dtype)
        if T > 1:
            temporal_edge_change = (edges[:, 1:] - edges[:, :-1]).abs().mean(dim=(2, 3))
            edge_change[:, 1:], edge_change[:, 0] = temporal_edge_change, temporal_edge_change[:, 0]
        return 1.0 - self._temporal_minmax(edge_change)

    # def _cycle_consistency(self, features):
    #     B, T, D = features.shape
    #     normalized_features = torch.nn.functional.normalize(features, p=2, dim=2)
    #     cycle_consistency = torch.zeros(B, T, device=features.device, dtype=features.dtype)
    #     cycle_lags = torch.zeros(B, device=features.device, dtype=torch.long)
    #     max_lag = min(self.max_cycle_lag, T - 1)

    #     for b in range(B):
    #         best_lag, best_similarity = self.min_cycle_lag, None
    #         for lag in range(self.min_cycle_lag, max_lag + 1):
    #             similarity = (normalized_features[b, lag:] * normalized_features[b, :-lag]).sum(dim=1).mean()
    #             if best_similarity is None or similarity > best_similarity: best_similarity, best_lag = similarity, lag

    #         cycle_lags[b] = best_lag
    #         similarity = (normalized_features[b, best_lag:] * normalized_features[b, :-best_lag]).sum(dim=1)
    #         similarity = (similarity + 1.0) / 2.0
    #         cycle_consistency[b, best_lag:] = similarity
    #         cycle_consistency[b, :best_lag] = similarity.mean()

    #     return cycle_consistency, cycle_lags


    # C(t): cardiac-cycle consistency from latent motion.
    def _cycle_consistency(self, features):
        B, T, D = features.shape
        delta = features[:, 1:, :] - features[:, :-1, :]
        motion = torch.linalg.vector_norm(delta, dim=2) / math.sqrt(D)  # [B,T-1]

        # Standardize motion within each video for autocorrelation.
        motion_centered = motion - motion.mean(dim=1, keepdim=True)
        motion_standardized = motion_centered / (motion_centered.std(dim=1, keepdim=True) + self.eps)
        cycle_consistency = torch.zeros(B, T, device=features.device, dtype=features.dtype)
        cycle_lags = torch.zeros(B, device=features.device, dtype=torch.long)
        max_lag = min(self.max_cycle_lag, motion.shape[1] - 1)

        for b in range(B):
            signal = motion_standardized[b]
            best_lag, best_corr = self.min_cycle_lag, None

            # Find the lag with the highest cardiac-motion autocorrelation.
            for lag in range(self.min_cycle_lag, max_lag + 1):
                correlation = (signal[lag:] * signal[:-lag]).mean()
                if best_corr is None or correlation > best_corr: best_corr, best_lag = correlation, lag

            cycle_lags[b] = best_lag
            raw_difference = (signal[best_lag:] - signal[:-best_lag]).abs()
            consistency = torch.exp(-raw_difference)  # Smaller difference = higher consistency.
            start_index = best_lag + 1  # Motion has T-1 values; restore T-frame alignment.
            cycle_consistency[b, start_index:] = consistency
            initial_value = consistency.mean() if consistency.numel() > 0 else torch.tensor(0.5, device=features.device, dtype=features.dtype)
            cycle_consistency[b, :start_index] = initial_value

        return cycle_consistency, cycle_lags


    def forward(self, video, sequence_features):
        if video.ndim != 5: raise ValueError("video must have shape [B,T,3,H,W].")
        if sequence_features.ndim != 3: raise ValueError("sequence_features must have shape [B,T,D].")
        if video.shape[:2] != sequence_features.shape[:2]: raise ValueError("Video and feature batch/time dimensions must match.")

        gray = self._to_grayscale(video)
        Q = self._image_quality(gray)
        M = self._motion_consistency(sequence_features)
        B = self._boundary_consistency(gray)
        C, cycle_lags = self._cycle_consistency(sequence_features)
        return torch.stack([Q, M, B, C], dim=2).clamp(0.0, 1.0), cycle_lags

In [ ]:
# ============================================================
# SECTION 9.2 — REAL-VIDEO RELIABILITY CHECK
# ============================================================

reliability_extractor = ReliabilityFeatureExtractor(min_cycle_lag=3, max_cycle_lag=16).to(device)
reliability_extractor.eval()

sample = val_dataset[0]
video = sample["video"].unsqueeze(0).to(device)

with torch.no_grad():
    frame_features = best_transformer_model.spatial_encoder(video)
    sequence_features, video_feature = best_transformer_model.temporal_transformer(frame_features)
    reliability, cycle_lags = reliability_extractor(video, sequence_features)

print("=" * 60)
print("RELIABILITY FEATURE CHECK")
print("=" * 60)
print("Video:", video.shape)
print("Transformer sequence:", sequence_features.shape)
print("Reliability representation:", reliability.shape)
print("Estimated cycle lag:", cycle_lags.item(), "sampled frames\n")

reliability_names = ["Q - Image quality", "M - Motion consistency", "B - Boundary consistency proxy", "C - Cycle consistency"]
for i, name in enumerate(reliability_names):
    signal = reliability[0, :, i]
    print(f"{name}:")
    print(f"    min  = {signal.min().item():.3f}")
    print(f"    mean = {signal.mean().item():.3f}")
    print(f"    max  = {signal.max().item():.3f}")

print("\nContains NaN:", torch.isnan(reliability).any().item())
print("Contains Inf:", torch.isinf(reliability).any().item())

assert reliability.shape == (1, NUM_FRAMES, 4)
assert torch.isfinite(reliability).all()
assert reliability.min() >= 0
assert reliability.max() <= 1
assert cycle_lags.item() >= 3
assert cycle_lags.item() <= 16

print("\nReliability feature check passed.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 9.3 — RELIABILITY VISUALIZATION
# ============================================================

reliability_np = reliability[0].detach().cpu().numpy()
frame_number = np.arange(NUM_FRAMES)
fig, ax = plt.subplots(figsize=(12, 5))

for i, name in enumerate(["Q: Image quality", "M: Motion consistency", "B: Boundary consistency", "C: Cycle consistency"]):
    ax.plot(frame_number, reliability_np[:, i], label=name)

ax.set_xlabel("Sampled frame index")
ax.set_ylabel("Relative reliability")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Per-frame Reliability Indicators")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# 10. Checking the Reliability Features

Before adding the features to the Neural DDE, they are checked on several validation videos to see whether:

- Scores vary across frames and videos rather than staying nearly constant.
- Estimated heartbeat intervals are not stuck at the allowed limits.
- The four scores capture different information.

A fixed random seed makes the video selection **repeatable**. No model parameters are updated.

In [ ]:
# ============================================================
# SECTION 10.1 — MULTI-VIDEO RELIABILITY EXTRACTION
# ============================================================

RELIABILITY_CHECK_VIDEOS = 64
RELIABILITY_CHECK_BATCH_SIZE = 8

# Deterministic validation subset.
reliability_rng = np.random.default_rng(SEED)
reliability_indices = reliability_rng.choice(len(val_dataset), size=RELIABILITY_CHECK_VIDEOS, replace=False)
reliability_subset = Subset(val_dataset, reliability_indices.tolist())
reliability_loader = DataLoader(reliability_subset, batch_size=RELIABILITY_CHECK_BATCH_SIZE, shuffle=False,
                                num_workers=0, pin_memory=USE_AMP)

best_transformer_model.eval()
reliability_extractor.eval()
all_reliability, all_cycle_lags = [], []

with torch.no_grad():
    for batch_idx, batch in enumerate(reliability_loader):
        video = batch["video"].to(device, non_blocking=True)
        frame_features = best_transformer_model.spatial_encoder(video)
        sequence_features, _ = best_transformer_model.temporal_transformer(frame_features)
        reliability_batch, cycle_lags_batch = reliability_extractor(video, sequence_features)
        all_reliability.append(reliability_batch.cpu())
        all_cycle_lags.append(cycle_lags_batch.cpu())
        print(f"Processed {min((batch_idx + 1) * RELIABILITY_CHECK_BATCH_SIZE, RELIABILITY_CHECK_VIDEOS)}/{RELIABILITY_CHECK_VIDEOS} videos")

all_reliability = torch.cat(all_reliability, dim=0)
all_cycle_lags = torch.cat(all_cycle_lags, dim=0)

print(f"\n{'=' * 60}\nMULTI-VIDEO RELIABILITY EXTRACTION\n{'=' * 60}")
print("Reliability tensor:", all_reliability.shape)
print("Cycle-lag tensor:", all_cycle_lags.shape)
print("Contains NaN:", torch.isnan(all_reliability).any().item())
print("Contains Inf:", torch.isinf(all_reliability).any().item())

assert all_reliability.shape == (RELIABILITY_CHECK_VIDEOS, NUM_FRAMES, 4)
assert torch.isfinite(all_reliability).all()

print("\nMulti-video reliability extraction passed.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 10.2 — RELIABILITY DISTRIBUTION SUMMARY
# ============================================================

reliability_names = ["Q: Image quality", "M: Motion consistency", "B: Boundary consistency", "C: Cycle consistency"]
summary_rows = []

for feature_index, name in enumerate(reliability_names):
    values = all_reliability[:, :, feature_index]
    video_means = values.mean(dim=1)
    summary_rows.append({
        "Reliability Indicator": name,
        "Overall Mean": values.mean().item(),
        "Overall Std": values.std().item(),
        "Mean Within-Video Temporal Std": values.std(dim=1).mean().item(),
        "Across-Video Mean Std": video_means.std().item(),
        "Minimum": values.min().item(),
        "Maximum": values.max().item(),
    })

reliability_summary = pd.DataFrame(summary_rows)
display(reliability_summary.round(3))

In [ ]:
# ============================================================
# SECTION 10.3 — ESTIMATED CYCLE-LAG DISTRIBUTION
# ============================================================

cycle_lags_np = all_cycle_lags.numpy()
cycle_lag_counts = pd.Series(cycle_lags_np).value_counts().sort_index()

print("=" * 60)
print("CARDIAC-CYCLE LAG SUMMARY")
print("=" * 60)
print("Minimum estimated lag:", int(all_cycle_lags.min()))
print("Mean estimated lag:", f"{all_cycle_lags.float().mean().item():.2f}")
print("Maximum estimated lag:", int(all_cycle_lags.max()))
print("\nLag counts:")
display(cycle_lag_counts.rename("Number of videos").to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(cycle_lags_np, bins=np.arange(2.5, 17.5, 1), edgecolor="black")
ax.set_xlabel("Estimated cycle lag (sampled frames)")
ax.set_ylabel("Number of validation videos")
ax.set_title("Estimated Cardiac-Cycle Lag Distribution")
ax.set_xticks(range(3, 17))
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 10.4 — RELIABILITY FEATURE CORRELATIONS
# ============================================================

flattened_reliability = all_reliability.reshape(-1, 4).numpy()
reliability_correlation = pd.DataFrame(flattened_reliability, columns=["Q", "M", "B", "C"]).corr()

print("Reliability feature correlation matrix:")
display(reliability_correlation.round(3))

In [ ]:
# ============================================================
# SECTION 10.5 — RELIABILITY SANITY CHECKS
# ============================================================

overall_std = all_reliability.reshape(-1, 4).std(dim=0)
within_video_std = all_reliability.std(dim=1).mean(dim=0)

print("=" * 60)
print("RELIABILITY SANITY CHECK")
print("=" * 60)
for i, name in enumerate(["Q", "M", "B", "C"]):
    print(f"{name}: overall std = {overall_std[i].item():.3f}, mean temporal std = {within_video_std[i].item():.3f}")

# Weak collapse checks; not clinical-validity thresholds.
assert torch.all(overall_std > 0.01), "At least one reliability feature has nearly collapsed globally."
assert torch.all(within_video_std > 0.01), "At least one reliability feature has nearly collapsed temporally."

fraction_at_min_lag = (all_cycle_lags == reliability_extractor.min_cycle_lag).float().mean().item()
fraction_at_max_lag = (all_cycle_lags == reliability_extractor.max_cycle_lag).float().mean().item()

print("\nFraction at minimum cycle lag:", f"{fraction_at_min_lag:.3f}")
print("Fraction at maximum cycle lag:", f"{fraction_at_max_lag:.3f}")
print("\nMulti-video reliability sanity checks passed.")
print("=" * 60)

# 11. Video Timing for the Neural DDE

The Neural DDE uses time in **seconds** rather than frame positions. Each sampled frame's time is calculated as its original frame index divided by the video FPS.

Since 32 frames are sampled from videos of different lengths, the time between samples can vary by video.

The dataset now also returns `fps`, `sample_times`, and approximate video duration. Preprocessing, augmentation, model weights, and saved checkpoints stay unchanged.

In [ ]:
# ============================================================
# SECTION 11.1 — DATASET WITH PHYSICAL VIDEO TIME
# ============================================================

class EchoNetDynamicDataset(Dataset):
    # AVI -> sampling -> ROI -> optional augmentation -> preprocessing; returns physical sample times.

    def __init__(self, csv_file, video_dir, split, frame_sampler, preprocessor, augmenter=None, roi_processor=None):
        self.video_dir = Path(video_dir)
        self.frame_sampler, self.preprocessor = frame_sampler, preprocessor
        self.augmenter, self.roi_processor = augmenter, roi_processor

        metadata = pd.read_csv(csv_file)
        required_columns = {"FileName", "EF", "FPS", "Split"}
        missing_columns = required_columns - set(metadata.columns)
        if missing_columns: raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

        split = split.upper()
        metadata["Split"] = metadata["Split"].astype(str).str.upper()
        self.metadata = metadata[metadata["Split"] == split].reset_index(drop=True)
        if self.metadata.empty: raise ValueError(f"No samples found for split: {split}")
        self.split = split

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        filename, ef, fps = str(row["FileName"]), float(row["EF"]), float(row["FPS"])
        if not np.isfinite(ef) or not 0 <= ef <= 100: raise ValueError(f"Invalid EF value for {filename}: {ef}")
        if not np.isfinite(fps) or fps <= 0: raise ValueError(f"Invalid FPS value for {filename}: {fps}")

        frames = self._read_video(self._get_video_path(filename))
        original_num_frames = len(frames)
        sampled_frames, frame_indices = self.frame_sampler(frames)
        sample_times = frame_indices.astype(np.float32) / fps
        video_duration = (original_num_frames - 1) / fps

        if self.roi_processor is not None:
            sampled_frames, roi_bbox = self.roi_processor(sampled_frames)
        else:
            h, w = sampled_frames.shape[1:3]
            roi_bbox = (0, 0, w, h)

        if self.augmenter is not None: sampled_frames = self.augmenter(sampled_frames)
        video = self.preprocessor(sampled_frames)
        if not isinstance(video, torch.Tensor): raise TypeError("Preprocessor must return a torch.Tensor.")

        return {
            "video": video,
            "ef": torch.tensor([ef], dtype=torch.float32),
            "filename": filename,
            "frame_indices": torch.as_tensor(frame_indices, dtype=torch.long),
            "sample_times": torch.as_tensor(sample_times, dtype=torch.float32),
            "fps": torch.tensor(fps, dtype=torch.float32),
            "video_duration": torch.tensor(video_duration, dtype=torch.float32),
            "original_num_frames": original_num_frames,
            "roi_bbox": torch.tensor(roi_bbox, dtype=torch.long),
        }

    def _get_video_path(self, filename):
        path = Path(filename)
        return self.video_dir / path.name if path.suffix else self.video_dir / f"{filename}.avi"

    @staticmethod
    def _read_video(video_path):
        capture = cv2.VideoCapture(str(video_path))
        if not capture.isOpened(): raise RuntimeError(f"Could not open video: {video_path}")

        frames = []
        try:
            while True:
                success, frame = capture.read()
                if not success: break
                frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        finally:
            capture.release()

        if not frames: raise RuntimeError(f"No frames were read from: {video_path}")
        return np.stack(frames)

In [ ]:
# ============================================================
# SECTION 11.2 — RECREATE DATASETS WITH TEMPORAL METADATA
# ============================================================

train_dataset = EchoNetDynamicDataset(csv_file=CSV_FILE, video_dir=VIDEO_DIR, split="TRAIN",
                                      frame_sampler=sampler, preprocessor=preprocessor,
                                      augmenter=train_augmenter, roi_processor=roi_detector)
val_dataset = EchoNetDynamicDataset(csv_file=CSV_FILE, video_dir=VIDEO_DIR, split="VAL",
                                    frame_sampler=sampler, preprocessor=preprocessor,
                                    augmenter=None, roi_processor=roi_detector)

print("TRAIN:", len(train_dataset))
print("VAL:", len(val_dataset))

assert len(train_dataset) == 7465
assert len(val_dataset) == 1288

In [ ]:
# ============================================================
# SECTION 11.3 — PHYSICAL TIME CHECK
# ============================================================

sample = val_dataset[0]
frame_indices, sample_times = sample["frame_indices"], sample["sample_times"]
fps, duration = sample["fps"].item(), sample["video_duration"].item()
time_steps = sample_times[1:] - sample_times[:-1]

print("=" * 60)
print("ECHOCARDIOGRAPHY TIME CHECK")
print("=" * 60)
print("Filename:", sample["filename"])
print("FPS:", fps)
print("Original frames:", sample["original_num_frames"])
print("Video duration:", f"{duration:.3f} seconds")
print("\nFirst sampled frame:", frame_indices[0].item())
print("First sample time:", f"{sample_times[0].item():.3f} s")
print("\nLast sampled frame:", frame_indices[-1].item())
print("Last sample time:", f"{sample_times[-1].item():.3f} s")
print("\nMean sampled Δt:", f"{time_steps.mean().item():.4f} s")
print("Minimum sampled Δt:", f"{time_steps.min().item():.4f} s")
print("Maximum sampled Δt:", f"{time_steps.max().item():.4f} s")

assert len(sample_times) == NUM_FRAMES
assert torch.all(sample_times[1:] > sample_times[:-1])
assert abs(sample_times[0].item()) < 1e-6
assert abs(sample_times[-1].item() - duration) < 1e-4
assert torch.all(time_steps > 0)

print("\nPhysical-time metadata check passed.")
print("=" * 60)

# 12. Reliability-Conditioned Neural DDE

The **Neural DDE** uses the current and delayed latent states, the 768 Transformer features, and the four reliability scores to model changes over time:

\[
\frac{dz(t)}{dt} = f_\theta(z(t), z(t-\tau), x(t), R(t)).
\]

Each video's delay \(\tau\) comes from its estimated heartbeat interval, converted to seconds using the sample timestamps.

The delayed state is found by linear interpolation between past states. When there is not enough history, the initial state is used instead.

This section only tests a forward pass. No parameters are trained yet.

In [ ]:
# ============================================================
# SECTION 12.1 — RELIABILITY-CONDITIONED DDE DYNAMICS
# ============================================================

class ReliabilityConditionedDDEDynamics(nn.Module):
    # dz/dt = f(z_current, z_delayed, x_current, reliability).

    def __init__(self, latent_dim=256, reliability_dim=4, hidden_dim=512, dropout=0.1):
        super().__init__()
        input_dim = 3 * latent_dim + reliability_dim
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.GELU(), nn.LayerNorm(hidden_dim), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, z_current, z_delayed, x_current, reliability):
        combined = torch.cat([z_current, z_delayed, x_current, reliability], dim=-1)
        return self.network(combined)

In [ ]:
# ============================================================
# SECTION 12.2 — NEURAL DDE BLOCK
# ============================================================

class ReliabilityConditionedNeuralDDE(nn.Module):
    # Transformer features [B,T,768] + reliability [B,T,4] -> DDE states [B,T,D], pooled feature, and delay seconds.

    def __init__(self, input_dim=768, latent_dim=256, reliability_dim=4, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.input_dim, self.latent_dim = input_dim, latent_dim
        self.input_projection = nn.Sequential(nn.Linear(input_dim, latent_dim), nn.LayerNorm(latent_dim), nn.GELU())
        self.dynamics = ReliabilityConditionedDDEDynamics(
            latent_dim=latent_dim, reliability_dim=reliability_dim, hidden_dim=hidden_dim, dropout=dropout,
        )
        self.final_norm = nn.LayerNorm(latent_dim)

    def _compute_delay_seconds(self, sample_times, cycle_lags):
        B, T = sample_times.shape
        delays = []
        for b in range(B):
            lag = int(cycle_lags[b].item())
            if lag <= 0 or lag >= T: raise ValueError(f"Invalid cycle lag: {lag}")
            delays.append(torch.median(sample_times[b, lag:] - sample_times[b, :-lag]))
        return torch.stack(delays)

    def _interpolate_delayed_state(self, history_states, history_times, delayed_times):
        # Linearly interpolate z(t - tau), with constant initial-history boundary conditions.
        B, H, D = history_states.shape
        delayed_states = []

        for b in range(B):
            target_time, times, states = delayed_times[b], history_times[b], history_states[b]
            if target_time <= times[0]:
                delayed_state = states[0]
            elif target_time >= times[-1]:
                delayed_state = states[-1]
            else:
                right_index = int(torch.searchsorted(times, target_time).item())
                left_index = right_index - 1
                t_left, t_right = times[left_index], times[right_index]
                z_left, z_right = states[left_index], states[right_index]
                alpha = (target_time - t_left) / (t_right - t_left + 1e-8)
                delayed_state = z_left + alpha * (z_right - z_left)
            delayed_states.append(delayed_state)

        return torch.stack(delayed_states, dim=0)

    def forward(self, sequence_features, reliability, sample_times, cycle_lags):
        if sequence_features.ndim != 3: raise ValueError("sequence_features must have shape [B,T,D].")
        if reliability.ndim != 3: raise ValueError("reliability must have shape [B,T,4].")
        if sample_times.ndim != 2: raise ValueError("sample_times must have shape [B,T].")

        B, T, D = sequence_features.shape
        if reliability.shape[:2] != (B, T): raise ValueError("Reliability dimensions do not match Transformer sequence.")
        if sample_times.shape != (B, T): raise ValueError("Timestamp dimensions do not match Transformer sequence.")
        if not torch.all(sample_times[:, 1:] - sample_times[:, :-1] > 0): raise ValueError("Sample times must be strictly increasing.")

        projected_features = self.input_projection(sequence_features)
        delay_seconds = self._compute_delay_seconds(sample_times, cycle_lags)
        z_current = projected_features[:, 0, :]
        latent_states = [z_current]

        # Explicit Euler DDE integration over observed physical timestamps.
        for i in range(1, T):
            current_time, previous_time = sample_times[:, i], sample_times[:, i - 1]
            dt, delayed_time = current_time - previous_time, current_time - delay_seconds
            history_states, history_times = torch.stack(latent_states, dim=1), sample_times[:, :i]
            z_delayed = self._interpolate_delayed_state(history_states, history_times, delayed_time)
            x_current, r_current = projected_features[:, i, :], reliability[:, i, :]
            z_next = z_current + dt.unsqueeze(-1) * self.dynamics(z_current, z_delayed, x_current, r_current)
            latent_states.append(z_next)
            z_current = z_next

        dde_sequence = self.final_norm(torch.stack(latent_states, dim=1))
        return dde_sequence, dde_sequence.mean(dim=1), delay_seconds

In [ ]:
# ============================================================
# SECTION 12.3 — INITIALIZE NEURAL DDE
# ============================================================

DDE_LATENT_DIM = 256
DDE_HIDDEN_DIM = 512
neural_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768, latent_dim=DDE_LATENT_DIM, reliability_dim=4, hidden_dim=DDE_HIDDEN_DIM, dropout=0.1,
).to(device)

total_dde_parameters = sum(p.numel() for p in neural_dde.parameters())

print("=" * 60)
print("NEURAL DDE")
print("=" * 60)
print("Input feature dimension:", 768)
print("DDE latent dimension:", DDE_LATENT_DIM)
print("DDE hidden dimension:", DDE_HIDDEN_DIM)
print("Total DDE parameters:", f"{total_dde_parameters:,}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 12.4 — REAL-VIDEO NEURAL DDE CHECK
# ============================================================

best_transformer_model.eval()
reliability_extractor.eval()
neural_dde.eval()

# Use two validation videos to verify batch behavior.
dde_check_loader = DataLoader(Subset(val_dataset, [0, 1]), batch_size=2, shuffle=False, num_workers=0)
batch = next(iter(dde_check_loader))
video, sample_times = batch["video"].to(device), batch["sample_times"].to(device)

with torch.no_grad():
    frame_features = best_transformer_model.spatial_encoder(video)
    sequence_features, transformer_video_feature = best_transformer_model.temporal_transformer(frame_features)
    reliability, cycle_lags = reliability_extractor(video, sequence_features)
    dde_sequence, dde_video_feature, delay_seconds = neural_dde(sequence_features, reliability, sample_times, cycle_lags)

print("=" * 60)
print("NEURAL DDE FORWARD CHECK")
print("=" * 60)
print("Video:", video.shape)
print("Transformer sequence:", sequence_features.shape)
print("Reliability:", reliability.shape)
print("Sample times:", sample_times.shape)
print("Cycle lags:", cycle_lags.tolist())
print("Physical delays:", [round(x, 3) for x in delay_seconds.cpu().tolist()], "seconds")
print("\nDDE sequence:", dde_sequence.shape)
print("DDE video feature:", dde_video_feature.shape)
print("Contains NaN:", torch.isnan(dde_sequence).any().item())
print("Contains Inf:", torch.isinf(dde_sequence).any().item())

assert dde_sequence.shape == (2, NUM_FRAMES, DDE_LATENT_DIM)
assert dde_video_feature.shape == (2, DDE_LATENT_DIM)
assert delay_seconds.shape == (2,)
assert torch.isfinite(dde_sequence).all()
assert torch.isfinite(dde_video_feature).all()
assert torch.all(delay_seconds > 0)

print("\nNeural DDE forward check passed.")
print("=" * 60)

# 13. Experiment 4: Reliability-Aware Neural DDE

This experiment tests whether the **Neural DDE** improves EF prediction over the Transformer baseline. The best ConvNeXt and Transformer models from Experiment 3 stay frozen.

The Transformer frame features and reliability scores feed into the Neural DDE. A separate branch projects the Transformer video feature. A reliability-aware gate combines the two branches, and an EF prediction layer uses the combined feature to predict EF.

In [ ]:
# ============================================================
# SECTION 13.1 — RELIABILITY-AWARE DYNAMIC FUSION
# ============================================================

# class ReliabilityAwareFusion(nn.Module):
#     # Fuse Transformer [B,768] and DDE [B,256] features into [B,fusion_dim]; gate≈1 favors Transformer.

#     def __init__(self, transformer_dim=768, dde_dim=256, reliability_dim=4, fusion_dim=256, hidden_dim=256, dropout=0.1):
#         super().__init__()
#         self.transformer_projection = nn.Sequential(nn.Linear(transformer_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.GELU())
#         self.dde_projection = nn.Sequential(nn.Linear(dde_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.GELU())
#         self.gate_network = nn.Sequential(
#             nn.Linear(2 * fusion_dim + reliability_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(hidden_dim, 1), nn.Sigmoid(),
#         )

#     def forward(self, transformer_feature, dde_feature, reliability_summary):
#         transformer_projected = self.transformer_projection(transformer_feature)
#         dde_projected = self.dde_projection(dde_feature)
#         fusion_gate = self.gate_network(torch.cat([transformer_projected, dde_projected, reliability_summary], dim=1))
#         fused_feature = fusion_gate * transformer_projected + (1.0 - fusion_gate) * dde_projected
#         return fused_feature, fusion_gate





# ============================================================
# SECTION 13.1 — RELIABILITY-AWARE RESIDUAL FUSION
# ============================================================

class ReliabilityAwareResidualFusion(nn.Module):
    """
    Learn a reliability-aware correction to the frozen
    Transformer EF prediction.
    """

    def __init__(
        self,
        transformer_dim=768,
        dde_dim=256,
        reliability_dim=4,
        hidden_dim=256,
        dropout=0.1,
    ):
        super().__init__()

        input_dim = transformer_dim + dde_dim + reliability_dim

        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
        )

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

        self.residual_head = nn.Linear(hidden_dim, 1)

        # Start exactly at the frozen Transformer baseline.
        nn.init.zeros_(self.residual_head.weight)
        nn.init.zeros_(self.residual_head.bias)

    def forward(
        self,
        transformer_feature,
        dde_feature,
        reliability_summary,
    ):
        x = torch.cat(
            [transformer_feature, dde_feature, reliability_summary],
            dim=1,
        )

        x = self.shared(x)
        gate = self.gate(x)
        residual = self.residual_head(x)

        correction = gate * residual

        return correction, gate, residual

In [ ]:
# ============================================================
# SECTION 13.2 — COMPLETE RELIABILITY-AWARE DDE EF MODEL
# ============================================================

# class ReliabilityAwareDDEEFModel(nn.Module):
#     """Frozen ConvNeXt/Transformer backbone + Neural DDE, fusion, and EF head."""

#     def __init__(self, backbone, reliability_extractor, neural_dde, transformer_dim=768, dde_dim=256, fusion_dim=256):
#         super().__init__()
#         self.backbone = backbone
#         self.reliability_extractor = reliability_extractor
#         self.neural_dde = neural_dde

#         self.fusion = ReliabilityAwareFusion(
#             transformer_dim=transformer_dim, dde_dim=dde_dim, reliability_dim=4,
#             fusion_dim=fusion_dim, hidden_dim=fusion_dim, dropout=0.1,
#         )

#         self.ef_head = nn.Sequential(
#             nn.Linear(fusion_dim, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, 1),
#         )

#     def forward(self, video, sample_times):
#         frame_features = self.backbone.spatial_encoder(video)
#         sequence_features, transformer_video_feature = self.backbone.temporal_transformer(frame_features)

#         reliability, cycle_lags = self.reliability_extractor(video, sequence_features)

#         dde_sequence, dde_video_feature, delay_seconds = self.neural_dde(
#             sequence_features, reliability, sample_times, cycle_lags,
#         )

#         reliability_summary = reliability.mean(dim=1)
#         fused_feature, fusion_gate = self.fusion(
#             transformer_video_feature, dde_video_feature, reliability_summary,
#         )
#         predicted_ef = self.ef_head(fused_feature)

#         return {
#             "predicted_ef": predicted_ef,
#             "dde_sequence": dde_sequence,
#             "dde_video_feature": dde_video_feature,
#             "reliability": reliability,
#             "cycle_lags": cycle_lags,
#             "delay_seconds": delay_seconds,
#             "fusion_gate": fusion_gate,
#         }










# ============================================================
# SECTION 13.2 — RESIDUAL DDE EF MODEL
# ============================================================

class ReliabilityAwareDDEEFModel(nn.Module):
    """
    Frozen Transformer EF baseline + reliability-conditioned
    Neural DDE residual correction.
    """

    def __init__(
        self,
        backbone,
        reliability_extractor,
        neural_dde,
        transformer_dim=768,
        dde_dim=256,
    ):
        super().__init__()

        self.backbone = backbone
        self.reliability_extractor = reliability_extractor
        self.neural_dde = neural_dde

        self.residual_fusion = ReliabilityAwareResidualFusion(
            transformer_dim=transformer_dim,
            dde_dim=dde_dim,
            reliability_dim=4,
            hidden_dim=256,
            dropout=0.1,
        )

    def forward(self, video, sample_times):

        # Frozen ConvNeXt + Transformer
        frame_features = self.backbone.spatial_encoder(video)

        sequence_features, transformer_feature = (
            self.backbone.temporal_transformer(frame_features)
        )

        # Original trained Transformer EF prediction
        baseline_ef = self.backbone.ef_head(transformer_feature)

        # Reliability features
        reliability, cycle_lags = self.reliability_extractor(
            video,
            sequence_features,
        )

        # Neural DDE
        dde_sequence, dde_feature, delay_seconds = self.neural_dde(
            sequence_features,
            reliability,
            sample_times,
            cycle_lags,
        )

        reliability_summary = reliability.mean(dim=1)

        # Reliability-aware residual correction
        correction, gate, raw_residual = self.residual_fusion(
            transformer_feature,
            dde_feature,
            reliability_summary,
        )

        predicted_ef = baseline_ef + correction

        return {
            "predicted_ef": predicted_ef,
            "baseline_ef": baseline_ef,
            "correction": correction,
            "raw_residual": raw_residual,
            "fusion_gate": gate,
            "dde_sequence": dde_sequence,
            "dde_video_feature": dde_feature,
            "reliability": reliability,
            "cycle_lags": cycle_lags,
            "delay_seconds": delay_seconds,
        }

In [ ]:
# ============================================================
# SECTION 13.3 — BUILD EXPERIMENT 4 MODEL
# ============================================================

# set_seed(SEED)

# trainable_neural_dde = ReliabilityConditionedNeuralDDE(
#     input_dim=768,
#     latent_dim=256,
#     reliability_dim=4,
#     hidden_dim=512,
#     dropout=0.1,
# ).to(device)

# dde_ef_model = ReliabilityAwareDDEEFModel(
#     backbone=best_transformer_model,
#     reliability_extractor=reliability_extractor,
#     neural_dde=trainable_neural_dde,
#     transformer_dim=768,
#     dde_dim=256,
#     fusion_dim=256,
# ).to(device)

# # Freeze pretrained backbone
# for p in dde_ef_model.backbone.parameters():
#     p.requires_grad = False

# dde_ef_model.reliability_extractor.eval()

# # Verify trainable components
# checks = {
#     "Backbone trainable": any(p.requires_grad for p in dde_ef_model.backbone.parameters()),
#     "Neural DDE trainable": any(p.requires_grad for p in dde_ef_model.neural_dde.parameters()),
#     "Fusion trainable": any(p.requires_grad for p in dde_ef_model.fusion.parameters()),
#     "EF head trainable": any(p.requires_grad for p in dde_ef_model.ef_head.parameters()),
# }

# total_params = sum(p.numel() for p in dde_ef_model.parameters())
# trainable_params = sum(p.numel() for p in dde_ef_model.parameters() if p.requires_grad)

# print("=" * 60)
# print("RELIABILITY-AWARE DDE MODEL")
# print("=" * 60)

# for name, value in checks.items():
#     print(f"{name}: {value}")

# print()
# print(f"Total parameters: {total_params:,}")
# print(f"Trainable parameters: {trainable_params:,}")
# print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")

# assert checks["Backbone trainable"] is False
# assert checks["Neural DDE trainable"] is True
# assert checks["Fusion trainable"] is True
# assert checks["EF head trainable"] is True

# print("\nExperiment 4 model configured successfully.")
# print("=" * 60)











# ============================================================
# SECTION 13.3 — BUILD RESIDUAL DDE MODEL
# ============================================================

set_seed(SEED)

# Fresh DDE for the new controlled experiment
trainable_neural_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

dde_ef_model = ReliabilityAwareDDEEFModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=trainable_neural_dde,
    transformer_dim=768,
    dde_dim=256,
).to(device)

# Freeze original ConvNeXt + Transformer + EF head
for p in dde_ef_model.backbone.parameters():
    p.requires_grad = False

dde_ef_model.reliability_extractor.eval()

checks = {
    "Backbone trainable": any(
        p.requires_grad for p in dde_ef_model.backbone.parameters()
    ),
    "Neural DDE trainable": any(
        p.requires_grad for p in dde_ef_model.neural_dde.parameters()
    ),
    "Residual fusion trainable": any(
        p.requires_grad for p in dde_ef_model.residual_fusion.parameters()
    ),
}

total_params = sum(p.numel() for p in dde_ef_model.parameters())
trainable_params = sum(
    p.numel() for p in dde_ef_model.parameters() if p.requires_grad
)

print("=" * 60)
print("RESIDUAL DDE MODEL")
print("=" * 60)

for name, value in checks.items():
    print(f"{name}: {value}")

print()
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")

assert checks["Backbone trainable"] is False
assert checks["Neural DDE trainable"] is True
assert checks["Residual fusion trainable"] is True

print("\nResidual DDE model configured successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 13.4 — COMPLETE MODEL FORWARD CHECK
# ============================================================

# dde_ef_model.eval()

# check_loader = DataLoader(
#     Subset(val_dataset, [0, 1]),
#     batch_size=2,
#     shuffle=False,
#     num_workers=0,
# )

# batch = next(iter(check_loader))
# video = batch["video"].to(device)
# sample_times = batch["sample_times"].to(device)

# with torch.no_grad():
#     output = dde_ef_model(video, sample_times)

# pred = output["predicted_ef"]
# dde_seq = output["dde_sequence"]
# gates = output["fusion_gate"]

# print("=" * 60)
# print("COMPLETE DDE EF MODEL CHECK")
# print("=" * 60)
# print("Prediction:", pred.shape)
# print("DDE sequence:", dde_seq.shape)
# print("Reliability:", output["reliability"].shape)
# print("Cycle lags:", output["cycle_lags"].tolist())
# print("Physical delays:", [round(x, 3) for x in output["delay_seconds"].cpu().tolist()])
# print("Fusion gates:", [round(x, 3) for x in gates.squeeze(1).cpu().tolist()])
# print("Prediction finite:", torch.isfinite(pred).all().item())

# assert pred.shape == (2, 1)
# assert dde_seq.shape == (2, NUM_FRAMES, 256)
# assert gates.shape == (2, 1)
# assert torch.isfinite(pred).all()
# assert torch.isfinite(dde_seq).all()

# print("\nComplete DDE EF model forward check passed.")
# print("=" * 60)













# ============================================================
# SECTION 13.4 — RESIDUAL BASELINE VERIFICATION
# ============================================================

dde_ef_model.eval()

check_loader = DataLoader(
    Subset(val_dataset, [0, 1]),
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

batch = next(iter(check_loader))
video = batch["video"].to(device)
sample_times = batch["sample_times"].to(device)

with torch.no_grad():
    output = dde_ef_model(video, sample_times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]
correction = output["correction"]
gate = output["fusion_gate"]

max_difference = (predicted - baseline).abs().max().item()

print("=" * 60)
print("RESIDUAL BASELINE CHECK")
print("=" * 60)
print("Baseline EF:", baseline.squeeze(1).cpu().tolist())
print("Final EF:", predicted.squeeze(1).cpu().tolist())
print("Correction:", correction.squeeze(1).cpu().tolist())
print("Fusion gate:", gate.squeeze(1).cpu().tolist())
print(f"Maximum baseline/final difference: {max_difference:.8f}")

assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()
assert max_difference < 1e-6

print("\nResidual model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 13.5 — DDE TRAINING CONFIGURATION
# ============================================================

# DDE_E2E_BEST_PATH = E2E_RUN_DIR / "04_reliability_dde_best.pth"
# DDE_E2E_LATEST_PATH = E2E_RUN_DIR / "04_reliability_dde_latest.pth"

# DDE_BATCH_SIZE = 8
# DDE_MAX_EPOCHS = 10
# DDE_PATIENCE = 3

# DDE_LR = 1e-4
# DDE_FUSION_LR = 3e-4
# DDE_HEAD_LR = 1e-3
# DDE_WEIGHT_DECAY = 1e-4

# dde_train_generator = torch.Generator().manual_seed(SEED)

# dde_train_loader = DataLoader(
#     train_dataset,
#     batch_size=DDE_BATCH_SIZE,
#     shuffle=True,
#     num_workers=0,
#     pin_memory=USE_AMP,
#     generator=dde_train_generator,
# )

# dde_val_loader = DataLoader(
#     val_dataset,
#     batch_size=DDE_BATCH_SIZE,
#     shuffle=False,
#     num_workers=0,
#     pin_memory=USE_AMP,
# )

# TRANSFORMER_BASELINE_MAE = transformer_result["val_mae"]

# print("=" * 60)
# print("DDE TRAINING CONFIGURATION")
# print("=" * 60)
# print(f"Batch size: {DDE_BATCH_SIZE}")
# print(f"TRAIN batches: {len(dde_train_loader)}")
# print(f"VAL batches: {len(dde_val_loader)}")
# print(f"Transformer baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
# print(f"Maximum epochs: {DDE_MAX_EPOCHS}")
# print(f"Early stopping patience: {DDE_PATIENCE}")
# print(f"BEST checkpoint: {DDE_E2E_BEST_PATH}")
# print("=" * 60)

















# ============================================================
# SECTION 13.5 — RESIDUAL DDE TRAINING CONFIGURATION
# ============================================================

RESIDUAL_DDE_BEST_PATH = E2E_RUN_DIR / "04_residual_dde_best.pth"
RESIDUAL_DDE_LATEST_PATH = E2E_RUN_DIR / "04_residual_dde_latest.pth"

DDE_BATCH_SIZE = 8
DDE_MAX_EPOCHS = 10
DDE_PATIENCE = 3

DDE_LR = 1e-4
RESIDUAL_LR = 3e-4
DDE_WEIGHT_DECAY = 1e-4

# Exclude the 3 TRAIN videos with < 32 frames
train_frames = train_dataset.metadata["NumberOfFrames"].astype(int)
dde_train_indices = train_dataset.metadata.index[
    train_frames >= NUM_FRAMES
].tolist()

dde_train_dataset = Subset(train_dataset, dde_train_indices)

dde_train_generator = torch.Generator().manual_seed(SEED)

dde_train_loader = DataLoader(
    dde_train_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=USE_AMP,
    generator=dde_train_generator,
)

dde_val_loader = DataLoader(
    val_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=USE_AMP,
)

TRANSFORMER_BASELINE_MAE = transformer_result["val_mae"]

print("=" * 60)
print("RESIDUAL DDE TRAINING CONFIGURATION")
print("=" * 60)
print(f"DDE TRAIN size: {len(dde_train_dataset)}")
print(f"DDE TRAIN batches: {len(dde_train_loader)}")
print(f"VAL size: {len(val_dataset)}")
print(f"VAL batches: {len(dde_val_loader)}")
print(f"Transformer baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print(f"Maximum epochs: {DDE_MAX_EPOCHS}")
print(f"Early stopping patience: {DDE_PATIENCE}")
print(f"BEST checkpoint: {RESIDUAL_DDE_BEST_PATH}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 13.6 — OPTIMIZER AND AUTO-RESUME
# ============================================================

# dde_criterion = nn.MSELoss()

# dde_optimizer = torch.optim.AdamW(
#     [
#         {"params": dde_ef_model.neural_dde.parameters(), "lr": DDE_LR},
#         {"params": dde_ef_model.fusion.parameters(), "lr": DDE_FUSION_LR},
#         {"params": dde_ef_model.ef_head.parameters(), "lr": DDE_HEAD_LR},
#     ],
#     weight_decay=DDE_WEIGHT_DECAY,
# )

# dde_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# dde_start_epoch = 0
# dde_best_val_mae = float("inf")
# dde_epochs_without_improvement = 0
# dde_history = []

# if DDE_E2E_LATEST_PATH.exists():
#     checkpoint = torch.load(
#         DDE_E2E_LATEST_PATH,
#         map_location=device,
#         weights_only=False,
#     )

#     dde_ef_model.load_state_dict(checkpoint["model_state_dict"])
#     dde_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
#     dde_scaler.load_state_dict(checkpoint["scaler_state_dict"])

#     dde_start_epoch = checkpoint["epoch"]
#     dde_best_val_mae = checkpoint["best_val_mae"]
#     dde_epochs_without_improvement = checkpoint["epochs_without_improvement"]
#     dde_history = checkpoint.get("history", [])

#     # Restore reproducibility state
#     random.setstate(checkpoint["python_rng_state"])
#     np.random.set_state(checkpoint["numpy_rng_state"])
#     torch.set_rng_state(checkpoint["torch_rng_state"].cpu())

#     if torch.cuda.is_available() and checkpoint.get("cuda_rng_state_all") is not None:
#         torch.cuda.set_rng_state_all(
#             [state.cpu() for state in checkpoint["cuda_rng_state_all"]]
#         )

#     dde_train_generator.set_state(
#         checkpoint["train_generator_state"].cpu()
#     )

#     print("=" * 60)
#     print("DDE CHECKPOINT RESTORED")
#     print("=" * 60)
#     print(f"Completed epochs: {dde_start_epoch}")
#     print(f"Best VAL MAE: {dde_best_val_mae:.3f}")
#     print(f"No-improvement epochs: {dde_epochs_without_improvement}")
#     print(f"Next epoch: {dde_start_epoch + 1}")
#     print("=" * 60)

# else:
#     print("=" * 60)
#     print("NO DDE CHECKPOINT FOUND")
#     print("=" * 60)
#     print("Training will begin from epoch 1.")
#     print("=" * 60)















# ============================================================
# SECTION 13.6 — OPTIMIZER AND AUTO-RESUME
# ============================================================

dde_criterion = nn.MSELoss()

dde_optimizer = torch.optim.AdamW(
    [
        {"params": dde_ef_model.neural_dde.parameters(), "lr": DDE_LR},
        {"params": dde_ef_model.residual_fusion.parameters(), "lr": RESIDUAL_LR},
    ],
    weight_decay=DDE_WEIGHT_DECAY,
)

dde_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

dde_start_epoch = 0
dde_best_val_mae = TRANSFORMER_BASELINE_MAE
dde_epochs_without_improvement = 0
dde_history = []

# ------------------------------------------------------------
# Resume an interrupted residual-DDE run
# ------------------------------------------------------------

if RESIDUAL_DDE_LATEST_PATH.exists():

    checkpoint = torch.load(
        RESIDUAL_DDE_LATEST_PATH,
        map_location=device,
        weights_only=False,
    )

    dde_ef_model.load_state_dict(checkpoint["model_state_dict"])
    dde_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    dde_scaler.load_state_dict(checkpoint["scaler_state_dict"])

    dde_start_epoch = checkpoint["epoch"]
    dde_best_val_mae = checkpoint["best_val_mae"]
    dde_epochs_without_improvement = checkpoint["epochs_without_improvement"]
    dde_history = checkpoint.get("history", [])

    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())

    if torch.cuda.is_available() and checkpoint["cuda_rng_state_all"] is not None:
        torch.cuda.set_rng_state_all(
            [x.cpu() for x in checkpoint["cuda_rng_state_all"]]
        )

    dde_train_generator.set_state(
        checkpoint["train_generator_state"].cpu()
    )

    print("=" * 60)
    print("RESIDUAL DDE CHECKPOINT RESTORED")
    print("=" * 60)
    print(f"Completed epochs: {dde_start_epoch}")
    print(f"Best VAL MAE: {dde_best_val_mae:.3f}")
    print(f"Next epoch: {dde_start_epoch + 1}")
    print("=" * 60)

# ------------------------------------------------------------
# New run: save exact Transformer-equivalent starting point
# ------------------------------------------------------------

else:

    initial_checkpoint = {
        "epoch": 0,
        "model_state_dict": dde_ef_model.state_dict(),
        "optimizer_state_dict": dde_optimizer.state_dict(),
        "scaler_state_dict": dde_scaler.state_dict(),
        "best_val_mae": TRANSFORMER_BASELINE_MAE,
        "epochs_without_improvement": 0,
        "history": [],
        "stage": "residual_dde_baseline",
    }

    torch.save(initial_checkpoint, RESIDUAL_DDE_BEST_PATH)

    print("=" * 60)
    print("NEW RESIDUAL DDE RUN")
    print("=" * 60)
    print("Training will begin from epoch 1.")
    print(f"Initial BEST MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
    print("Initial BEST = exact frozen Transformer baseline.")
    print("=" * 60)

In [ ]:
# ============================================================
# SECTION 13.7 — RELIABILITY-AWARE NEURAL DDE TRAINING
# ============================================================

# torch.cuda.empty_cache()

# print("=" * 60)
# print("RELIABILITY-AWARE DDE TRAINING")
# print("=" * 60)
# print(f"Starting epoch: {dde_start_epoch + 1}")
# print(f"Transformer baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
# print("=" * 60)

# for epoch in range(dde_start_epoch, DDE_MAX_EPOCHS):

#     # --------------------------------------------------------
#     # TRAIN
#     # --------------------------------------------------------

#     dde_ef_model.train()
#     dde_ef_model.backbone.eval()
#     dde_ef_model.reliability_extractor.eval()

#     train_sq_error = 0.0
#     train_samples = 0

#     for batch_idx, batch in enumerate(dde_train_loader):

#         video = batch["video"].to(device, non_blocking=True)
#         times = batch["sample_times"].to(device, non_blocking=True)
#         true_ef = batch["ef"].to(device, non_blocking=True)

#         dde_optimizer.zero_grad(set_to_none=True)

#         # Frozen ConvNeXt + Transformer + reliability
#         with torch.no_grad():
#             frame_features = dde_ef_model.backbone.spatial_encoder(video)

#             sequence_features, transformer_feature = (
#                 dde_ef_model.backbone.temporal_transformer(frame_features)
#             )

#             reliability, cycle_lags = (
#                 dde_ef_model.reliability_extractor(
#                     video,
#                     sequence_features,
#                 )
#             )

#         # Train DDE + fusion + EF head
#         with torch.amp.autocast(
#             device_type=device.type,
#             enabled=USE_AMP,
#         ):
#             _, dde_feature, _ = dde_ef_model.neural_dde(
#                 sequence_features,
#                 reliability,
#                 times,
#                 cycle_lags,
#             )

#             reliability_summary = reliability.mean(dim=1)

#             fused_feature, fusion_gate = dde_ef_model.fusion(
#                 transformer_feature,
#                 dde_feature,
#                 reliability_summary,
#             )

#             predicted_ef = dde_ef_model.ef_head(fused_feature)
#             loss = dde_criterion(predicted_ef, true_ef)

#         dde_scaler.scale(loss).backward()
#         dde_scaler.step(dde_optimizer)
#         dde_scaler.update()

#         error = predicted_ef.detach().float() - true_ef.float()
#         train_sq_error += error.pow(2).sum().item()
#         train_samples += true_ef.numel()

#         if (batch_idx + 1) % 200 == 0:
#             running_rmse = math.sqrt(train_sq_error / train_samples)
#             print(
#                 f"Epoch {epoch + 1} | "
#                 f"Batch {batch_idx + 1}/{len(dde_train_loader)} | "
#                 f"Train RMSE: {running_rmse:.3f}"
#             )

#     train_rmse = math.sqrt(train_sq_error / train_samples)

#     # --------------------------------------------------------
#     # VALIDATION
#     # --------------------------------------------------------

#     dde_ef_model.eval()

#     val_abs_error = 0.0
#     val_sq_error = 0.0
#     val_samples = 0
#     gate_sum = 0.0

#     with torch.no_grad():

#         for batch in dde_val_loader:

#             video = batch["video"].to(device, non_blocking=True)
#             times = batch["sample_times"].to(device, non_blocking=True)
#             true_ef = batch["ef"].to(device, non_blocking=True)

#             output = dde_ef_model(video, times)

#             predicted_ef = output["predicted_ef"]
#             error = predicted_ef.float() - true_ef.float()

#             val_abs_error += error.abs().sum().item()
#             val_sq_error += error.pow(2).sum().item()
#             val_samples += true_ef.numel()
#             gate_sum += output["fusion_gate"].sum().item()

#     val_mae = val_abs_error / val_samples
#     val_rmse = math.sqrt(val_sq_error / val_samples)
#     mean_gate = gate_sum / val_samples

#     # --------------------------------------------------------
#     # MODEL SELECTION
#     # --------------------------------------------------------

#     improved = val_mae < dde_best_val_mae

#     if improved:
#         dde_best_val_mae = val_mae
#         dde_epochs_without_improvement = 0
#     else:
#         dde_epochs_without_improvement += 1

#     dde_history.append({
#         "epoch": epoch + 1,
#         "train_rmse": train_rmse,
#         "val_mae": val_mae,
#         "val_rmse": val_rmse,
#         "mean_fusion_gate": mean_gate,
#     })

#     print()
#     print(f"DDE Epoch {epoch + 1}/{DDE_MAX_EPOCHS}")
#     print(f"Train RMSE: {train_rmse:.3f}")
#     print(f"Validation MAE: {val_mae:.3f}")
#     print(f"Validation RMSE: {val_rmse:.3f}")
#     print(f"Mean fusion gate: {mean_gate:.3f}")
#     print(f"Transformer baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")

#     # --------------------------------------------------------
#     # CHECKPOINT
#     # --------------------------------------------------------

#     checkpoint = {
#         "epoch": epoch + 1,
#         "model_state_dict": dde_ef_model.state_dict(),
#         "optimizer_state_dict": dde_optimizer.state_dict(),
#         "scaler_state_dict": dde_scaler.state_dict(),
#         "best_val_mae": dde_best_val_mae,
#         "epochs_without_improvement": dde_epochs_without_improvement,
#         "history": dde_history,
#         "val_mae": val_mae,
#         "val_rmse": val_rmse,
#         "train_rmse": train_rmse,
#         "mean_fusion_gate": mean_gate,
#         "python_rng_state": random.getstate(),
#         "numpy_rng_state": np.random.get_state(),
#         "torch_rng_state": torch.get_rng_state(),
#         "cuda_rng_state_all": (
#             torch.cuda.get_rng_state_all()
#             if torch.cuda.is_available() else None
#         ),
#         "train_generator_state": dde_train_generator.get_state(),
#     }

#     torch.save(checkpoint, DDE_E2E_LATEST_PATH)

#     if improved:
#         torch.save(checkpoint, DDE_E2E_BEST_PATH)
#         print("Saved new best DDE model.")

#     print(f"Best DDE VAL MAE: {dde_best_val_mae:.3f}")
#     print(f"No-improvement epochs: {dde_epochs_without_improvement}")
#     print("-" * 60)

#     dde_start_epoch = epoch + 1

#     if dde_epochs_without_improvement >= DDE_PATIENCE:
#         print("Early stopping triggered.")
#         break


# print()
# print("=" * 60)
# print("DDE TRAINING FINISHED")
# print("=" * 60)
# print(f"Completed epochs: {dde_start_epoch}")
# print(f"Transformer baseline VAL MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
# print(f"Best DDE VAL MAE: {dde_best_val_mae:.3f}")
# print(
#     f"MAE improvement: "
#     f"{TRANSFORMER_BASELINE_MAE - dde_best_val_mae:.3f}"
# )
# print("=" * 60)














# ============================================================
# SECTION 13.7 — RESIDUAL NEURAL DDE TRAINING
# ============================================================

torch.cuda.empty_cache()

print("=" * 60)
print("RESIDUAL DDE TRAINING")
print("=" * 60)
print(f"Starting epoch: {dde_start_epoch + 1}")
print(f"Baseline VAL MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print("=" * 60)

for epoch in range(dde_start_epoch, DDE_MAX_EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    dde_ef_model.train()
    dde_ef_model.backbone.eval()
    dde_ef_model.reliability_extractor.eval()

    train_sq_error = 0.0
    train_samples = 0

    for batch_idx, batch in enumerate(dde_train_loader):

        video = batch["video"].to(device, non_blocking=True)
        times = batch["sample_times"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)

        dde_optimizer.zero_grad(set_to_none=True)

        # Frozen ConvNeXt + Transformer + reliability
        with torch.no_grad():
            frame_features = dde_ef_model.backbone.spatial_encoder(video)

            sequence_features, transformer_feature = (
                dde_ef_model.backbone.temporal_transformer(frame_features)
            )

            baseline_ef = dde_ef_model.backbone.ef_head(
                transformer_feature
            )

            reliability, cycle_lags = (
                dde_ef_model.reliability_extractor(
                    video,
                    sequence_features,
                )
            )

        # Trainable DDE + residual correction
        with torch.amp.autocast(
            device_type=device.type,
            enabled=USE_AMP,
        ):
            _, dde_feature, _ = dde_ef_model.neural_dde(
                sequence_features,
                reliability,
                times,
                cycle_lags,
            )

            reliability_summary = reliability.mean(dim=1)

            correction, gate, raw_residual = (
                dde_ef_model.residual_fusion(
                    transformer_feature,
                    dde_feature,
                    reliability_summary,
                )
            )

            predicted_ef = baseline_ef + correction
            loss = dde_criterion(predicted_ef, true_ef)

        dde_scaler.scale(loss).backward()
        dde_scaler.step(dde_optimizer)
        dde_scaler.update()

        error = predicted_ef.detach().float() - true_ef.float()
        train_sq_error += error.pow(2).sum().item()
        train_samples += true_ef.numel()

        if (batch_idx + 1) % 200 == 0:
            rmse = math.sqrt(train_sq_error / train_samples)
            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_idx + 1}/{len(dde_train_loader)} | "
                f"Train RMSE: {rmse:.3f}"
            )

    train_rmse = math.sqrt(train_sq_error / train_samples)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    dde_ef_model.eval()

    val_abs_error = 0.0
    val_sq_error = 0.0
    val_samples = 0

    gate_sum = 0.0
    correction_abs_sum = 0.0

    with torch.no_grad():

        for batch in dde_val_loader:

            video = batch["video"].to(device, non_blocking=True)
            times = batch["sample_times"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)

            output = dde_ef_model(video, times)

            predicted_ef = output["predicted_ef"]
            error = predicted_ef.float() - true_ef.float()

            val_abs_error += error.abs().sum().item()
            val_sq_error += error.pow(2).sum().item()
            val_samples += true_ef.numel()

            gate_sum += output["fusion_gate"].sum().item()
            correction_abs_sum += output["correction"].abs().sum().item()

    val_mae = val_abs_error / val_samples
    val_rmse = math.sqrt(val_sq_error / val_samples)

    mean_gate = gate_sum / val_samples
    mean_abs_correction = correction_abs_sum / val_samples

    # --------------------------------------------------------
    # MODEL SELECTION
    # --------------------------------------------------------

    improved = val_mae < dde_best_val_mae

    if improved:
        dde_best_val_mae = val_mae
        dde_epochs_without_improvement = 0
    else:
        dde_epochs_without_improvement += 1

    dde_history.append({
        "epoch": epoch + 1,
        "train_rmse": train_rmse,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "mean_gate": mean_gate,
        "mean_abs_correction": mean_abs_correction,
    })

    print()
    print(f"Residual DDE Epoch {epoch + 1}/{DDE_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")
    print(f"Mean gate: {mean_gate:.3f}")
    print(f"Mean |EF correction|: {mean_abs_correction:.3f}")
    print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": dde_ef_model.state_dict(),
        "optimizer_state_dict": dde_optimizer.state_dict(),
        "scaler_state_dict": dde_scaler.state_dict(),
        "best_val_mae": dde_best_val_mae,
        "epochs_without_improvement": dde_epochs_without_improvement,
        "history": dde_history,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "train_rmse": train_rmse,
        "mean_gate": mean_gate,
        "mean_abs_correction": mean_abs_correction,
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "train_generator_state": dde_train_generator.get_state(),
    }

    torch.save(checkpoint, RESIDUAL_DDE_LATEST_PATH)

    if improved:
        torch.save(checkpoint, RESIDUAL_DDE_BEST_PATH)
        print("Saved new best residual DDE model.")

    print(f"Best VAL MAE: {dde_best_val_mae:.3f}")
    print(f"No-improvement epochs: {dde_epochs_without_improvement}")
    print("-" * 60)

    dde_start_epoch = epoch + 1

    if dde_epochs_without_improvement >= DDE_PATIENCE:
        print("Early stopping triggered.")
        break


print()
print("=" * 60)
print("RESIDUAL DDE TRAINING FINISHED")
print("=" * 60)
print(f"Completed epochs: {dde_start_epoch}")
print(f"Baseline VAL MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print(f"Best residual DDE VAL MAE: {dde_best_val_mae:.3f}")
print(
    f"MAE improvement: "
    f"{TRANSFORMER_BASELINE_MAE - dde_best_val_mae:.3f}"
)
print("=" * 60)

# 14....

In [ ]:
# ============================================================
# SECTION 14.1 — PRESERVE TRAINED STEP 13 MODEL
# ============================================================

RESIDUAL_DDE_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "04_residual_dde_best_trained.pth"
)

# Load final trained checkpoint
trained_checkpoint = torch.load(
    RESIDUAL_DDE_LATEST_PATH,
    map_location="cpu",
    weights_only=False,
)

history = trained_checkpoint["history"]

# Best epoch among actual DDE training epochs (1+)
best_trained = min(history, key=lambda x: x["val_mae"])

print("=" * 60)
print("STEP 13 CHECKPOINT PRESERVATION")
print("=" * 60)
print(f"Final trained epoch: {trained_checkpoint['epoch']}")
print(f"Best trained epoch: {best_trained['epoch']}")
print(f"Best trained VAL MAE: {best_trained['val_mae']:.3f}")
print(f"Best trained VAL RMSE: {best_trained['val_rmse']:.3f}")

# Our current run ended at the best trained epoch
assert trained_checkpoint["epoch"] == best_trained["epoch"]
assert abs(
    trained_checkpoint["val_mae"] - best_trained["val_mae"]
) < 1e-8

trained_checkpoint["stage"] = "residual_dde_best_trained"
trained_checkpoint["best_trained_epoch"] = best_trained["epoch"]
trained_checkpoint["best_trained_val_mae"] = best_trained["val_mae"]

torch.save(
    trained_checkpoint,
    RESIDUAL_DDE_TRAINED_BEST_PATH,
)

print()
print("Saved:", RESIDUAL_DDE_TRAINED_BEST_PATH)
print("Checkpoint exists:", RESIDUAL_DDE_TRAINED_BEST_PATH.exists())
print()
print(f"Transformer baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print(f"Trained DDE MAE: {best_trained['val_mae']:.3f}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 14.2 — VALIDATION METRICS + PREDICTIONS CSV
# ============================================================

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_VAL_PATH = RESULTS_DIR / "predictions_val.csv"

# Load preserved best-trained DDE model
checkpoint = torch.load(
    RESIDUAL_DDE_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

dde_ef_model.load_state_dict(checkpoint["model_state_dict"])
dde_ef_model.eval()

records = []

with torch.no_grad():
    for batch in dde_val_loader:

        video = batch["video"].to(device, non_blocking=True)
        times = batch["sample_times"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)

        output = dde_ef_model(video, times)
        pred_ef = output["predicted_ef"]

        for i in range(len(true_ef)):
            true = true_ef[i].item()
            pred = pred_ef[i].item()
            error = pred - true

            records.append({
                "video_id": batch["filename"][i],
                "split": "VAL",
                "ground_truth_ef": true,
                "predicted_ef": pred,
                "absolute_error": abs(error),
                "squared_error": error ** 2,
                "model_name": "Residual_DDE_Reliability_Fusion",
            })

predictions_val = pd.DataFrame(records)
predictions_val.to_csv(PREDICTIONS_VAL_PATH, index=False)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

y_true = predictions_val["ground_truth_ef"].to_numpy()
y_pred = predictions_val["predicted_ef"].to_numpy()

mae = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
bias = np.mean(y_pred - y_true)

ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - y_true.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

pearson_r = np.corrcoef(y_true, y_pred)[0, 1]

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert len(predictions_val) == len(val_dataset)
assert np.isfinite(
    predictions_val.select_dtypes(include=np.number)
).all().all()

print("=" * 60)
print("STEP 13 VALIDATION RESULTS")
print("=" * 60)
print(f"Videos: {len(predictions_val)}")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")
print(f"Pearson r: {pearson_r:.3f}")
print(f"Bias: {bias:.3f}")
print()
print("Saved:", PREDICTIONS_VAL_PATH)
print("CSV exists:", PREDICTIONS_VAL_PATH.exists())
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 14.3 — SAVE STEP 13 EXPERIMENT RECORD
# ============================================================

import json

TRAINING_HISTORY_PATH = RESULTS_DIR / "training_history.csv"
METRICS_SUMMARY_PATH = RESULTS_DIR / "metrics_summary.csv"
CONFIG_PATH = RESULTS_DIR / "step13_experiment_config.json"

# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

checkpoint = torch.load(
    RESIDUAL_DDE_TRAINED_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

history_df = pd.DataFrame(checkpoint["history"])

history_df["train_mse"] = history_df["train_rmse"] ** 2
history_df["dde_lr"] = DDE_LR
history_df["residual_lr"] = RESIDUAL_LR
history_df["weight_decay"] = DDE_WEIGHT_DECAY
history_df["checkpoint_epoch"] = history_df["epoch"]

history_df.to_csv(TRAINING_HISTORY_PATH, index=False)

# ------------------------------------------------------------
# Metrics summary
# ------------------------------------------------------------

metrics_row = pd.DataFrame([{
    "model": "Residual_DDE_Reliability_Fusion",
    "split": "VAL",
    "mae": mae,
    "rmse": rmse,
    "r2": r2,
    "pearson_r": pearson_r,
    "bias": bias,
    "seed": SEED,
    "checkpoint": str(RESIDUAL_DDE_TRAINED_BEST_PATH),
}])

if METRICS_SUMMARY_PATH.exists():
    old_metrics = pd.read_csv(METRICS_SUMMARY_PATH)
    old_metrics = old_metrics[
        ~(
            (old_metrics["model"] == "Residual_DDE_Reliability_Fusion")
            & (old_metrics["split"] == "VAL")
        )
    ]
    metrics_row = pd.concat([old_metrics, metrics_row], ignore_index=True)

metrics_row.to_csv(METRICS_SUMMARY_PATH, index=False)

# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

config = {
    "experiment": "Step 13 - Residual Reliability-Aware Neural DDE",
    "seed": SEED,
    "train_videos": len(dde_train_dataset),
    "validation_videos": len(val_dataset),
    "excluded_short_train_videos": 3,
    "num_sampled_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,

    "backbone": "ConvNeXt-Tiny + Temporal Transformer",
    "backbone_frozen": True,
    "transformer_baseline_mae": TRANSFORMER_BASELINE_MAE,

    "dde_latent_dim": 256,
    "dde_hidden_dim": 512,
    "reliability_features": ["Q", "M", "B", "C"],

    "delay_method":
        "Estimated cycle lag converted to seconds using sample timestamps; "
        "physical delay is median time difference at the estimated lag.",

    "time_units": "seconds",
    "numerical_update": "Explicit Euler",
    "delayed_state_method": "Linear interpolation over prior latent states",

    "batch_size": DDE_BATCH_SIZE,
    "max_epochs": DDE_MAX_EPOCHS,
    "early_stopping_patience": DDE_PATIENCE,
    "dde_learning_rate": DDE_LR,
    "residual_learning_rate": RESIDUAL_LR,
    "weight_decay": DDE_WEIGHT_DECAY,

    "optimizer": "AdamW",
    "loss": "MSE",
    "amp_enabled": USE_AMP,

    "best_trained_epoch": int(checkpoint["best_trained_epoch"]),
    "best_trained_val_mae": float(mae),
    "best_trained_val_rmse": float(rmse),

    "checkpoint": str(RESIDUAL_DDE_TRAINED_BEST_PATH),
}

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=4)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("=" * 60)
print("STEP 13 EXPERIMENT RECORD")
print("=" * 60)
print(f"Training history rows: {len(history_df)}")
print(f"Metrics summary saved: {METRICS_SUMMARY_PATH.exists()}")
print(f"Training history saved: {TRAINING_HISTORY_PATH.exists()}")
print(f"Experiment config saved: {CONFIG_PATH.exists()}")
print()
print("Files:")
print(" -", METRICS_SUMMARY_PATH)
print(" -", TRAINING_HISTORY_PATH)
print(" -", CONFIG_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 14.4 — TRANSFORMER BASELINE PREDICTIONS
# ============================================================

best_transformer_model.eval()
records = []

with torch.no_grad():
    for batch in dde_val_loader:
        video = batch["video"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device)

        frame_features = best_transformer_model.spatial_encoder(video)
        _, video_feature = best_transformer_model.temporal_transformer(frame_features)
        pred_ef = best_transformer_model.ef_head(video_feature)

        for i in range(len(true_ef)):
            true = true_ef[i].item()
            pred = pred_ef[i].item()
            error = pred - true

            records.append({
                "video_id": batch["filename"][i],
                "split": "VAL",
                "ground_truth_ef": true,
                "predicted_ef": pred,
                "absolute_error": abs(error),
                "squared_error": error ** 2,
                "model_name": "Temporal_Transformer",
            })

transformer_preds = pd.DataFrame(records)

y_true = transformer_preds["ground_truth_ef"].to_numpy()
y_pred = transformer_preds["predicted_ef"].to_numpy()

t_mae = np.mean(np.abs(y_pred - y_true))
t_rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
t_bias = np.mean(y_pred - y_true)
t_r = np.corrcoef(y_true, y_pred)[0, 1]
t_r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

# Append to per-video prediction file
all_preds = pd.read_csv(PREDICTIONS_VAL_PATH)
all_preds = all_preds[all_preds["model_name"] != "Temporal_Transformer"]
all_preds = pd.concat([all_preds, transformer_preds], ignore_index=True)
all_preds.to_csv(PREDICTIONS_VAL_PATH, index=False)

# Append to metrics summary
metrics = pd.read_csv(METRICS_SUMMARY_PATH)
metrics = metrics[
    ~((metrics["model"] == "Temporal_Transformer") & (metrics["split"] == "VAL"))
]

transformer_row = pd.DataFrame([{
    "model": "Temporal_Transformer",
    "split": "VAL",
    "mae": t_mae,
    "rmse": t_rmse,
    "r2": t_r2,
    "pearson_r": t_r,
    "bias": t_bias,
    "seed": SEED,
    "checkpoint": str(TRANSFORMER_E2E_BEST_PATH),
}])

pd.concat([metrics, transformer_row], ignore_index=True).to_csv(
    METRICS_SUMMARY_PATH, index=False
)

print("=" * 60)
print("TRANSFORMER BASELINE VALIDATION RESULTS")
print("=" * 60)
print(f"Videos: {len(transformer_preds)}")
print(f"MAE: {t_mae:.3f}")
print(f"RMSE: {t_rmse:.3f}")
print(f"R²: {t_r2:.3f}")
print(f"Pearson r: {t_r:.3f}")
print(f"Bias: {t_bias:.3f}")
print(f"Total prediction rows saved: {len(all_preds)}")
print("=" * 60)

assert len(transformer_preds) == 1288
assert abs(t_mae - TRANSFORMER_BASELINE_MAE) < 0.01

# 15...

In [ ]:
# ============================================================
# SECTION 15.1 — INITIALIZE ABLATION RESULTS
# ============================================================

ABLATION_RESULTS_PATH = RESULTS_DIR / "ablation_results.csv"

metrics = pd.read_csv(METRICS_SUMMARY_PATH)

baseline = metrics[
    (metrics["model"] == "Temporal_Transformer") &
    (metrics["split"] == "VAL")
].iloc[0]

full = metrics[
    (metrics["model"] == "Residual_DDE_Reliability_Fusion") &
    (metrics["split"] == "VAL")
].iloc[0]

ablation_results = pd.DataFrame([
    {
        "model_variant": "Temporal Transformer baseline",
        "mae": baseline["mae"],
        "rmse": baseline["rmse"],
        "r2": baseline["r2"],
        "pearson_r": baseline["pearson_r"],
        "bias": baseline["bias"],
        "seed": SEED,
    },
    {
        "model_variant": "Full DDE + reliability + adaptive fusion",
        "mae": full["mae"],
        "rmse": full["rmse"],
        "r2": full["r2"],
        "pearson_r": full["pearson_r"],
        "bias": full["bias"],
        "seed": SEED,
    },
])

ablation_results.to_csv(ABLATION_RESULTS_PATH, index=False)

print("=" * 60)
print("ABLATION STUDY INITIALIZED")
print("=" * 60)
print(ablation_results.to_string(index=False))
print()
print("Saved:", ABLATION_RESULTS_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.2 — DDE WITHOUT RELIABILITY CONDITIONING
# ============================================================

class DDEWithoutReliabilityModel(nn.Module):
    """
    Same residual Neural DDE architecture, but Q/M/B/C are
    replaced by zeros. Cycle lag is retained only to define
    the delayed state tau.
    """

    def __init__(
        self,
        backbone,
        reliability_extractor,
        neural_dde,
        transformer_dim=768,
        dde_dim=256,
    ):
        super().__init__()

        self.backbone = backbone
        self.reliability_extractor = reliability_extractor
        self.neural_dde = neural_dde

        self.residual_fusion = ReliabilityAwareResidualFusion(
            transformer_dim=transformer_dim,
            dde_dim=dde_dim,
            reliability_dim=4,
            hidden_dim=256,
            dropout=0.1,
        )

    def forward(self, video, sample_times):

        frame_features = self.backbone.spatial_encoder(video)

        sequence_features, transformer_feature = (
            self.backbone.temporal_transformer(frame_features)
        )

        baseline_ef = self.backbone.ef_head(transformer_feature)

        # Extract cycle lag, but remove Q/M/B/C conditioning
        reliability, cycle_lags = self.reliability_extractor(
            video,
            sequence_features,
        )

        zero_reliability = torch.zeros_like(reliability)

        dde_sequence, dde_feature, delay_seconds = self.neural_dde(
            sequence_features,
            zero_reliability,
            sample_times,
            cycle_lags,
        )

        zero_summary = zero_reliability.mean(dim=1)

        correction, gate, raw_residual = self.residual_fusion(
            transformer_feature,
            dde_feature,
            zero_summary,
        )

        predicted_ef = baseline_ef + correction

        return {
            "predicted_ef": predicted_ef,
            "baseline_ef": baseline_ef,
            "correction": correction,
            "fusion_gate": gate,
            "dde_sequence": dde_sequence,
            "cycle_lags": cycle_lags,
            "delay_seconds": delay_seconds,
        }

In [ ]:
# ============================================================
# SECTION 15.3 — BUILD + VERIFY DDE WITHOUT RELIABILITY
# ============================================================

set_seed(SEED)

ablation_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

no_rel_model = DDEWithoutReliabilityModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=ablation_dde,
).to(device)

# Freeze pretrained backbone
for p in no_rel_model.backbone.parameters():
    p.requires_grad = False

no_rel_model.reliability_extractor.eval()
no_rel_model.eval()

# Forward check
batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = no_rel_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]
max_diff = (predicted - baseline).abs().max().item()

trainable_params = sum(
    p.numel() for p in no_rel_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("DDE WITHOUT RELIABILITY — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print("Baseline EF:", baseline.squeeze(1).cpu().tolist())
print("Final EF:", predicted.squeeze(1).cpu().tolist())
print("Correction:", output["correction"].squeeze(1).cpu().tolist())
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()

print("\nAblation model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.4 — NO-RELIABILITY DDE TRAINING SETUP
# ============================================================

NO_REL_BEST_PATH = E2E_RUN_DIR / "05_dde_without_reliability_best.pth"
NO_REL_LATEST_PATH = E2E_RUN_DIR / "05_dde_without_reliability_latest.pth"

no_rel_optimizer = torch.optim.AdamW(
    [
        {"params": no_rel_model.neural_dde.parameters(), "lr": DDE_LR},
        {"params": no_rel_model.residual_fusion.parameters(), "lr": RESIDUAL_LR},
    ],
    weight_decay=DDE_WEIGHT_DECAY,
)

no_rel_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

no_rel_start_epoch = 0
no_rel_best_val_mae = TRANSFORMER_BASELINE_MAE
no_rel_no_improve = 0
no_rel_history = []

# Save epoch-0 Transformer-equivalent starting point
initial_checkpoint = {
    "epoch": 0,
    "model_state_dict": no_rel_model.state_dict(),
    "optimizer_state_dict": no_rel_optimizer.state_dict(),
    "scaler_state_dict": no_rel_scaler.state_dict(),
    "best_val_mae": TRANSFORMER_BASELINE_MAE,
    "history": [],
    "stage": "dde_without_reliability_baseline",
}

torch.save(initial_checkpoint, NO_REL_BEST_PATH)

print("=" * 60)
print("DDE WITHOUT RELIABILITY — TRAINING SETUP")
print("=" * 60)
print(f"TRAIN videos: {len(dde_train_dataset)}")
print(f"TRAIN batches: {len(dde_train_loader)}")
print(f"VAL videos: {len(val_dataset)}")
print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print(f"DDE LR: {DDE_LR}")
print(f"Residual LR: {RESIDUAL_LR}")
print(f"Initial BEST saved: {NO_REL_BEST_PATH.exists()}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.5 — TRAIN DDE WITHOUT RELIABILITY
# ============================================================

NO_REL_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "05_dde_without_reliability_best_trained.pth"
)

# Fresh seeded loader for controlled comparison
no_rel_generator = torch.Generator().manual_seed(SEED)

no_rel_train_loader = DataLoader(
    dde_train_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=USE_AMP,
    generator=no_rel_generator,
)

criterion = nn.MSELoss()
best_trained_mae = float("inf")

torch.cuda.empty_cache()

print("=" * 60)
print("DDE WITHOUT RELIABILITY — TRAINING")
print("=" * 60)

for epoch in range(DDE_MAX_EPOCHS):

    # ---------------- TRAIN ----------------
    no_rel_model.train()
    no_rel_model.backbone.eval()
    no_rel_model.reliability_extractor.eval()

    train_sq, train_n = 0.0, 0

    for batch_idx, batch in enumerate(no_rel_train_loader):

        video = batch["video"].to(device, non_blocking=True)
        times = batch["sample_times"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)

        no_rel_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=USE_AMP,
        ):
            output = no_rel_model(video, times)
            pred = output["predicted_ef"]
            loss = criterion(pred, true_ef)

        no_rel_scaler.scale(loss).backward()
        no_rel_scaler.step(no_rel_optimizer)
        no_rel_scaler.update()

        error = pred.detach().float() - true_ef.float()
        train_sq += error.pow(2).sum().item()
        train_n += true_ef.numel()

        if (batch_idx + 1) % 200 == 0:
            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_idx + 1}/{len(no_rel_train_loader)} | "
                f"Train RMSE: {math.sqrt(train_sq / train_n):.3f}"
            )

    train_rmse = math.sqrt(train_sq / train_n)

    # ---------------- VALIDATION ----------------
    no_rel_model.eval()

    val_abs, val_sq, val_n = 0.0, 0.0, 0
    gate_sum, correction_sum = 0.0, 0.0

    with torch.no_grad():
        for batch in dde_val_loader:

            video = batch["video"].to(device, non_blocking=True)
            times = batch["sample_times"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)

            output = no_rel_model(video, times)
            pred = output["predicted_ef"]

            error = pred.float() - true_ef.float()

            val_abs += error.abs().sum().item()
            val_sq += error.pow(2).sum().item()
            val_n += true_ef.numel()

            gate_sum += output["fusion_gate"].sum().item()
            correction_sum += output["correction"].abs().sum().item()

    val_mae = val_abs / val_n
    val_rmse = math.sqrt(val_sq / val_n)
    mean_gate = gate_sum / val_n
    mean_correction = correction_sum / val_n

    # ---------------- MODEL SELECTION ----------------
    improved_baseline = val_mae < no_rel_best_val_mae
    improved_trained = val_mae < best_trained_mae

    if improved_baseline:
        no_rel_best_val_mae = val_mae
        no_rel_no_improve = 0
    else:
        no_rel_no_improve += 1

    if improved_trained:
        best_trained_mae = val_mae

    no_rel_history.append({
        "epoch": epoch + 1,
        "train_rmse": train_rmse,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "mean_gate": mean_gate,
        "mean_abs_correction": mean_correction,
    })

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": no_rel_model.state_dict(),
        "optimizer_state_dict": no_rel_optimizer.state_dict(),
        "scaler_state_dict": no_rel_scaler.state_dict(),
        "best_val_mae": no_rel_best_val_mae,
        "best_trained_mae": best_trained_mae,
        "epochs_without_improvement": no_rel_no_improve,
        "history": no_rel_history,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
    }

    torch.save(checkpoint, NO_REL_LATEST_PATH)

    if improved_baseline:
        torch.save(checkpoint, NO_REL_BEST_PATH)

    if improved_trained:
        torch.save(checkpoint, NO_REL_TRAINED_BEST_PATH)

    print()
    print(f"No-Reliability DDE Epoch {epoch + 1}/{DDE_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")
    print(f"Mean gate: {mean_gate:.3f}")
    print(f"Mean |EF correction|: {mean_correction:.3f}")
    print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
    print(f"Best trained MAE: {best_trained_mae:.3f}")
    print(f"No-improvement epochs: {no_rel_no_improve}")
    print("-" * 60)

    if no_rel_no_improve >= DDE_PATIENCE:
        print("Early stopping triggered.")
        break

In [ ]:
# ============================================================
# SECTION 15.5A — RESTORE NO-RELIABILITY DDE RUN
# ============================================================

checkpoint = torch.load(
    NO_REL_LATEST_PATH,
    map_location=device,
    weights_only=False,
)

no_rel_model.load_state_dict(checkpoint["model_state_dict"])
no_rel_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
no_rel_scaler.load_state_dict(checkpoint["scaler_state_dict"])

no_rel_start_epoch = checkpoint["epoch"]
no_rel_best_val_mae = checkpoint["best_val_mae"]
best_trained_mae = checkpoint["best_trained_mae"]
no_rel_no_improve = checkpoint["epochs_without_improvement"]
no_rel_history = checkpoint["history"]

print("=" * 60)
print("NO-RELIABILITY DDE RUN RESTORED")
print("=" * 60)
print(f"Completed epochs: {no_rel_start_epoch}")
print(f"Best trained MAE: {best_trained_mae:.3f}")
print(f"Best overall MAE: {no_rel_best_val_mae:.3f}")
print(f"No-improvement epochs: {no_rel_no_improve}")
print(f"Resume from epoch: {no_rel_start_epoch + 1}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.6 — SAVE NO-RELIABILITY ABLATION RESULTS
# ============================================================

NO_REL_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "05_dde_without_reliability_best_trained.pth"
)

assert NO_REL_TRAINED_BEST_PATH.exists(), (
    f"Missing checkpoint: {NO_REL_TRAINED_BEST_PATH}"
)

checkpoint = torch.load(
    NO_REL_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

no_rel_model.load_state_dict(checkpoint["model_state_dict"])
no_rel_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:
        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = no_rel_model(video, times)["predicted_ef"]

        true_vals.extend(true_ef.squeeze(1).cpu().tolist())
        pred_vals.extend(pred.squeeze(1).cpu().tolist())

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

nr_mae = np.mean(np.abs(y_pred - y_true))
nr_rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
nr_bias = np.mean(y_pred - y_true)
nr_r = np.corrcoef(y_true, y_pred)[0, 1]

nr_r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

# Add/update ablation table
ablation_results = pd.read_csv(ABLATION_RESULTS_PATH)

ablation_results = ablation_results[
    ablation_results["model_variant"] != "Neural DDE without reliability"
]

new_row = pd.DataFrame([{
    "model_variant": "Neural DDE without reliability",
    "mae": nr_mae,
    "rmse": nr_rmse,
    "r2": nr_r2,
    "pearson_r": nr_r,
    "bias": nr_bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("DDE WITHOUT RELIABILITY — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {nr_mae:.3f}")
print(f"RMSE: {nr_rmse:.3f}")
print(f"R²: {nr_r2:.3f}")
print(f"Pearson r: {nr_r:.3f}")
print(f"Bias: {nr_bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.7 — DDE WITHOUT ADAPTIVE FUSION
# ============================================================

class ResidualWithoutAdaptiveFusion(nn.Module):
    """
    Reliability-conditioned residual correction without
    a learned fusion gate.
    """

    def __init__(
        self,
        transformer_dim=768,
        dde_dim=256,
        reliability_dim=4,
        hidden_dim=256,
        dropout=0.1,
    ):
        super().__init__()

        input_dim = transformer_dim + dde_dim + reliability_dim

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
        )

        self.residual_head = nn.Linear(hidden_dim, 1)

        # Start exactly at Transformer baseline
        nn.init.zeros_(self.residual_head.weight)
        nn.init.zeros_(self.residual_head.bias)

    def forward(
        self,
        transformer_feature,
        dde_feature,
        reliability_summary,
    ):
        x = torch.cat(
            [transformer_feature, dde_feature, reliability_summary],
            dim=1,
        )

        x = self.network(x)
        correction = self.residual_head(x)

        return correction


class DDEWithoutAdaptiveFusionModel(nn.Module):
    """
    Reliability-conditioned Neural DDE with direct residual
    correction and no learned adaptive fusion gate.
    """

    def __init__(
        self,
        backbone,
        reliability_extractor,
        neural_dde,
    ):
        super().__init__()

        self.backbone = backbone
        self.reliability_extractor = reliability_extractor
        self.neural_dde = neural_dde

        self.residual_head = ResidualWithoutAdaptiveFusion()

    def forward(self, video, sample_times):

        frame_features = self.backbone.spatial_encoder(video)

        sequence_features, transformer_feature = (
            self.backbone.temporal_transformer(frame_features)
        )

        baseline_ef = self.backbone.ef_head(transformer_feature)

        reliability, cycle_lags = self.reliability_extractor(
            video,
            sequence_features,
        )

        dde_sequence, dde_feature, delay_seconds = self.neural_dde(
            sequence_features,
            reliability,
            sample_times,
            cycle_lags,
        )

        reliability_summary = reliability.mean(dim=1)

        correction = self.residual_head(
            transformer_feature,
            dde_feature,
            reliability_summary,
        )

        predicted_ef = baseline_ef + correction

        return {
            "predicted_ef": predicted_ef,
            "baseline_ef": baseline_ef,
            "correction": correction,
            "dde_sequence": dde_sequence,
            "reliability": reliability,
            "cycle_lags": cycle_lags,
            "delay_seconds": delay_seconds,
        }

In [ ]:
# ============================================================
# SECTION 15.8 — BUILD + VERIFY WITHOUT ADAPTIVE FUSION
# ============================================================

set_seed(SEED)

no_fusion_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

no_fusion_model = DDEWithoutAdaptiveFusionModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=no_fusion_dde,
).to(device)

# Freeze pretrained backbone
for p in no_fusion_model.backbone.parameters():
    p.requires_grad = False

no_fusion_model.reliability_extractor.eval()
no_fusion_model.eval()

# Forward check
batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = no_fusion_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]
max_diff = (predicted - baseline).abs().max().item()

trainable_params = sum(
    p.numel() for p in no_fusion_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("DDE WITHOUT ADAPTIVE FUSION — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print("Baseline EF:", baseline.squeeze(1).cpu().tolist())
print("Final EF:", predicted.squeeze(1).cpu().tolist())
print("Correction:", output["correction"].squeeze(1).cpu().tolist())
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()

print("\nModel exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.9 — NO-FUSION DDE TRAINING SETUP
# ============================================================

NO_FUSION_BEST_PATH = (
    E2E_RUN_DIR / "06_dde_without_adaptive_fusion_best.pth"
)

NO_FUSION_LATEST_PATH = (
    E2E_RUN_DIR / "06_dde_without_adaptive_fusion_latest.pth"
)

NO_FUSION_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "06_dde_without_adaptive_fusion_best_trained.pth"
)

no_fusion_optimizer = torch.optim.AdamW(
    [
        {"params": no_fusion_model.neural_dde.parameters(), "lr": DDE_LR},
        {"params": no_fusion_model.residual_head.parameters(), "lr": RESIDUAL_LR},
    ],
    weight_decay=DDE_WEIGHT_DECAY,
)

no_fusion_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

no_fusion_generator = torch.Generator().manual_seed(SEED)

no_fusion_train_loader = DataLoader(
    dde_train_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=USE_AMP,
    generator=no_fusion_generator,
)

no_fusion_best_mae = TRANSFORMER_BASELINE_MAE
no_fusion_best_trained_mae = float("inf")
no_fusion_no_improve = 0
no_fusion_history = []

# Save exact Transformer-equivalent starting point
torch.save(
    {
        "epoch": 0,
        "model_state_dict": no_fusion_model.state_dict(),
        "optimizer_state_dict": no_fusion_optimizer.state_dict(),
        "scaler_state_dict": no_fusion_scaler.state_dict(),
        "best_val_mae": TRANSFORMER_BASELINE_MAE,
        "history": [],
        "stage": "dde_without_adaptive_fusion_baseline",
    },
    NO_FUSION_BEST_PATH,
)

print("=" * 60)
print("DDE WITHOUT ADAPTIVE FUSION — TRAINING SETUP")
print("=" * 60)
print(f"TRAIN videos: {len(dde_train_dataset)}")
print(f"TRAIN batches: {len(no_fusion_train_loader)}")
print(f"VAL videos: {len(val_dataset)}")
print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print(f"DDE LR: {DDE_LR}")
print(f"Residual LR: {RESIDUAL_LR}")
print(f"Initial BEST saved: {NO_FUSION_BEST_PATH.exists()}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.10 — TRAIN DDE WITHOUT ADAPTIVE FUSION
# ============================================================

criterion = nn.MSELoss()

start_epoch = 0

# Resume safely after disconnect
if NO_FUSION_LATEST_PATH.exists():
    ckpt = torch.load(
        NO_FUSION_LATEST_PATH,
        map_location=device,
        weights_only=False,
    )

    no_fusion_model.load_state_dict(ckpt["model_state_dict"])
    no_fusion_optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    no_fusion_scaler.load_state_dict(ckpt["scaler_state_dict"])

    start_epoch = ckpt["epoch"]
    no_fusion_best_mae = ckpt["best_val_mae"]
    no_fusion_best_trained_mae = ckpt["best_trained_mae"]
    no_fusion_no_improve = ckpt["epochs_without_improvement"]
    no_fusion_history = ckpt["history"]

    if "train_generator_state" in ckpt:
        no_fusion_generator.set_state(ckpt["train_generator_state"].cpu())

    print(f"Resuming from epoch {start_epoch + 1}")

torch.cuda.empty_cache()

print("=" * 60)
print("DDE WITHOUT ADAPTIVE FUSION — TRAINING")
print("=" * 60)

for epoch in range(start_epoch, DDE_MAX_EPOCHS):

    # ---------------- TRAIN ----------------
    no_fusion_model.train()
    no_fusion_model.backbone.eval()
    no_fusion_model.reliability_extractor.eval()

    train_sq, train_n = 0.0, 0

    for batch_idx, batch in enumerate(no_fusion_train_loader):

        video = batch["video"].to(device, non_blocking=True)
        times = batch["sample_times"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)

        no_fusion_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=USE_AMP,
        ):
            output = no_fusion_model(video, times)
            pred = output["predicted_ef"]
            loss = criterion(pred, true_ef)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss detected.")

        no_fusion_scaler.scale(loss).backward()
        no_fusion_scaler.step(no_fusion_optimizer)
        no_fusion_scaler.update()

        error = pred.detach().float() - true_ef.float()
        train_sq += error.pow(2).sum().item()
        train_n += true_ef.numel()

        if (batch_idx + 1) % 200 == 0:
            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_idx + 1}/{len(no_fusion_train_loader)} | "
                f"Train RMSE: {math.sqrt(train_sq / train_n):.3f}"
            )

    train_rmse = math.sqrt(train_sq / train_n)

    # ---------------- VALIDATION ----------------
    no_fusion_model.eval()

    val_abs, val_sq, val_n = 0.0, 0.0, 0
    correction_sum = 0.0

    with torch.no_grad():
        for batch in dde_val_loader:

            video = batch["video"].to(device, non_blocking=True)
            times = batch["sample_times"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)

            output = no_fusion_model(video, times)
            pred = output["predicted_ef"]

            error = pred.float() - true_ef.float()

            val_abs += error.abs().sum().item()
            val_sq += error.pow(2).sum().item()
            val_n += true_ef.numel()

            correction_sum += (
                output["correction"].abs().sum().item()
            )

    val_mae = val_abs / val_n
    val_rmse = math.sqrt(val_sq / val_n)
    mean_correction = correction_sum / val_n

    # ---------------- MODEL SELECTION ----------------
    improved_baseline = val_mae < no_fusion_best_mae
    improved_trained = val_mae < no_fusion_best_trained_mae

    if improved_baseline:
        no_fusion_best_mae = val_mae
        no_fusion_no_improve = 0
    else:
        no_fusion_no_improve += 1

    if improved_trained:
        no_fusion_best_trained_mae = val_mae

    no_fusion_history.append({
        "epoch": epoch + 1,
        "train_rmse": train_rmse,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "mean_abs_correction": mean_correction,
    })

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": no_fusion_model.state_dict(),
        "optimizer_state_dict": no_fusion_optimizer.state_dict(),
        "scaler_state_dict": no_fusion_scaler.state_dict(),
        "best_val_mae": no_fusion_best_mae,
        "best_trained_mae": no_fusion_best_trained_mae,
        "epochs_without_improvement": no_fusion_no_improve,
        "history": no_fusion_history,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "train_generator_state": no_fusion_generator.get_state(),
    }

    torch.save(checkpoint, NO_FUSION_LATEST_PATH)

    if improved_baseline:
        torch.save(checkpoint, NO_FUSION_BEST_PATH)

    if improved_trained:
        torch.save(checkpoint, NO_FUSION_TRAINED_BEST_PATH)

    print()
    print(f"No-Fusion DDE Epoch {epoch + 1}/{DDE_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")
    print(f"Mean |EF correction|: {mean_correction:.3f}")
    print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
    print(f"Best trained MAE: {no_fusion_best_trained_mae:.3f}")
    print(f"No-improvement epochs: {no_fusion_no_improve}")
    print("-" * 60)

    if no_fusion_no_improve >= DDE_PATIENCE:
        print("Early stopping triggered.")
        break

In [ ]:
# ============================================================
# SECTION 15.11 — SAVE NO-FUSION ABLATION RESULTS
# ============================================================

checkpoint = torch.load(
    NO_FUSION_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

no_fusion_model.load_state_dict(checkpoint["model_state_dict"])
no_fusion_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:
        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = no_fusion_model(video, times)["predicted_ef"]

        true_vals.extend(true_ef.squeeze(1).cpu().tolist())
        pred_vals.extend(pred.squeeze(1).cpu().tolist())

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

nf_mae = np.mean(np.abs(y_pred - y_true))
nf_rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
nf_bias = np.mean(y_pred - y_true)
nf_r = np.corrcoef(y_true, y_pred)[0, 1]

nf_r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

ablation_results = pd.read_csv(ABLATION_RESULTS_PATH)

ablation_results = ablation_results[
    ablation_results["model_variant"]
    != "Reliability-conditioned DDE without adaptive fusion"
]

new_row = pd.DataFrame([{
    "model_variant":
        "Reliability-conditioned DDE without adaptive fusion",
    "mae": nf_mae,
    "rmse": nf_rmse,
    "r2": nf_r2,
    "pearson_r": nf_r,
    "bias": nf_bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("DDE WITHOUT ADAPTIVE FUSION — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {nf_mae:.3f}")
print(f"RMSE: {nf_rmse:.3f}")
print(f"R²: {nf_r2:.3f}")
print(f"Pearson r: {nf_r:.3f}")
print(f"Bias: {nf_bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.12 — PARAMETER-MATCHED NEURAL ODE
# ============================================================

class ReliabilityConditionedNeuralODE(nn.Module):
    """
    No-delay control for the Neural DDE.

    Uses the same latent dimensions and dynamics network, but
    removes delayed-state information by replacing z(t-tau)
    with zeros.
    """

    def __init__(
        self,
        input_dim=768,
        latent_dim=256,
        reliability_dim=4,
        hidden_dim=512,
        dropout=0.1,
    ):
        super().__init__()

        self.latent_dim = latent_dim

        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.GELU(),
        )

        self.dynamics = ReliabilityConditionedDDEDynamics(
            latent_dim=latent_dim,
            reliability_dim=reliability_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.final_norm = nn.LayerNorm(latent_dim)

    def forward(
        self,
        sequence_features,
        reliability,
        sample_times,
    ):
        B, T, _ = sequence_features.shape

        delta_t = sample_times[:, 1:] - sample_times[:, :-1]

        if not torch.all(delta_t > 0):
            raise ValueError("Sample times must be strictly increasing.")

        x = self.input_projection(sequence_features)
        z_current = x[:, 0]

        states = [z_current]

        for i in range(1, T):

            dt = sample_times[:, i] - sample_times[:, i - 1]

            # No delayed-state information
            z_no_delay = torch.zeros_like(z_current)

            dz_dt = self.dynamics(
                z_current,
                z_no_delay,
                x[:, i],
                reliability[:, i],
            )

            z_current = z_current + dt.unsqueeze(-1) * dz_dt
            states.append(z_current)

        ode_sequence = torch.stack(states, dim=1)
        ode_sequence = self.final_norm(ode_sequence)

        video_feature = ode_sequence.mean(dim=1)

        return ode_sequence, video_feature

In [ ]:
# ============================================================
# SECTION 15.13 — RELIABILITY-CONDITIONED ODE EF MODEL
# ============================================================

class ReliabilityAwareODEEFModel(nn.Module):

    def __init__(
        self,
        backbone,
        reliability_extractor,
        neural_ode,
    ):
        super().__init__()

        self.backbone = backbone
        self.reliability_extractor = reliability_extractor
        self.neural_ode = neural_ode

        self.residual_fusion = ReliabilityAwareResidualFusion(
            transformer_dim=768,
            dde_dim=256,
            reliability_dim=4,
            hidden_dim=256,
            dropout=0.1,
        )

    def forward(self, video, sample_times):

        frame_features = self.backbone.spatial_encoder(video)

        sequence_features, transformer_feature = (
            self.backbone.temporal_transformer(frame_features)
        )

        baseline_ef = self.backbone.ef_head(transformer_feature)

        reliability, _ = self.reliability_extractor(
            video,
            sequence_features,
        )

        ode_sequence, ode_feature = self.neural_ode(
            sequence_features,
            reliability,
            sample_times,
        )

        reliability_summary = reliability.mean(dim=1)

        correction, gate, raw_residual = self.residual_fusion(
            transformer_feature,
            ode_feature,
            reliability_summary,
        )

        predicted_ef = baseline_ef + correction

        return {
            "predicted_ef": predicted_ef,
            "baseline_ef": baseline_ef,
            "correction": correction,
            "fusion_gate": gate,
            "ode_sequence": ode_sequence,
            "reliability": reliability,
        }

In [ ]:
# ============================================================
# SECTION 15.14 — BUILD + VERIFY NEURAL ODE MODEL
# ============================================================

set_seed(SEED)

neural_ode = ReliabilityConditionedNeuralODE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

ode_ef_model = ReliabilityAwareODEEFModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_ode=neural_ode,
).to(device)

# Freeze pretrained backbone
for p in ode_ef_model.backbone.parameters():
    p.requires_grad = False

ode_ef_model.reliability_extractor.eval()
ode_ef_model.eval()

# Forward check
batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = ode_ef_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]
max_diff = (predicted - baseline).abs().max().item()

trainable_params = sum(
    p.numel()
    for p in ode_ef_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("NEURAL ODE — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print("Baseline EF:", baseline.squeeze(1).cpu().tolist())
print("Final EF:", predicted.squeeze(1).cpu().tolist())
print("Correction:", output["correction"].squeeze(1).cpu().tolist())
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["ode_sequence"]).all()

print("\nNeural ODE model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.15 — NEURAL ODE TRAINING SETUP
# ============================================================

ODE_BEST_PATH = E2E_RUN_DIR / "07_neural_ode_best.pth"
ODE_LATEST_PATH = E2E_RUN_DIR / "07_neural_ode_latest.pth"
ODE_TRAINED_BEST_PATH = E2E_RUN_DIR / "07_neural_ode_best_trained.pth"

ode_optimizer = torch.optim.AdamW(
    [
        {"params": ode_ef_model.neural_ode.parameters(), "lr": DDE_LR},
        {"params": ode_ef_model.residual_fusion.parameters(), "lr": RESIDUAL_LR},
    ],
    weight_decay=DDE_WEIGHT_DECAY,
)

ode_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
ode_generator = torch.Generator().manual_seed(SEED)

ode_train_loader = DataLoader(
    dde_train_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=USE_AMP,
    generator=ode_generator,
)

ode_best_mae = TRANSFORMER_BASELINE_MAE
ode_best_trained_mae = float("inf")
ode_no_improve = 0
ode_history = []

# Exact Transformer-equivalent epoch-0 checkpoint
if not ODE_BEST_PATH.exists():
    torch.save(
        {
            "epoch": 0,
            "model_state_dict": ode_ef_model.state_dict(),
            "optimizer_state_dict": ode_optimizer.state_dict(),
            "scaler_state_dict": ode_scaler.state_dict(),
            "best_val_mae": TRANSFORMER_BASELINE_MAE,
            "history": [],
            "stage": "neural_ode_baseline",
        },
        ODE_BEST_PATH,
    )

print("=" * 60)
print("NEURAL ODE — TRAINING SETUP")
print("=" * 60)
print(f"TRAIN videos: {len(dde_train_dataset)}")
print(f"TRAIN batches: {len(ode_train_loader)}")
print(f"VAL videos: {len(val_dataset)}")
print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print(f"ODE LR: {DDE_LR}")
print(f"Residual LR: {RESIDUAL_LR}")
print(f"Initial BEST saved: {ODE_BEST_PATH.exists()}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 15.16 — TRAIN NEURAL ODE
# ============================================================

criterion = nn.MSELoss()
start_epoch = 0

# Resume safely if interrupted
if ODE_LATEST_PATH.exists():
    ckpt = torch.load(
        ODE_LATEST_PATH,
        map_location=device,
        weights_only=False,
    )

    ode_ef_model.load_state_dict(ckpt["model_state_dict"])
    ode_optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    ode_scaler.load_state_dict(ckpt["scaler_state_dict"])

    start_epoch = ckpt["epoch"]
    ode_best_mae = ckpt["best_val_mae"]
    ode_best_trained_mae = ckpt["best_trained_mae"]
    ode_no_improve = ckpt["epochs_without_improvement"]
    ode_history = ckpt["history"]

    if "train_generator_state" in ckpt:
        ode_generator.set_state(ckpt["train_generator_state"].cpu())

    print(f"Resuming from epoch {start_epoch + 1}")

torch.cuda.empty_cache()

print("=" * 60)
print("NEURAL ODE — TRAINING")
print("=" * 60)

for epoch in range(start_epoch, DDE_MAX_EPOCHS):

    # ---------------- TRAIN ----------------
    ode_ef_model.train()
    ode_ef_model.backbone.eval()
    ode_ef_model.reliability_extractor.eval()

    train_sq, train_n = 0.0, 0

    for batch_idx, batch in enumerate(ode_train_loader):

        video = batch["video"].to(device, non_blocking=True)
        times = batch["sample_times"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)

        ode_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=USE_AMP,
        ):
            output = ode_ef_model(video, times)
            pred = output["predicted_ef"]
            loss = criterion(pred, true_ef)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss detected.")

        ode_scaler.scale(loss).backward()
        ode_scaler.step(ode_optimizer)
        ode_scaler.update()

        error = pred.detach().float() - true_ef.float()
        train_sq += error.pow(2).sum().item()
        train_n += true_ef.numel()

        if (batch_idx + 1) % 200 == 0:
            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_idx + 1}/{len(ode_train_loader)} | "
                f"Train RMSE: {math.sqrt(train_sq / train_n):.3f}"
            )

    train_rmse = math.sqrt(train_sq / train_n)

    # ---------------- VALIDATION ----------------
    ode_ef_model.eval()

    val_abs, val_sq, val_n = 0.0, 0.0, 0
    gate_sum, correction_sum = 0.0, 0.0

    with torch.no_grad():
        for batch in dde_val_loader:

            video = batch["video"].to(device, non_blocking=True)
            times = batch["sample_times"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)

            output = ode_ef_model(video, times)
            pred = output["predicted_ef"]

            error = pred.float() - true_ef.float()

            val_abs += error.abs().sum().item()
            val_sq += error.pow(2).sum().item()
            val_n += true_ef.numel()

            gate_sum += output["fusion_gate"].sum().item()
            correction_sum += output["correction"].abs().sum().item()

    val_mae = val_abs / val_n
    val_rmse = math.sqrt(val_sq / val_n)
    mean_gate = gate_sum / val_n
    mean_correction = correction_sum / val_n

    # ---------------- MODEL SELECTION ----------------
    improved_baseline = val_mae < ode_best_mae
    improved_trained = val_mae < ode_best_trained_mae

    if improved_baseline:
        ode_best_mae = val_mae
        ode_no_improve = 0
    else:
        ode_no_improve += 1

    if improved_trained:
        ode_best_trained_mae = val_mae

    ode_history.append({
        "epoch": epoch + 1,
        "train_rmse": train_rmse,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "mean_gate": mean_gate,
        "mean_abs_correction": mean_correction,
    })

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": ode_ef_model.state_dict(),
        "optimizer_state_dict": ode_optimizer.state_dict(),
        "scaler_state_dict": ode_scaler.state_dict(),
        "best_val_mae": ode_best_mae,
        "best_trained_mae": ode_best_trained_mae,
        "epochs_without_improvement": ode_no_improve,
        "history": ode_history,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "train_generator_state": ode_generator.get_state(),
    }

    torch.save(checkpoint, ODE_LATEST_PATH)

    if improved_baseline:
        torch.save(checkpoint, ODE_BEST_PATH)

    if improved_trained:
        torch.save(checkpoint, ODE_TRAINED_BEST_PATH)

    print()
    print(f"Neural ODE Epoch {epoch + 1}/{DDE_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")
    print(f"Mean gate: {mean_gate:.3f}")
    print(f"Mean |EF correction|: {mean_correction:.3f}")
    print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
    print(f"Best trained MAE: {ode_best_trained_mae:.3f}")
    print(f"No-improvement epochs: {ode_no_improve}")
    print("-" * 60)

    if ode_no_improve >= DDE_PATIENCE:
        print("Early stopping triggered.")
        break

In [ ]:
# ============================================================
# SECTION 15.17 — SAVE NEURAL ODE ABLATION RESULTS
# ============================================================

checkpoint = torch.load(
    ODE_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

ode_ef_model.load_state_dict(checkpoint["model_state_dict"])
ode_ef_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:
        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = ode_ef_model(video, times)["predicted_ef"]

        true_vals.extend(true_ef.squeeze(1).cpu().tolist())
        pred_vals.extend(pred.squeeze(1).cpu().tolist())

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

ode_mae = np.mean(np.abs(y_pred - y_true))
ode_rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
ode_bias = np.mean(y_pred - y_true)
ode_r = np.corrcoef(y_true, y_pred)[0, 1]

ode_r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

ablation_results = pd.read_csv(ABLATION_RESULTS_PATH)

ablation_results = ablation_results[
    ablation_results["model_variant"] != "Neural ODE (no delayed state)"
]

new_row = pd.DataFrame([{
    "model_variant": "Neural ODE (no delayed state)",
    "mae": ode_mae,
    "rmse": ode_rmse,
    "r2": ode_r2,
    "pearson_r": ode_r,
    "bias": ode_bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("NEURAL ODE — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {ode_mae:.3f}")
print(f"RMSE: {ode_rmse:.3f}")
print(f"R²: {ode_r2:.3f}")
print(f"Pearson r: {ode_r:.3f}")
print(f"Bias: {ode_bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

# 16...

In [ ]:
# ============================================================
# SECTION 16.1 — DELAY, RELIABILITY, AND GATE RESULTS
# ============================================================

DELAY_GATE_PATH = RESULTS_DIR / "delay_and_gate.csv"

# Restore best actually trained full DDE
checkpoint = torch.load(
    RESIDUAL_DDE_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

dde_ef_model.load_state_dict(checkpoint["model_state_dict"])
dde_ef_model.eval()

records = []

with torch.no_grad():
    for batch in dde_val_loader:

        video = batch["video"].to(device, non_blocking=True)
        times = batch["sample_times"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device)

        output = dde_ef_model(video, times)

        reliability = output["reliability"]
        rel_mean = reliability.mean(dim=1)

        baseline = output["baseline_ef"]
        predicted = output["predicted_ef"]
        gate = output["fusion_gate"]
        delay = output["delay_seconds"]
        correction = output["correction"]

        for i in range(len(true_ef)):

            records.append({
                "video_id": batch["filename"][i],
                "split": "VAL",

                "ground_truth_ef": true_ef[i].item(),
                "baseline_ef": baseline[i].item(),
                "dde_ef": predicted[i].item(),

                "baseline_abs_error":
                    abs(baseline[i].item() - true_ef[i].item()),

                "dde_abs_error":
                    abs(predicted[i].item() - true_ef[i].item()),

                "delay_seconds": delay[i].item(),
                "fusion_gate": gate[i].item(),
                "ef_correction": correction[i].item(),

                "Q_mean": rel_mean[i, 0].item(),
                "M_mean": rel_mean[i, 1].item(),
                "B_mean": rel_mean[i, 2].item(),
                "C_mean": rel_mean[i, 3].item(),
            })

delay_gate_df = pd.DataFrame(records)

delay_gate_df["reliability_mean"] = delay_gate_df[
    ["Q_mean", "M_mean", "B_mean", "C_mean"]
].mean(axis=1)

delay_gate_df["error_improvement"] = (
    delay_gate_df["baseline_abs_error"]
    - delay_gate_df["dde_abs_error"]
)

delay_gate_df.to_csv(DELAY_GATE_PATH, index=False)

print("=" * 60)
print("DELAY / RELIABILITY / GATE RESULTS")
print("=" * 60)
print(f"Videos: {len(delay_gate_df)}")
print(
    f"Delay: {delay_gate_df['delay_seconds'].mean():.3f} ± "
    f"{delay_gate_df['delay_seconds'].std():.3f} s"
)
print(
    f"Gate: {delay_gate_df['fusion_gate'].mean():.3f} ± "
    f"{delay_gate_df['fusion_gate'].std():.3f}"
)
print(
    f"Mean reliability: "
    f"{delay_gate_df['reliability_mean'].mean():.3f}"
)
print()
print("Saved:", DELAY_GATE_PATH)
print("CSV exists:", DELAY_GATE_PATH.exists())

assert len(delay_gate_df) == len(val_dataset)
assert np.isfinite(
    delay_gate_df.select_dtypes(include=np.number)
).all().all()

print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.2 — RELIABILITY-STRATIFIED PERFORMANCE
# ============================================================

RELIABILITY_STRATA_PATH = RESULTS_DIR / "reliability_stratified_metrics.csv"

# Split validation set into equal-sized reliability tertiles
q1, q2 = delay_gate_df["reliability_mean"].quantile([1/3, 2/3])

delay_gate_df["reliability_group"] = pd.cut(
    delay_gate_df["reliability_mean"],
    bins=[-np.inf, q1, q2, np.inf],
    labels=["Low", "Medium", "High"],
)

rows = []

for group in ["Low", "Medium", "High"]:
    df = delay_gate_df[
        delay_gate_df["reliability_group"] == group
    ]

    true = df["ground_truth_ef"].to_numpy()

    for model, column in [
        ("Temporal Transformer", "baseline_ef"),
        ("Residual DDE", "dde_ef"),
    ]:
        pred = df[column].to_numpy()
        error = pred - true

        rows.append({
            "reliability_group": group,
            "model": model,
            "n": len(df),
            "mae": np.mean(np.abs(error)),
            "rmse": np.sqrt(np.mean(error ** 2)),
            "bias": np.mean(error),
        })

strata_df = pd.DataFrame(rows)
strata_df.to_csv(RELIABILITY_STRATA_PATH, index=False)

print("=" * 60)
print("RELIABILITY-STRATIFIED PERFORMANCE")
print("=" * 60)
print(f"Low/Medium threshold: {q1:.3f}")
print(f"Medium/High threshold: {q2:.3f}")
print()
print(strata_df.to_string(index=False))
print()
print("Saved:", RELIABILITY_STRATA_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.3 — GATE VS. RELIABILITY BEHAVIOR
# ============================================================

GATE_CORR_PATH = RESULTS_DIR / "gate_reliability_correlations.csv"

variables = [
    "reliability_mean",
    "Q_mean",
    "M_mean",
    "B_mean",
    "C_mean",
    "delay_seconds",
    "error_improvement",
]

rows = []

for variable in variables:

    pearson = delay_gate_df[
        ["fusion_gate", variable]
    ].corr(method="pearson").iloc[0, 1]

    spearman = delay_gate_df[
        ["fusion_gate", variable]
    ].corr(method="spearman").iloc[0, 1]

    rows.append({
        "variable": variable,
        "pearson_r": pearson,
        "spearman_rho": spearman,
    })

gate_corr_df = pd.DataFrame(rows)
gate_corr_df.to_csv(GATE_CORR_PATH, index=False)

print("=" * 60)
print("FUSION GATE VS. RELIABILITY")
print("=" * 60)
print(gate_corr_df.to_string(index=False))
print()
print("Saved:", GATE_CORR_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.4 — REPRESENTATIVE RELIABILITY TRACES
# ============================================================

RELIABILITY_TRACES_PATH = RESULTS_DIR / "reliability_traces.csv"

# Pick video nearest median reliability within each group
representatives = []

for group in ["Low", "Medium", "High"]:
    group_df = delay_gate_df[
        delay_gate_df["reliability_group"] == group
    ].copy()

    median_rel = group_df["reliability_mean"].median()

    row = group_df.loc[
        (group_df["reliability_mean"] - median_rel).abs().idxmin()
    ]

    representatives.append((group, row["video_id"]))

# Map filename -> validation dataset index
filename_to_idx = {
    str(row["FileName"]): i
    for i, row in val_dataset.metadata.iterrows()
}

trace_records = []
dde_ef_model.eval()

with torch.no_grad():
    for group, video_id in representatives:

        sample = val_dataset[filename_to_idx[video_id]]

        video = sample["video"].unsqueeze(0).to(device)
        times = sample["sample_times"].to(device)

        output = dde_ef_model(video, times.unsqueeze(0))

        reliability = output["reliability"][0].cpu()
        delay = output["delay_seconds"][0].item()
        gate = output["fusion_gate"][0].item()

        for t in range(NUM_FRAMES):
            trace_records.append({
                "video_id": video_id,
                "reliability_group": group,
                "time_seconds": times[t].item(),
                "Q": reliability[t, 0].item(),
                "M": reliability[t, 1].item(),
                "B": reliability[t, 2].item(),
                "C": reliability[t, 3].item(),
                "delay_seconds": delay,
                "fusion_gate": gate,
            })

reliability_traces = pd.DataFrame(trace_records)
reliability_traces.to_csv(RELIABILITY_TRACES_PATH, index=False)

print("=" * 60)
print("REPRESENTATIVE RELIABILITY TRACES")
print("=" * 60)

for group, video_id in representatives:
    print(f"{group}: {video_id}")

print()
print(f"Videos saved: {len(representatives)}")
print(f"Time points per video: {NUM_FRAMES}")
print(f"Total rows: {len(reliability_traces)}")
print("Saved:", RELIABILITY_TRACES_PATH)
print("=" * 60)

assert len(reliability_traces) == 3 * NUM_FRAMES
assert np.isfinite(
    reliability_traces.select_dtypes(include=np.number)
).all().all()

In [ ]:
# ============================================================
# SECTION 16.5 — ROI ROBUSTNESS INSPECTION
# ============================================================

ROI_DIR = RESULTS_DIR / "roi_inspection"
ROI_DIR.mkdir(parents=True, exist_ok=True)

roi_records = []

for group, video_id in representatives:

    idx = filename_to_idx[video_id]
    sample = val_dataset[idx]

    # Reload original video
    video_path = val_dataset._get_video_path(video_id)
    frames = val_dataset._read_video(video_path)

    frame_indices = sample["frame_indices"].numpy()
    mid = len(frame_indices) // 2
    frame = frames[frame_indices[mid]]

    x1, y1, x2, y2 = sample["roi_bbox"].tolist()
    crop = frame[y1:y2, x1:x2]

    # Save original + ROI box + crop
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(frame)
    axes[0].add_patch(
        plt.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=2,
        )
    )
    axes[0].set_title(f"{group} reliability — ROI")
    axes[0].axis("off")

    axes[1].imshow(crop)
    axes[1].set_title("Detected cardiac ROI")
    axes[1].axis("off")

    fig.tight_layout()

    save_path = ROI_DIR / f"{group.lower()}_{video_id}.png"
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    frame_area = frame.shape[0] * frame.shape[1]
    roi_area = max(0, x2 - x1) * max(0, y2 - y1)

    roi_records.append({
        "video_id": video_id,
        "reliability_group": group,
        "roi_x1": x1,
        "roi_y1": y1,
        "roi_x2": x2,
        "roi_y2": y2,
        "roi_area_fraction": roi_area / frame_area,
        "inspection_file": str(save_path),
        "manual_failure": "",
        "manual_notes": "",
    })

roi_review_df = pd.DataFrame(roi_records)

ROI_REVIEW_PATH = RESULTS_DIR / "roi_manual_review.csv"
roi_review_df.to_csv(ROI_REVIEW_PATH, index=False)

print("=" * 60)
print("ROI INSPECTION EXAMPLES")
print("=" * 60)

for _, row in roi_review_df.iterrows():
    print(
        f"{row['reliability_group']}: "
        f"{row['video_id']} | "
        f"ROI area = {row['roi_area_fraction']:.3f}"
    )

print()
print("Images saved to:", ROI_DIR)
print("Review CSV:", ROI_REVIEW_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.6 — RECORD MANUAL ROI REVIEW
# ============================================================

manual_reviews = {
    "Low": (
        "Yes",
        "ROI too tight/off-center; substantial cardiac anatomy is excluded."
    ),
    "Medium": (
        "No",
        "Relevant cardiac anatomy preserved, but ROI is overly broad."
    ),
    "High": (
        "No",
        "ROI appropriately contains the relevant cardiac region."
    ),
}

for i, row in roi_review_df.iterrows():
    failure, notes = manual_reviews[row["reliability_group"]]
    roi_review_df.loc[i, "manual_failure"] = failure
    roi_review_df.loc[i, "manual_notes"] = notes

roi_review_df.to_csv(ROI_REVIEW_PATH, index=False)

print("=" * 60)
print("MANUAL ROI REVIEW")
print("=" * 60)

print(
    roi_review_df[
        [
            "reliability_group",
            "video_id",
            "roi_area_fraction",
            "manual_failure",
            "manual_notes",
        ]
    ].to_string(index=False)
)

print()
print("Saved:", ROI_REVIEW_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.7 — FIXED-DELAY VALUE
# ============================================================

FIXED_DELAY_SECONDS = float(
    delay_gate_df["delay_seconds"].median()
)

print("=" * 60)
print("FIXED-DELAY CONTROL")
print("=" * 60)
print(f"Mean estimated delay: {delay_gate_df['delay_seconds'].mean():.3f} s")
print(f"Median estimated delay: {FIXED_DELAY_SECONDS:.3f} s")
print(f"Minimum delay: {delay_gate_df['delay_seconds'].min():.3f} s")
print(f"Maximum delay: {delay_gate_df['delay_seconds'].max():.3f} s")
print()
print(f"Fixed delay selected: {FIXED_DELAY_SECONDS:.3f} s")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.8 — FIXED-DELAY NEURAL DDE
# ============================================================

class FixedDelayNeuralDDE(ReliabilityConditionedNeuralDDE):

    def __init__(
        self,
        fixed_delay_seconds,
        input_dim=768,
        latent_dim=256,
        reliability_dim=4,
        hidden_dim=512,
        dropout=0.1,
    ):
        super().__init__(
            input_dim=input_dim,
            latent_dim=latent_dim,
            reliability_dim=reliability_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.fixed_delay_seconds = float(fixed_delay_seconds)

    def _compute_delay_seconds(
        self,
        sample_times,
        cycle_lags,
    ):
        # Same fixed physical delay for every video
        return torch.full(
            (sample_times.shape[0],),
            self.fixed_delay_seconds,
            dtype=sample_times.dtype,
            device=sample_times.device,
        )

In [ ]:
# ============================================================
# SECTION 16.9 — BUILD + VERIFY FIXED-DELAY DDE
# ============================================================

set_seed(SEED)

fixed_delay_dde = FixedDelayNeuralDDE(
    fixed_delay_seconds=FIXED_DELAY_SECONDS,
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

fixed_delay_model = ReliabilityAwareDDEEFModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=fixed_delay_dde,
    transformer_dim=768,
    dde_dim=256,
).to(device)

# Freeze pretrained Transformer baseline
for p in fixed_delay_model.backbone.parameters():
    p.requires_grad = False

fixed_delay_model.reliability_extractor.eval()
fixed_delay_model.eval()

# Forward check
batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = fixed_delay_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]
delays = output["delay_seconds"]

max_diff = (predicted - baseline).abs().max().item()

trainable_params = sum(
    p.numel()
    for p in fixed_delay_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("FIXED-DELAY DDE — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print("Physical delays:", delays.cpu().tolist())
print("Baseline EF:", baseline.squeeze(1).cpu().tolist())
print("Final EF:", predicted.squeeze(1).cpu().tolist())
print("Correction:", output["correction"].squeeze(1).cpu().tolist())
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert torch.allclose(
    delays,
    torch.full_like(delays, FIXED_DELAY_SECONDS),
)

assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()

print("\nFixed-delay model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.10 — FIXED-DELAY DDE TRAINING SETUP
# ============================================================

FIXED_BEST_PATH = E2E_RUN_DIR / "08_fixed_delay_dde_best.pth"
FIXED_LATEST_PATH = E2E_RUN_DIR / "08_fixed_delay_dde_latest.pth"
FIXED_TRAINED_BEST_PATH = E2E_RUN_DIR / "08_fixed_delay_dde_best_trained.pth"

fixed_optimizer = torch.optim.AdamW(
    [
        {"params": fixed_delay_model.neural_dde.parameters(), "lr": DDE_LR},
        {"params": fixed_delay_model.residual_fusion.parameters(), "lr": RESIDUAL_LR},
    ],
    weight_decay=DDE_WEIGHT_DECAY,
)

fixed_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
fixed_generator = torch.Generator().manual_seed(SEED)

fixed_train_loader = DataLoader(
    dde_train_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=USE_AMP,
    generator=fixed_generator,
)

fixed_best_mae = TRANSFORMER_BASELINE_MAE
fixed_best_trained_mae = float("inf")
fixed_no_improve = 0
fixed_history = []

# Save exact Transformer-equivalent starting model
if not FIXED_BEST_PATH.exists():
    torch.save(
        {
            "epoch": 0,
            "model_state_dict": fixed_delay_model.state_dict(),
            "optimizer_state_dict": fixed_optimizer.state_dict(),
            "scaler_state_dict": fixed_scaler.state_dict(),
            "best_val_mae": TRANSFORMER_BASELINE_MAE,
            "history": [],
            "fixed_delay_seconds": FIXED_DELAY_SECONDS,
            "stage": "fixed_delay_dde_baseline",
        },
        FIXED_BEST_PATH,
    )

print("=" * 60)
print("FIXED-DELAY DDE — TRAINING SETUP")
print("=" * 60)
print(f"Fixed delay: {FIXED_DELAY_SECONDS:.3f} s")
print(f"TRAIN videos: {len(dde_train_dataset)}")
print(f"TRAIN batches: {len(fixed_train_loader)}")
print(f"VAL videos: {len(val_dataset)}")
print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
print(f"DDE LR: {DDE_LR}")
print(f"Residual LR: {RESIDUAL_LR}")
print(f"Initial BEST saved: {FIXED_BEST_PATH.exists()}")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 16.11 — TRAIN FIXED-DELAY DDE
# ============================================================

criterion = nn.MSELoss()
start_epoch = 0

# Resume if interrupted
if FIXED_LATEST_PATH.exists():
    ckpt = torch.load(FIXED_LATEST_PATH, map_location=device, weights_only=False)

    fixed_delay_model.load_state_dict(ckpt["model_state_dict"])
    fixed_optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    fixed_scaler.load_state_dict(ckpt["scaler_state_dict"])

    start_epoch = ckpt["epoch"]
    fixed_best_mae = ckpt["best_val_mae"]
    fixed_best_trained_mae = ckpt["best_trained_mae"]
    fixed_no_improve = ckpt["epochs_without_improvement"]
    fixed_history = ckpt["history"]

    if "train_generator_state" in ckpt:
        fixed_generator.set_state(ckpt["train_generator_state"].cpu())

    print(f"Resuming from epoch {start_epoch + 1}")

torch.cuda.empty_cache()

print("=" * 60)
print("FIXED-DELAY DDE — TRAINING")
print("=" * 60)

for epoch in range(start_epoch, DDE_MAX_EPOCHS):

    # ---------------- TRAIN ----------------
    fixed_delay_model.train()
    fixed_delay_model.backbone.eval()
    fixed_delay_model.reliability_extractor.eval()

    train_sq, train_n = 0.0, 0

    for batch_idx, batch in enumerate(fixed_train_loader):

        video = batch["video"].to(device, non_blocking=True)
        times = batch["sample_times"].to(device, non_blocking=True)
        true_ef = batch["ef"].to(device, non_blocking=True)

        fixed_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            output = fixed_delay_model(video, times)
            pred = output["predicted_ef"]
            loss = criterion(pred, true_ef)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss detected.")

        fixed_scaler.scale(loss).backward()
        fixed_scaler.step(fixed_optimizer)
        fixed_scaler.update()

        error = pred.detach().float() - true_ef.float()
        train_sq += error.pow(2).sum().item()
        train_n += true_ef.numel()

        if (batch_idx + 1) % 200 == 0:
            print(
                f"Epoch {epoch + 1} | "
                f"Batch {batch_idx + 1}/{len(fixed_train_loader)} | "
                f"Train RMSE: {math.sqrt(train_sq / train_n):.3f}"
            )

    train_rmse = math.sqrt(train_sq / train_n)

    # ---------------- VALIDATION ----------------
    fixed_delay_model.eval()

    val_abs, val_sq, val_n = 0.0, 0.0, 0
    gate_sum, correction_sum = 0.0, 0.0

    with torch.no_grad():
        for batch in dde_val_loader:

            video = batch["video"].to(device, non_blocking=True)
            times = batch["sample_times"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)

            output = fixed_delay_model(video, times)
            pred = output["predicted_ef"]

            error = pred.float() - true_ef.float()

            val_abs += error.abs().sum().item()
            val_sq += error.pow(2).sum().item()
            val_n += true_ef.numel()

            gate_sum += output["fusion_gate"].sum().item()
            correction_sum += output["correction"].abs().sum().item()

    val_mae = val_abs / val_n
    val_rmse = math.sqrt(val_sq / val_n)
    mean_gate = gate_sum / val_n
    mean_correction = correction_sum / val_n

    # ---------------- MODEL SELECTION ----------------
    improved_baseline = val_mae < fixed_best_mae
    improved_trained = val_mae < fixed_best_trained_mae

    if improved_baseline:
        fixed_best_mae = val_mae
        fixed_no_improve = 0
    else:
        fixed_no_improve += 1

    if improved_trained:
        fixed_best_trained_mae = val_mae

    fixed_history.append({
        "epoch": epoch + 1,
        "train_rmse": train_rmse,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "mean_gate": mean_gate,
        "mean_abs_correction": mean_correction,
    })

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": fixed_delay_model.state_dict(),
        "optimizer_state_dict": fixed_optimizer.state_dict(),
        "scaler_state_dict": fixed_scaler.state_dict(),
        "best_val_mae": fixed_best_mae,
        "best_trained_mae": fixed_best_trained_mae,
        "epochs_without_improvement": fixed_no_improve,
        "history": fixed_history,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "fixed_delay_seconds": FIXED_DELAY_SECONDS,
        "train_generator_state": fixed_generator.get_state(),
    }

    torch.save(checkpoint, FIXED_LATEST_PATH)

    if improved_baseline:
        torch.save(checkpoint, FIXED_BEST_PATH)

    if improved_trained:
        torch.save(checkpoint, FIXED_TRAINED_BEST_PATH)

    print()
    print(f"Fixed-Delay DDE Epoch {epoch + 1}/{DDE_MAX_EPOCHS}")
    print(f"Train RMSE: {train_rmse:.3f}")
    print(f"Validation MAE: {val_mae:.3f}")
    print(f"Validation RMSE: {val_rmse:.3f}")
    print(f"Mean gate: {mean_gate:.3f}")
    print(f"Mean |EF correction|: {mean_correction:.3f}")
    print(f"Baseline MAE: {TRANSFORMER_BASELINE_MAE:.3f}")
    print(f"Best trained MAE: {fixed_best_trained_mae:.3f}")
    print(f"No-improvement epochs: {fixed_no_improve}")
    print("-" * 60)

    if fixed_no_improve >= DDE_PATIENCE:
        print("Early stopping triggered.")
        break

In [ ]:
# ============================================================
# SECTION 16.12 — SAVE FIXED-DELAY ABLATION RESULTS
# ============================================================

checkpoint = torch.load(
    FIXED_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

fixed_delay_model.load_state_dict(checkpoint["model_state_dict"])
fixed_delay_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:

        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = fixed_delay_model(
            video,
            times,
        )["predicted_ef"]

        true_vals.extend(
            true_ef.squeeze(1).cpu().tolist()
        )

        pred_vals.extend(
            pred.squeeze(1).cpu().tolist()
        )

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

fd_mae = np.mean(np.abs(y_pred - y_true))
fd_rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
fd_bias = np.mean(y_pred - y_true)
fd_r = np.corrcoef(y_true, y_pred)[0, 1]

fd_r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

ablation_results = pd.read_csv(
    ABLATION_RESULTS_PATH
)

ablation_results = ablation_results[
    ablation_results["model_variant"]
    != "Fixed-delay DDE (1.1 s)"
]

new_row = pd.DataFrame([{
    "model_variant": "Fixed-delay DDE (1.1 s)",
    "mae": fd_mae,
    "rmse": fd_rmse,
    "r2": fd_r2,
    "pearson_r": fd_r,
    "bias": fd_bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("FIXED-DELAY DDE — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {fd_mae:.3f}")
print(f"RMSE: {fd_rmse:.3f}")
print(f"R²: {fd_r2:.3f}")
print(f"Pearson r: {fd_r:.3f}")
print(f"Bias: {fd_bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

# 17...

In [ ]:
# ============================================================
# SECTION 17.1 — RELIABILITY COMPONENT ABLATION MODEL
# ============================================================

RELIABILITY_INDEX = {
    "Q": 0,
    "M": 1,
    "B": 2,
    "C": 3,
}


class ReliabilityComponentAblationModel(nn.Module):

    def __init__(
        self,
        backbone,
        reliability_extractor,
        neural_dde,
        removed_component,
    ):
        super().__init__()

        self.backbone = backbone
        self.reliability_extractor = reliability_extractor
        self.neural_dde = neural_dde
        self.removed_component = removed_component

        self.residual_fusion = ReliabilityAwareResidualFusion(
            transformer_dim=768,
            dde_dim=256,
            reliability_dim=4,
            hidden_dim=256,
            dropout=0.1,
        )

    def forward(self, video, sample_times):

        frame_features = self.backbone.spatial_encoder(video)

        sequence_features, transformer_feature = (
            self.backbone.temporal_transformer(frame_features)
        )

        baseline_ef = self.backbone.ef_head(transformer_feature)

        reliability, cycle_lags = self.reliability_extractor(
            video,
            sequence_features,
        )

        # Remove exactly one reliability component
        masked_reliability = reliability.clone()
        idx = RELIABILITY_INDEX[self.removed_component]
        masked_reliability[:, :, idx] = 0.0

        dde_sequence, dde_feature, delay_seconds = self.neural_dde(
            sequence_features,
            masked_reliability,
            sample_times,
            cycle_lags,
        )

        reliability_summary = masked_reliability.mean(dim=1)

        correction, gate, _ = self.residual_fusion(
            transformer_feature,
            dde_feature,
            reliability_summary,
        )

        return {
            "predicted_ef": baseline_ef + correction,
            "baseline_ef": baseline_ef,
            "correction": correction,
            "fusion_gate": gate,
            "dde_sequence": dde_sequence,
            "reliability": masked_reliability,
            "cycle_lags": cycle_lags,
            "delay_seconds": delay_seconds,
        }

In [ ]:
# ============================================================
# SECTION 17.2 — BUILD + VERIFY WITHOUT Q
# ============================================================

set_seed(SEED)

without_q_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

without_q_model = ReliabilityComponentAblationModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=without_q_dde,
    removed_component="Q",
).to(device)

# Freeze pretrained backbone
for p in without_q_model.backbone.parameters():
    p.requires_grad = False

without_q_model.reliability_extractor.eval()
without_q_model.eval()

# Forward check
batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = without_q_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]
max_diff = (predicted - baseline).abs().max().item()

q_values = output["reliability"][:, :, 0]

trainable_params = sum(
    p.numel()
    for p in without_q_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("WITHOUT Q — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print(f"Q max after masking: {q_values.abs().max().item():.6f}")
print("Baseline EF:", baseline.squeeze(1).cpu().tolist())
print("Final EF:", predicted.squeeze(1).cpu().tolist())
print("Correction:", output["correction"].squeeze(1).cpu().tolist())
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert q_values.abs().max().item() == 0.0
assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()

print("\nWithout-Q model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 17.3 — COMPONENT ABLATION TRAINING FUNCTION
# ============================================================

def train_component_ablation(model, name, file_tag):

    best_path = E2E_RUN_DIR / f"{file_tag}_best.pth"
    latest_path = E2E_RUN_DIR / f"{file_tag}_latest.pth"
    trained_best_path = E2E_RUN_DIR / f"{file_tag}_best_trained.pth"

    optimizer = torch.optim.AdamW(
        [
            {"params": model.neural_dde.parameters(), "lr": DDE_LR},
            {"params": model.residual_fusion.parameters(), "lr": RESIDUAL_LR},
        ],
        weight_decay=DDE_WEIGHT_DECAY,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    generator = torch.Generator().manual_seed(SEED)

    train_loader = DataLoader(
        dde_train_dataset,
        batch_size=DDE_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=USE_AMP,
        generator=generator,
    )

    best_mae = TRANSFORMER_BASELINE_MAE
    best_trained_mae = float("inf")
    no_improve = 0
    history = []
    start_epoch = 0

    # Resume
    if latest_path.exists():
        ckpt = torch.load(latest_path, map_location=device, weights_only=False)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])

        start_epoch = ckpt["epoch"]
        best_mae = ckpt["best_val_mae"]
        best_trained_mae = ckpt["best_trained_mae"]
        no_improve = ckpt["epochs_without_improvement"]
        history = ckpt["history"]

        if "train_generator_state" in ckpt:
            generator.set_state(ckpt["train_generator_state"].cpu())

        print(f"Resuming {name} from epoch {start_epoch + 1}")

    else:
        torch.save(
            {
                "epoch": 0,
                "model_state_dict": model.state_dict(),
                "best_val_mae": TRANSFORMER_BASELINE_MAE,
            },
            best_path,
        )

    criterion = nn.MSELoss()
    torch.cuda.empty_cache()

    print("=" * 60)
    print(f"{name} — TRAINING")
    print("=" * 60)

    for epoch in range(start_epoch, DDE_MAX_EPOCHS):

        # ---------------- TRAIN ----------------
        model.train()
        model.backbone.eval()
        model.reliability_extractor.eval()

        train_sq, train_n = 0.0, 0

        for batch_idx, batch in enumerate(train_loader):

            video = batch["video"].to(device, non_blocking=True)
            times = batch["sample_times"].to(device, non_blocking=True)
            true_ef = batch["ef"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(
                device_type=device.type,
                enabled=USE_AMP,
            ):
                output = model(video, times)
                pred = output["predicted_ef"]
                loss = criterion(pred, true_ef)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            error = pred.detach().float() - true_ef.float()
            train_sq += error.pow(2).sum().item()
            train_n += true_ef.numel()

            if (batch_idx + 1) % 200 == 0:
                print(
                    f"Epoch {epoch + 1} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Train RMSE: {math.sqrt(train_sq / train_n):.3f}"
                )

        train_rmse = math.sqrt(train_sq / train_n)

        # ---------------- VALIDATION ----------------
        model.eval()

        val_abs, val_sq, val_n = 0.0, 0.0, 0

        with torch.no_grad():
            for batch in dde_val_loader:

                video = batch["video"].to(device, non_blocking=True)
                times = batch["sample_times"].to(device, non_blocking=True)
                true_ef = batch["ef"].to(device, non_blocking=True)

                pred = model(video, times)["predicted_ef"]
                error = pred.float() - true_ef.float()

                val_abs += error.abs().sum().item()
                val_sq += error.pow(2).sum().item()
                val_n += true_ef.numel()

        val_mae = val_abs / val_n
        val_rmse = math.sqrt(val_sq / val_n)

        improved_baseline = val_mae < best_mae
        improved_trained = val_mae < best_trained_mae

        if improved_baseline:
            best_mae = val_mae
            no_improve = 0
        else:
            no_improve += 1

        if improved_trained:
            best_trained_mae = val_mae

        history.append({
            "epoch": epoch + 1,
            "train_rmse": train_rmse,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
        })

        ckpt = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_val_mae": best_mae,
            "best_trained_mae": best_trained_mae,
            "epochs_without_improvement": no_improve,
            "history": history,
            "val_mae": val_mae,
            "val_rmse": val_rmse,
            "train_generator_state": generator.get_state(),
        }

        torch.save(ckpt, latest_path)

        if improved_baseline:
            torch.save(ckpt, best_path)

        if improved_trained:
            torch.save(ckpt, trained_best_path)

        print()
        print(f"{name} Epoch {epoch + 1}/{DDE_MAX_EPOCHS}")
        print(f"Train RMSE: {train_rmse:.3f}")
        print(f"Validation MAE: {val_mae:.3f}")
        print(f"Validation RMSE: {val_rmse:.3f}")
        print(f"Best trained MAE: {best_trained_mae:.3f}")
        print(f"No-improvement epochs: {no_improve}")
        print("-" * 60)

        if no_improve >= DDE_PATIENCE:
            print("Early stopping triggered.")
            break

    return trained_best_path

In [ ]:
# ============================================================
# SECTION 17.4 — TRAIN WITHOUT Q
# ============================================================

WITHOUT_Q_TRAINED_BEST_PATH = train_component_ablation(
    model=without_q_model,
    name="WITHOUT Q",
    file_tag="09_without_q",
)

In [ ]:
# ============================================================
# SECTION 17.5 — SAVE WITHOUT-Q ABLATION RESULT
# ============================================================

ckpt = torch.load(
    WITHOUT_Q_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

without_q_model.load_state_dict(ckpt["model_state_dict"])
without_q_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:

        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = without_q_model(video, times)["predicted_ef"]

        true_vals.extend(true_ef.squeeze(1).cpu().tolist())
        pred_vals.extend(pred.squeeze(1).cpu().tolist())

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

mae = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
bias = np.mean(y_pred - y_true)
r = np.corrcoef(y_true, y_pred)[0, 1]

r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

ablation_results = pd.read_csv(ABLATION_RESULTS_PATH)

ablation_results = ablation_results[
    ablation_results["model_variant"] != "DDE without Q"
]

new_row = pd.DataFrame([{
    "model_variant": "DDE without Q",
    "mae": mae,
    "rmse": rmse,
    "r2": r2,
    "pearson_r": r,
    "bias": bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("WITHOUT Q — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")
print(f"Pearson r: {r:.3f}")
print(f"Bias: {bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 17.6 — BUILD + VERIFY WITHOUT M
# ============================================================

set_seed(SEED)

without_m_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

without_m_model = ReliabilityComponentAblationModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=without_m_dde,
    removed_component="M",
).to(device)

for p in without_m_model.backbone.parameters():
    p.requires_grad = False

without_m_model.reliability_extractor.eval()
without_m_model.eval()

batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = without_m_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]

m_values = output["reliability"][:, :, 1]

max_diff = (
    predicted - baseline
).abs().max().item()

trainable_params = sum(
    p.numel()
    for p in without_m_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("WITHOUT M — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print(f"M max after masking: {m_values.abs().max().item():.6f}")
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert m_values.abs().max().item() == 0.0
assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()

print("\nWithout-M model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 17.7 — TRAIN WITHOUT M
# ============================================================

WITHOUT_M_TRAINED_BEST_PATH = train_component_ablation(
    model=without_m_model,
    name="WITHOUT M",
    file_tag="10_without_m",
)

In [ ]:
# ============================================================
# SECTION 17.8 — SAVE WITHOUT-M ABLATION RESULT
# ============================================================

ckpt = torch.load(
    WITHOUT_M_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

without_m_model.load_state_dict(ckpt["model_state_dict"])
without_m_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:

        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = without_m_model(video, times)["predicted_ef"]

        true_vals.extend(true_ef.squeeze(1).cpu().tolist())
        pred_vals.extend(pred.squeeze(1).cpu().tolist())

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

mae = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
bias = np.mean(y_pred - y_true)
r = np.corrcoef(y_true, y_pred)[0, 1]

r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

ablation_results = pd.read_csv(ABLATION_RESULTS_PATH)

ablation_results = ablation_results[
    ablation_results["model_variant"] != "DDE without M"
]

new_row = pd.DataFrame([{
    "model_variant": "DDE without M",
    "mae": mae,
    "rmse": rmse,
    "r2": r2,
    "pearson_r": r,
    "bias": bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("WITHOUT M — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")
print(f"Pearson r: {r:.3f}")
print(f"Bias: {bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 17.9 — BUILD + VERIFY WITHOUT B
# ============================================================

set_seed(SEED)

without_b_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

without_b_model = ReliabilityComponentAblationModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=without_b_dde,
    removed_component="B",
).to(device)

for p in without_b_model.backbone.parameters():
    p.requires_grad = False

without_b_model.reliability_extractor.eval()
without_b_model.eval()

batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = without_b_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]

b_values = output["reliability"][:, :, 2]

max_diff = (
    predicted - baseline
).abs().max().item()

trainable_params = sum(
    p.numel()
    for p in without_b_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("WITHOUT B — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print(f"B max after masking: {b_values.abs().max().item():.6f}")
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert b_values.abs().max().item() == 0.0
assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()

print("\nWithout-B model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 17.10 — TRAIN WITHOUT B
# ============================================================

WITHOUT_B_TRAINED_BEST_PATH = train_component_ablation(
    model=without_b_model,
    name="WITHOUT B",
    file_tag="11_without_b",
)

In [ ]:
# ============================================================
# SECTION 17.11 — SAVE WITHOUT-B ABLATION RESULT
# ============================================================

WITHOUT_B_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "11_without_b_best_trained.pth"
)

ckpt = torch.load(
    WITHOUT_B_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

without_b_model.load_state_dict(ckpt["model_state_dict"])
without_b_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:

        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = without_b_model(video, times)["predicted_ef"]

        true_vals.extend(true_ef.squeeze(1).cpu().tolist())
        pred_vals.extend(pred.squeeze(1).cpu().tolist())

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

mae = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
bias = np.mean(y_pred - y_true)
r = np.corrcoef(y_true, y_pred)[0, 1]

r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

ablation_results = pd.read_csv(ABLATION_RESULTS_PATH)

ablation_results = ablation_results[
    ablation_results["model_variant"] != "DDE without B"
]

new_row = pd.DataFrame([{
    "model_variant": "DDE without B",
    "mae": mae,
    "rmse": rmse,
    "r2": r2,
    "pearson_r": r,
    "bias": bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("WITHOUT B — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")
print(f"Pearson r: {r:.3f}")
print(f"Bias: {bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 17.12 — BUILD + VERIFY WITHOUT C
# ============================================================

set_seed(SEED)

without_c_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

without_c_model = ReliabilityComponentAblationModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=without_c_dde,
    removed_component="C",
).to(device)

for p in without_c_model.backbone.parameters():
    p.requires_grad = False

without_c_model.reliability_extractor.eval()
without_c_model.eval()

batch = next(iter(dde_val_loader))
video = batch["video"][:2].to(device)
times = batch["sample_times"][:2].to(device)

with torch.no_grad():
    output = without_c_model(video, times)

baseline = output["baseline_ef"]
predicted = output["predicted_ef"]

c_values = output["reliability"][:, :, 3]

max_diff = (
    predicted - baseline
).abs().max().item()

trainable_params = sum(
    p.numel()
    for p in without_c_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("WITHOUT C — MODEL CHECK")
print("=" * 60)
print(f"Trainable parameters: {trainable_params:,}")
print(f"C max after masking: {c_values.abs().max().item():.6f}")
print(f"Maximum baseline/final difference: {max_diff:.8f}")

assert c_values.abs().max().item() == 0.0
assert max_diff < 1e-6
assert torch.isfinite(predicted).all()
assert torch.isfinite(output["dde_sequence"]).all()

print("\nWithout-C model exactly reproduces Transformer baseline.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 17.13 — TRAIN WITHOUT C
# ============================================================

WITHOUT_C_TRAINED_BEST_PATH = train_component_ablation(
    model=without_c_model,
    name="WITHOUT C",
    file_tag="12_without_c",
)

In [ ]:
# ============================================================
# SECTION 17.14 — SAVE WITHOUT-C ABLATION RESULT
# ============================================================

WITHOUT_C_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "12_without_c_best_trained.pth"
)

ckpt = torch.load(
    WITHOUT_C_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

without_c_model.load_state_dict(ckpt["model_state_dict"])
without_c_model.eval()

true_vals, pred_vals = [], []

with torch.no_grad():
    for batch in dde_val_loader:

        video = batch["video"].to(device)
        times = batch["sample_times"].to(device)
        true_ef = batch["ef"].to(device)

        pred = without_c_model(video, times)["predicted_ef"]

        true_vals.extend(true_ef.squeeze(1).cpu().tolist())
        pred_vals.extend(pred.squeeze(1).cpu().tolist())

y_true = np.array(true_vals)
y_pred = np.array(pred_vals)

mae = np.mean(np.abs(y_pred - y_true))
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
bias = np.mean(y_pred - y_true)
r = np.corrcoef(y_true, y_pred)[0, 1]

r2 = 1 - (
    np.sum((y_true - y_pred) ** 2)
    / np.sum((y_true - y_true.mean()) ** 2)
)

ablation_results = pd.read_csv(ABLATION_RESULTS_PATH)

ablation_results = ablation_results[
    ablation_results["model_variant"] != "DDE without C"
]

new_row = pd.DataFrame([{
    "model_variant": "DDE without C",
    "mae": mae,
    "rmse": rmse,
    "r2": r2,
    "pearson_r": r,
    "bias": bias,
    "seed": SEED,
}])

ablation_results = pd.concat(
    [ablation_results, new_row],
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_RESULTS_PATH,
    index=False,
)

print("=" * 60)
print("WITHOUT C — FINAL RESULTS")
print("=" * 60)
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")
print(f"Pearson r: {r:.3f}")
print(f"Bias: {bias:.3f}")
print()
print("Saved to:", ABLATION_RESULTS_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 18.1 — LOAD FULL TRAINED DDE FOR DIAGNOSTICS
# ============================================================

FULL_DDE_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "04_residual_dde_best_trained.pth"
)

assert FULL_DDE_TRAINED_BEST_PATH.exists()

diagnostic_neural_dde = ReliabilityConditionedNeuralDDE(
    input_dim=768,
    latent_dim=256,
    reliability_dim=4,
    hidden_dim=512,
    dropout=0.1,
).to(device)

diagnostic_model = ReliabilityAwareDDEEFModel(
    backbone=best_transformer_model,
    reliability_extractor=reliability_extractor,
    neural_dde=diagnostic_neural_dde,
).to(device)

for p in diagnostic_model.backbone.parameters():
    p.requires_grad = False

checkpoint = torch.load(
    FULL_DDE_TRAINED_BEST_PATH,
    map_location=device,
    weights_only=False,
)

diagnostic_model.load_state_dict(
    checkpoint["model_state_dict"]
)

diagnostic_model.eval()
diagnostic_model.reliability_extractor.eval()

trainable_params = sum(
    p.numel()
    for p in diagnostic_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("FULL DDE — DIAGNOSTIC MODEL LOADED")
print("=" * 60)
print("Checkpoint:", FULL_DDE_TRAINED_BEST_PATH.name)
print(f"Trainable parameters: {trainable_params:,}")
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Checkpoint VAL MAE:", checkpoint.get("val_mae"))
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 18.2 — NUMERICAL STABILITY + GRADIENT DIAGNOSTICS
# ============================================================

from torch.utils.data import DataLoader

diagnostic_loader = DataLoader(
    dde_train_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

batch = next(iter(diagnostic_loader))

video = batch["video"].to(device)
times = batch["sample_times"].to(device)
true_ef = batch["ef"].to(device)

diagnostic_model.train()
diagnostic_model.backbone.eval()
diagnostic_model.reliability_extractor.eval()

diagnostic_model.zero_grad(set_to_none=True)

output = diagnostic_model(
    video,
    times,
)

pred = output["predicted_ef"]
dde_sequence = output["dde_sequence"]
reliability = output["reliability"]
delay_seconds = output["delay_seconds"]

loss = nn.MSELoss()(pred, true_ef)
loss.backward()

# ------------------------------------------------------------
# Forward numerical checks
# ------------------------------------------------------------

forward_tensors = {
    "predicted_ef": pred,
    "dde_sequence": dde_sequence,
    "reliability": reliability,
    "delay_seconds": delay_seconds,
}

forward_nan = {}
forward_inf = {}

for name, tensor in forward_tensors.items():
    forward_nan[name] = torch.isnan(tensor).any().item()
    forward_inf[name] = torch.isinf(tensor).any().item()

# ------------------------------------------------------------
# Latent-state norms
# ------------------------------------------------------------

state_norms = torch.linalg.vector_norm(
    dde_sequence.detach().float(),
    dim=-1,
)

# ------------------------------------------------------------
# Gradient checks
# ------------------------------------------------------------

grad_norms = []
grad_nan = False
grad_inf = False

for name, param in diagnostic_model.named_parameters():

    if param.requires_grad and param.grad is not None:

        grad = param.grad.detach().float()

        grad_norms.append(
            (name, grad.norm().item())
        )

        grad_nan |= torch.isnan(grad).any().item()
        grad_inf |= torch.isinf(grad).any().item()

total_grad_norm = np.sqrt(
    sum(norm ** 2 for _, norm in grad_norms)
)

max_grad_name, max_grad_norm = max(
    grad_norms,
    key=lambda x: x[1],
)

print("=" * 60)
print("FULL DDE — NUMERICAL DIAGNOSTICS")
print("=" * 60)

print(f"Loss: {loss.item():.6f}")
print()

print("Forward NaN checks:")
for name in forward_tensors:
    print(
        f"  {name}: "
        f"NaN={forward_nan[name]}, "
        f"Inf={forward_inf[name]}"
    )

print()
print("Latent state norms:")
print(f"  Min:  {state_norms.min().item():.6f}")
print(f"  Mean: {state_norms.mean().item():.6f}")
print(f"  Max:  {state_norms.max().item():.6f}")

print()
print("Gradient diagnostics:")
print(f"  Parameters with gradients: {len(grad_norms)}")
print(f"  Total gradient norm: {total_grad_norm:.6f}")
print(f"  Maximum parameter gradient norm: {max_grad_norm:.6f}")
print(f"  Maximum-gradient parameter: {max_grad_name}")
print(f"  Gradient NaN: {grad_nan}")
print(f"  Gradient Inf: {grad_inf}")

assert not any(forward_nan.values())
assert not any(forward_inf.values())
assert not grad_nan
assert not grad_inf
assert np.isfinite(total_grad_norm)
assert torch.isfinite(state_norms).all()

print()
print("Numerical stability and gradient checks PASSED.")
print("=" * 60)

# No optimizer.step() — checkpoint weights remain unchanged.
diagnostic_model.zero_grad(set_to_none=True)

In [ ]:
# ============================================================
# SECTION 18.3 — DELAYED-STATE INDEXING DIAGNOSTIC
# ============================================================

dde_module = diagnostic_model.neural_dde

# ------------------------------------------------------------
# Synthetic history with easy-to-check values
# ------------------------------------------------------------

history_times = torch.tensor(
    [
        [0.0, 1.0, 2.0, 3.0],
        [0.0, 1.0, 2.0, 3.0],
        [0.0, 1.0, 2.0, 3.0],
    ],
    dtype=torch.float32,
    device=device,
)

history_states = torch.tensor(
    [
        [[0.0], [10.0], [20.0], [30.0]],
        [[0.0], [10.0], [20.0], [30.0]],
        [[0.0], [10.0], [20.0], [30.0]],
    ],
    dtype=torch.float32,
    device=device,
)

# Case 1: before available history -> initial state
# Case 2: between t=1 and t=2 -> linear interpolation = 15
# Case 3: after available history -> latest state
delayed_times = torch.tensor(
    [-0.5, 1.5, 3.5],
    dtype=torch.float32,
    device=device,
)

with torch.no_grad():

    delayed_states = (
        dde_module._interpolate_delayed_state(
            history_states,
            history_times,
            delayed_times,
        )
    )

expected = torch.tensor(
    [[0.0], [15.0], [30.0]],
    dtype=torch.float32,
    device=device,
)

max_error = (
    delayed_states - expected
).abs().max().item()

print("=" * 60)
print("DELAYED-STATE INTERPOLATION DIAGNOSTIC")
print("=" * 60)

print("Delayed times:")
print(delayed_times.cpu().tolist())

print("\nReturned states:")
print(delayed_states.squeeze(-1).cpu().tolist())

print("\nExpected states:")
print(expected.squeeze(-1).cpu().tolist())

print(f"\nMaximum interpolation error: {max_error:.8f}")

assert torch.allclose(
    delayed_states,
    expected,
    atol=1e-6,
)

# ------------------------------------------------------------
# Real-batch time-index sanity check
# ------------------------------------------------------------

diagnostic_model.eval()

batch = next(iter(diagnostic_loader))

video = batch["video"].to(device)
times = batch["sample_times"].to(device)

with torch.no_grad():

    real_output = diagnostic_model(
        video,
        times,
    )

real_delays = (
    real_output["delay_seconds"]
    .reshape(-1)
)

assert torch.all(
    torch.diff(times, dim=1) > 0
)

assert torch.all(real_delays > 0)

future_access_violations = 0
bracketing_violations = 0
interpolation_count = 0
prehistory_count = 0

B, T = times.shape

for b in range(B):

    tau = real_delays[b]

    for step in range(1, T):

        current_time = times[b, step]

        delayed_time = (
            current_time - tau
        )

        history = times[b, :step + 1]

        # Before first available state
        if delayed_time <= history[0]:

            prehistory_count += 1
            continue

        right = torch.searchsorted(
            history,
            delayed_time,
        ).item()

        right = min(
            right,
            step,
        )

        left = max(
            right - 1,
            0,
        )

        # Delayed state must never use a future index
        if right > step:
            future_access_violations += 1

        left_time = history[left]
        right_time = history[right]

        if right != left:

            if not (
                left_time <= delayed_time
                and delayed_time <= right_time
            ):
                bracketing_violations += 1

            alpha = (
                (delayed_time - left_time)
                / (right_time - left_time)
            )

            assert 0.0 <= alpha.item() <= 1.0

            interpolation_count += 1

print()
print("REAL-BATCH INDEXING CHECK")
print("-" * 60)

print(
    "Sample delays:",
    real_delays.cpu().tolist(),
)

print(
    "Prehistory fallback count:",
    prehistory_count,
)

print(
    "Interpolation count:",
    interpolation_count,
)

print(
    "Future-state access violations:",
    future_access_violations,
)

print(
    "Bracketing violations:",
    bracketing_violations,
)

assert future_access_violations == 0
assert bracketing_violations == 0

print()
print(
    "Delayed-state indexing and interpolation PASSED."
)

print("=" * 60)

In [ ]:
# ============================================================
# SECTION 18.4 — SAVE NUMERICAL DIAGNOSTICS
# ============================================================

import json

NUMERICAL_DIAGNOSTICS_PATH = (
    RESULTS_DIR / "numerical_diagnostics.json"
)

diagnostics = {
    "checkpoint": FULL_DDE_TRAINED_BEST_PATH.name,
    "checkpoint_epoch": int(checkpoint.get("epoch", -1)),
    "validation_mae": float(checkpoint.get("val_mae")),

    "forward": {
        "loss": float(loss.item()),
        "nan": {
            k: bool(v)
            for k, v in forward_nan.items()
        },
        "inf": {
            k: bool(v)
            for k, v in forward_inf.items()
        },
    },

    "latent_state_norms": {
        "min": float(state_norms.min().item()),
        "mean": float(state_norms.mean().item()),
        "max": float(state_norms.max().item()),
    },

    "gradients": {
        "parameters_with_gradients": len(grad_norms),
        "total_gradient_norm": float(total_grad_norm),
        "max_gradient_norm": float(max_grad_norm),
        "max_gradient_parameter": max_grad_name,
        "nan": bool(grad_nan),
        "inf": bool(grad_inf),
    },

    "delayed_state_interpolation": {
        "synthetic_max_error": float(max_error),
        "sample_delay_seconds": [
            float(x)
            for x in real_delays.detach().cpu().tolist()
        ],
        "prehistory_fallback_count": int(prehistory_count),
        "interpolation_count": int(interpolation_count),
        "future_state_access_violations":
            int(future_access_violations),
        "bracketing_violations":
            int(bracketing_violations),
    },

    "status": "PASSED",
}

with open(
    NUMERICAL_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        diagnostics,
        f,
        indent=2,
    )

print("=" * 60)
print("NUMERICAL DIAGNOSTICS SAVED")
print("=" * 60)
print("Status: PASSED")
print("Saved to:")
print(NUMERICAL_DIAGNOSTICS_PATH)
print("=" * 60)

# 19...

In [ ]:
# ============================================================
# SECTION 19.1 — FREEZE FINAL TEST CONFIGURATION
# ============================================================

import json

FINAL_TEST_CONFIG_PATH = (
    RESULTS_DIR / "final_test_config.json"
)

final_test_config = {
    "selection_split": "VAL",
    "test_split_used_for_selection": False,
    "seed": 42,

    "transformer_baseline": {
        "checkpoint": "03_transformer_best.pth",
        "val_mae": 6.467345480593095,
        "val_rmse": 8.920282765919419,
        "role": "reference baseline",
    },

    "proposed_full_dde": {
        "checkpoint": "04_residual_dde_best_trained.pth",
        "val_mae": 6.492574203088417,
        "val_rmse": 8.839189,
        "role": "proposed method",
    },

    "selection_note": (
        "Transformer achieved lower validation MAE; "
        "full DDE achieved slightly lower validation RMSE "
        "and lower bias. Both configurations are frozen "
        "before official TEST evaluation."
    ),

    "post_test_tuning_allowed": False,
}

with open(
    FINAL_TEST_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_test_config,
        f,
        indent=2,
    )

print("=" * 60)
print("FINAL TEST CONFIGURATION FROZEN")
print("=" * 60)
print("Transformer VAL MAE: 6.467")
print("Full DDE VAL MAE:    6.493")
print()
print("TEST has not been used for model selection.")
print("No architecture/hyperparameter tuning after TEST.")
print()
print("Saved to:")
print(FINAL_TEST_CONFIG_PATH)
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 19.2 — BUILD OFFICIAL TEST DATASET + LOADER
# ============================================================

from torch.utils.data import DataLoader

test_dataset = EchoNetDynamicDataset(
    csv_file=CSV_FILE,
    video_dir=VIDEO_DIR,
    split="TEST",
    frame_sampler=sampler,
    preprocessor=preprocessor,
    augmenter=None,
    roi_processor=roi_detector,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

# Check short videos before DDE evaluation
test_metadata = test_dataset.metadata.copy()

short_test = test_metadata[
    test_metadata["NumberOfFrames"] < 32
]

print("=" * 60)
print("OFFICIAL TEST DATASET")
print("=" * 60)
print("TEST videos:", len(test_dataset))
print("TEST batches:", len(test_loader))
print("Videos with <32 frames:", len(short_test))

if len(short_test) > 0:
    print("\nShort TEST videos:")
    print(
        short_test[
            ["FileName", "NumberOfFrames", "FPS"]
        ].to_string(index=False)
    )

assert len(test_dataset) == 1277

print()
print("TEST dataset prepared successfully.")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 19.3 — OFFICIAL TEST EVALUATION
# ============================================================

best_transformer_model.eval()
diagnostic_model.eval()
diagnostic_model.reliability_extractor.eval()

test_rows = []

print("=" * 60)
print("OFFICIAL TEST EVALUATION")
print("=" * 60)

with torch.no_grad():

    for batch_idx, batch in enumerate(test_loader):

        video = batch["video"].to(
            device,
            non_blocking=True,
        )

        times = batch["sample_times"].to(
            device,
            non_blocking=True,
        )

        true_ef = (
            batch["ef"]
            .to(device)
            .squeeze(1)
        )

        filenames = batch["filename"]

        # ----------------------------------------
        # Transformer baseline
        # ----------------------------------------
        transformer_pred = (
            best_transformer_model(video)
            .squeeze(1)
        )

        # ----------------------------------------
        # Full proposed DDE
        # ----------------------------------------
        dde_pred = (
            diagnostic_model(
                video,
                times,
            )["predicted_ef"]
            .squeeze(1)
        )

        for i, video_id in enumerate(filenames):

            gt = float(true_ef[i].cpu())
            t_pred = float(transformer_pred[i].cpu())
            d_pred = float(dde_pred[i].cpu())

            for model_name, pred in [
                ("Temporal Transformer", t_pred),
                (
                    "Full DDE + reliability + adaptive fusion",
                    d_pred,
                ),
            ]:

                error = pred - gt

                test_rows.append({
                    "video_id": str(video_id),
                    "split": "TEST",
                    "ground_truth_ef": gt,
                    "predicted_ef": pred,
                    "absolute_error": abs(error),
                    "squared_error": error ** 2,
                    "model_name": model_name,
                })

        if (batch_idx + 1) % 20 == 0:
            print(
                f"Completed "
                f"{batch_idx + 1}/{len(test_loader)} "
                f"TEST batches"
            )

# ============================================================
# SAVE PER-VIDEO PREDICTIONS
# ============================================================

predictions_test = pd.DataFrame(test_rows)

PREDICTIONS_TEST_PATH = (
    RESULTS_DIR / "predictions_test.csv"
)

predictions_test.to_csv(
    PREDICTIONS_TEST_PATH,
    index=False,
)

# ============================================================
# COMPUTE TEST METRICS
# ============================================================

test_metric_rows = []

for model_name, df in predictions_test.groupby(
    "model_name"
):

    y_true = df["ground_truth_ef"].to_numpy()
    y_pred = df["predicted_ef"].to_numpy()

    mae = np.mean(
        np.abs(y_pred - y_true)
    )

    rmse = np.sqrt(
        np.mean(
            (y_pred - y_true) ** 2
        )
    )

    bias = np.mean(
        y_pred - y_true
    )

    r = np.corrcoef(
        y_true,
        y_pred,
    )[0, 1]

    r2 = 1 - (
        np.sum(
            (y_true - y_pred) ** 2
        )
        /
        np.sum(
            (y_true - y_true.mean()) ** 2
        )
    )

    test_metric_rows.append({
        "model": model_name,
        "split": "TEST",
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "pearson_r": r,
        "bias": bias,
        "seed": SEED,
        "checkpoint": (
            "03_transformer_best.pth"
            if model_name == "Temporal Transformer"
            else "04_residual_dde_best_trained.pth"
        ),
    })

test_metrics = pd.DataFrame(
    test_metric_rows
)

# ============================================================
# APPEND TO METRICS SUMMARY
# ============================================================

METRICS_SUMMARY_PATH = (
    RESULTS_DIR / "metrics_summary.csv"
)

if METRICS_SUMMARY_PATH.exists():

    metrics_summary = pd.read_csv(
        METRICS_SUMMARY_PATH
    )

    metrics_summary = metrics_summary[
        metrics_summary["split"] != "TEST"
    ]

    metrics_summary = pd.concat(
        [
            metrics_summary,
            test_metrics,
        ],
        ignore_index=True,
    )

else:
    metrics_summary = test_metrics.copy()

metrics_summary.to_csv(
    METRICS_SUMMARY_PATH,
    index=False,
)

# ============================================================
# DISPLAY FINAL TEST RESULTS
# ============================================================

print()
print("=" * 60)
print("OFFICIAL TEST RESULTS")
print("=" * 60)

for _, row in test_metrics.iterrows():

    print(row["model"])
    print(f"  MAE:       {row['mae']:.3f}")
    print(f"  RMSE:      {row['rmse']:.3f}")
    print(f"  R²:        {row['r2']:.3f}")
    print(f"  Pearson r: {row['pearson_r']:.3f}")
    print(f"  Bias:      {row['bias']:.3f}")
    print("-" * 60)

print()
print("Predictions saved to:")
print(PREDICTIONS_TEST_PATH)

print()
print("Metrics saved to:")
print(METRICS_SUMMARY_PATH)

print("=" * 60)

# 20...

In [ ]:
# ============================================================
# SECTION 20.1 — FINAL RESULTS AUDIT
# ============================================================

from pathlib import Path
import pandas as pd

files_to_check = {
    "metrics_summary":
        RESULTS_DIR / "metrics_summary.csv",

    "predictions_val":
        RESULTS_DIR / "predictions_val.csv",

    "predictions_test":
        RESULTS_DIR / "predictions_test.csv",

    "training_history":
        RESULTS_DIR / "training_history.csv",

    "ablation_results":
        RESULTS_DIR / "ablation_results.csv",

    "reliability_traces":
        RESULTS_DIR / "reliability_traces.csv",

    "delay_and_gate":
        RESULTS_DIR / "delay_and_gate.csv",

    "reliability_stratified":
        RESULTS_DIR / "reliability_stratified_metrics.csv",

    "gate_correlations":
        RESULTS_DIR / "gate_reliability_correlations.csv",

    "numerical_diagnostics":
        RESULTS_DIR / "numerical_diagnostics.json",

    "final_test_config":
        RESULTS_DIR / "final_test_config.json",
}

print("=" * 70)
print("FINAL RESULTS AUDIT")
print("=" * 70)

for name, path in files_to_check.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:28s} {status:8s}  {path.name}")

print("\n" + "=" * 70)
print("ABLATION RESULTS")
print("=" * 70)

ablation_results = pd.read_csv(
    RESULTS_DIR / "ablation_results.csv"
)

print(
    ablation_results.to_string(
        index=False
    )
)

print("\n" + "=" * 70)
print("FINAL VAL + TEST METRICS")
print("=" * 70)

metrics_summary = pd.read_csv(
    RESULTS_DIR / "metrics_summary.csv"
)

print(
    metrics_summary.to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# SECTION 20.2 — TABLE III: CORE ABLATION RESULTS
# ============================================================

core_variants = [
    "Temporal Transformer baseline",
    "Neural ODE (no delayed state)",
    "Neural DDE without reliability",
    "Reliability-conditioned DDE without adaptive fusion",
    "Full DDE + reliability + adaptive fusion",
    "Fixed-delay DDE (1.1 s)",
]

table_iii = (
    ablation_results[
        ablation_results["model_variant"].isin(core_variants)
    ]
    .copy()
)

table_iii["model_variant"] = pd.Categorical(
    table_iii["model_variant"],
    categories=core_variants,
    ordered=True,
)

table_iii = (
    table_iii
    .sort_values("model_variant")
    [
        [
            "model_variant",
            "mae",
            "rmse",
            "r2",
            "pearson_r",
            "bias",
        ]
    ]
)

table_iii.columns = [
    "Model Variant",
    "MAE",
    "RMSE",
    "R²",
    "Pearson r",
    "Bias",
]

TABLE_III_PATH = (
    RESULTS_DIR / "table_iii_core_ablations.csv"
)

table_iii.to_csv(
    TABLE_III_PATH,
    index=False,
)

print("=" * 85)
print("TABLE III — CORE ABLATION RESULTS")
print("=" * 85)

print(
    table_iii.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

print()
print("Saved to:")
print(TABLE_III_PATH)

In [ ]:
# ============================================================
# SECTION 20.3 — INSPECT TABLE IV SOURCE DATA
# ============================================================

reliability_stratified = pd.read_csv(
    RESULTS_DIR / "reliability_stratified_metrics.csv"
)

component_variants = [
    "Full DDE + reliability + adaptive fusion",
    "DDE without Q",
    "DDE without M",
    "DDE without B",
    "DDE without C",
]

component_results = ablation_results[
    ablation_results["model_variant"].isin(component_variants)
].copy()

print("=" * 80)
print("RELIABILITY COMPONENT REMOVALS")
print("=" * 80)
print(component_results.to_string(index=False))

print("\n" + "=" * 80)
print("RELIABILITY-STRATIFIED METRICS")
print("=" * 80)
print("Columns:", reliability_stratified.columns.tolist())
print()
print(reliability_stratified.to_string(index=False))

In [ ]:
# ============================================================
# SECTION 20.4 — TABLE IV: RELIABILITY ANALYSIS
# ============================================================

# ------------------------------------------------------------
# PANEL A — Reliability-component removals
# ------------------------------------------------------------

component_order = [
    "Full DDE + reliability + adaptive fusion",
    "DDE without Q",
    "DDE without M",
    "DDE without B",
    "DDE without C",
]

table_iv_a = (
    component_results[
        component_results["model_variant"].isin(component_order)
    ]
    .copy()
)

table_iv_a["model_variant"] = pd.Categorical(
    table_iv_a["model_variant"],
    categories=component_order,
    ordered=True,
)

table_iv_a = table_iv_a.sort_values(
    "model_variant"
).reset_index(drop=True)

full_mae = table_iv_a.loc[
    table_iv_a["model_variant"]
    == "Full DDE + reliability + adaptive fusion",
    "mae",
].iloc[0]

full_rmse = table_iv_a.loc[
    table_iv_a["model_variant"]
    == "Full DDE + reliability + adaptive fusion",
    "rmse",
].iloc[0]

table_iv_a["delta_mae_vs_full"] = (
    table_iv_a["mae"] - full_mae
)

table_iv_a["delta_rmse_vs_full"] = (
    table_iv_a["rmse"] - full_rmse
)

table_iv_a = table_iv_a[
    [
        "model_variant",
        "mae",
        "rmse",
        "r2",
        "bias",
        "delta_mae_vs_full",
        "delta_rmse_vs_full",
    ]
]

table_iv_a.columns = [
    "Model Variant",
    "MAE",
    "RMSE",
    "R²",
    "Bias",
    "ΔMAE vs Full",
    "ΔRMSE vs Full",
]

# ------------------------------------------------------------
# PANEL B — Reliability-stratified performance
# ------------------------------------------------------------

group_order = ["Low", "Medium", "High"]

table_iv_b = reliability_stratified.copy()

table_iv_b["reliability_group"] = pd.Categorical(
    table_iv_b["reliability_group"],
    categories=group_order,
    ordered=True,
)

table_iv_b = (
    table_iv_b
    .sort_values(
        ["reliability_group", "model"]
    )
    .reset_index(drop=True)
)

table_iv_b = table_iv_b[
    [
        "reliability_group",
        "model",
        "n",
        "mae",
        "rmse",
        "bias",
    ]
]

table_iv_b.columns = [
    "Reliability Group",
    "Model",
    "N",
    "MAE",
    "RMSE",
    "Bias",
]

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

TABLE_IV_A_PATH = (
    RESULTS_DIR
    / "table_iv_a_reliability_components.csv"
)

TABLE_IV_B_PATH = (
    RESULTS_DIR
    / "table_iv_b_reliability_stratified.csv"
)

table_iv_a.to_csv(
    TABLE_IV_A_PATH,
    index=False,
)

table_iv_b.to_csv(
    TABLE_IV_B_PATH,
    index=False,
)

print("=" * 90)
print("TABLE IV-A — RELIABILITY COMPONENT REMOVALS")
print("=" * 90)

print(
    table_iv_a.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

print("\n" + "=" * 90)
print("TABLE IV-B — RELIABILITY-STRATIFIED PERFORMANCE")
print("=" * 90)

print(
    table_iv_b.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

print("\nSaved:")
print(TABLE_IV_A_PATH)
print(TABLE_IV_B_PATH)

print("\nReliability tertile thresholds:")
print("Low < 0.530")
print("Medium: 0.530–0.557")
print("High > 0.557")

In [ ]:
# ============================================================
# SECTION 20.5 — TABLE I: DATASET + MODEL CONFIGURATION
# ============================================================

table_i_rows = [
    ("Dataset", "EchoNet-Dynamic"),
    ("TRAIN videos", "7,465"),
    ("DDE TRAIN videos", "7,462"),
    ("Excluded DDE TRAIN videos", "3 (<32 frames)"),
    ("Validation videos", "1,288"),
    ("Test videos", "1,277"),

    ("Input view", "Apical four-chamber echocardiography video"),
    ("Frames per video", "32 ordered frames"),
    ("Input resolution", "224 × 224"),
    ("ROI", "Motion-based cardiac ROI"),
    ("TRAIN augmentation", "Temporally consistent"),
    ("VAL/TEST augmentation", "None"),

    ("Spatial encoder", "ConvNeXt-Tiny"),
    ("Frame feature dimension", "768"),
    ("Transformer layers", "2"),
    ("Transformer attention heads", "8"),
    ("Transformer feedforward dimension", "1536"),
    ("Transformer dropout", "0.1"),

    ("DDE latent dimension", "256"),
    ("DDE hidden dimension", "512"),
    ("Reliability inputs", "Q, M, B, C"),
    ("Delay units", "Seconds"),
    ("Delayed-state retrieval", "Linear interpolation"),
    ("Numerical update", "Explicit Euler"),

    ("DDE batch size", "8"),
    ("Maximum epochs", "10"),
    ("Early-stopping patience", "3"),
    ("Loss", "Mean squared error"),
    ("Optimizer", "AdamW"),
    ("DDE learning rate", "1e-4"),
    ("Residual-fusion learning rate", "3e-4"),
    ("Weight decay", "1e-4"),
    ("Random seed", "42"),
    ("Mixed precision", "Enabled"),

    (
        "Frozen during DDE training",
        "ConvNeXt + Temporal Transformer + pretrained EF head",
    ),
    (
        "Trainable during DDE training",
        "Neural DDE + residual adaptive-fusion module",
    ),
]

table_i = pd.DataFrame(
    table_i_rows,
    columns=["Setting", "Value"],
)

TABLE_I_PATH = (
    RESULTS_DIR / "table_i_dataset_model_training.csv"
)

table_i.to_csv(
    TABLE_I_PATH,
    index=False,
)

print("=" * 80)
print("TABLE I — DATASET, INPUT, MODEL, AND TRAINING SETTINGS")
print("=" * 80)

print(
    table_i.to_string(
        index=False
    )
)

print()
print("Saved to:")
print(TABLE_I_PATH)

In [ ]:
# ============================================================
# SECTION 20.6 — VERIFIED EXTERNAL LITERATURE COMPARISON
# ============================================================

import numpy as np
import pandas as pd

literature_rows = [
    {
        "author_year": "Ouyang et al., 2020",
        "paper_title":
            "Video-based AI for beat-to-beat assessment of cardiac function",
        "model": "EchoNet-Dynamic / R(2+1)D CNN",
        "reported_mae": 4.10,
        "reported_rmse": np.nan,
        "reported_r2": np.nan,
        "dataset_split":
            "EchoNet-Dynamic TEST (1,277 videos)",
        "evaluation_protocol":
            "Video-based EF regression; original EchoNet-Dynamic study",
        "source_doi":
            "10.1038/s41586-020-2145-8",
        "result_type": "Literature-reported",
    },

    {
        "author_year": "Reynaud et al., 2021",
        "paper_title":
            "Ultrasound Video Transformers for Cardiac Ejection Fraction Estimation",
        "model": "Residual Autoencoder + BERT",
        "reported_mae": 5.95,
        "reported_rmse": np.nan,
        "reported_r2": 0.52,
        "dataset_split":
            "EchoNet-Dynamic; paper-reported evaluation",
        "evaluation_protocol":
            "Arbitrary-length video; ED/ES detection + EF estimation",
        "source_doi":
            "10.1007/978-3-030-87231-1_48",
        "result_type": "Literature-reported",
    },

    {
        "author_year": "Muhtaseb & Yaqub, 2022",
        "paper_title":
            "EchoCoTr: Estimation of the Left Ventricular Ejection Fraction "
            "from Spatiotemporal Echocardiography",
        "model": "EchoCoTr-S",
        "reported_mae": 3.95,
        "reported_rmse": np.nan,
        "reported_r2": 0.82,
        "dataset_split":
            "EchoNet-Dynamic; paper-reported evaluation",
        "evaluation_protocol":
            "CNN + Transformer; best reported 36-frame configuration",
        "source_doi":
            "10.1007/978-3-031-16440-8_36",
        "result_type": "Literature-reported",
    },

    {
        "author_year": "Zeng et al., 2023",
        "paper_title":
            "MAEF-Net: Multi-attention efficient feature fusion network "
            "for left ventricular segmentation and quantitative analysis "
            "in two-dimensional echocardiography",
        "model": "MAEF-Net",
        "reported_mae": 6.29,
        "reported_rmse": np.nan,
        "reported_r2": np.nan,
        "dataset_split":
            "EchoNet-Dynamic subset (n=1,226)",
        "evaluation_protocol":
            "LV segmentation + ED/ES detection + LVEF computation",
        "source_doi":
            "10.1016/j.ultras.2022.106855",
        "result_type": "Literature-reported",
    },

    {
        "author_year": "Maani et al., 2024",
        "paper_title":
            "CoReEcho: Continuous Representation Learning for "
            "2D+time Echocardiography Analysis",
        "model": "CoReEcho",
        "reported_mae": 3.90,
        "reported_rmse": np.nan,
        "reported_r2": 0.8244,
        "dataset_split":
            "EchoNet-Dynamic; paper-reported evaluation",
        "evaluation_protocol":
            "Continuous-representation learning for direct EF regression",
        "source_doi":
            "10.1007/978-3-031-72083-3_55",
        "result_type": "Literature-reported",
    },

    {
        "author_year": "Ali et al., 2025",
        "paper_title":
            "EFNet: estimation of left ventricular ejection fraction "
            "from cardiac ultrasound videos using deep learning",
        "model": "EFNet (16-frame configuration)",
        "reported_mae": 3.79,
        "reported_rmse": 4.95,
        "reported_r2": 0.82,
        "dataset_split":
            "EchoNet-Dynamic TEST; paper-reported configuration",
        "evaluation_protocol":
            "CNN + residual transformer; 16-frame reported best configuration",
        "source_doi":
            "10.7717/peerj-cs.2506",
        "result_type": "Literature-reported",
    },

    {
        "author_year": "Akan et al., 2025",
        "paper_title":
            "ViViEchoformer: Deep Video Regressor Predicting Ejection Fraction",
        "model": "ViViEchoformer",
        "reported_mae": 6.14,
        "reported_rmse": 8.40,
        "reported_r2": 0.55,
        "dataset_split":
            "EchoNet-Dynamic; paper-reported evaluation",
        "evaluation_protocol":
            "Video Vision Transformer direct LVEF regression",
        "source_doi":
            "10.1007/s10278-024-01336-y",
        "result_type": "Literature-reported",
    },

    {
        "author_year": "Echo-SMADS authors, 2026",
        "paper_title":
            "Echo-SMADS: A hierarchical planning model for predicting "
            "ejection fraction using echocardiography",
        "model": "Echo-SMADS",
        "reported_mae": 5.48,
        "reported_rmse": np.nan,
        "reported_r2": np.nan,
        "dataset_split":
            "EchoNet-Dynamic; paper-reported evaluation",
        "evaluation_protocol":
            "Hierarchical structure, cardiac phase, and volume estimation",
        "source_doi":
            "10.1016/j.cmpb.2026.109377",
        "result_type": "Literature-reported",
    },
]

literature_comparison = pd.DataFrame(
    literature_rows
)

LITERATURE_COMPARISON_PATH = (
    RESULTS_DIR / "literature_comparison.csv"
)

literature_comparison.to_csv(
    LITERATURE_COMPARISON_PATH,
    index=False,
)

print("=" * 100)
print("VERIFIED EXTERNAL LITERATURE COMPARISON")
print("=" * 100)

print(
    literature_comparison[
        [
            "author_year",
            "model",
            "reported_mae",
            "reported_rmse",
            "reported_r2",
            "dataset_split",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

print()
print("Saved to:")
print(LITERATURE_COMPARISON_PATH)

print()
print(
    "IMPORTANT: Literature results use different architectures, "
    "frame sampling, preprocessing, and evaluation protocols. "
    "Do not treat them as directly controlled comparisons."
)

In [ ]:
# ============================================================
# SECTION 20.7 — TABLE II: BASELINE PROGRESSION + LITERATURE
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PANEL A — OUR LOCALLY REPRODUCED MODEL PROGRESSION
# ------------------------------------------------------------

internal_rows = [
    {
        "Model": "Frozen ConvNeXt baseline",
        "Split": "VAL",
        "MAE": 8.495,
        "RMSE": 11.098,
        "R²": np.nan,
        "Pearson r": np.nan,
        "Bias": np.nan,
    },
    {
        "Model": "Partially fine-tuned ConvNeXt",
        "Split": "VAL",
        "MAE": 7.386,
        "RMSE": 9.584,
        "R²": np.nan,
        "Pearson r": np.nan,
        "Bias": np.nan,
    },
    {
        "Model": "Temporal Transformer",
        "Split": "VAL",
        "MAE": 6.467105,
        "RMSE": 8.919678,
        "R²": 0.474550,
        "Pearson r": 0.699192,
        "Bias": 1.449843,
    },
    {
        "Model": "Full DDE + reliability + adaptive fusion",
        "Split": "VAL",
        "MAE": 6.492574,
        "RMSE": 8.839189,
        "R²": 0.483991,
        "Pearson r": 0.698814,
        "Bias": 0.601863,
    },
    {
        "Model": "Temporal Transformer",
        "Split": "TEST",
        "MAE": 6.510764,
        "RMSE": 8.989979,
        "R²": 0.459265,
        "Pearson r": 0.692352,
        "Bias": 1.729047,
    },
    {
        "Model": "Full DDE + reliability + adaptive fusion",
        "Split": "TEST",
        "MAE": 6.500757,
        "RMSE": 8.898299,
        "R²": 0.470237,
        "Pearson r": 0.691812,
        "Bias": 0.876408,
    },
]

table_ii_a = pd.DataFrame(internal_rows)

# ------------------------------------------------------------
# PANEL B — PUBLISHED LITERATURE
# ------------------------------------------------------------

table_ii_b = literature_comparison[
    [
        "author_year",
        "model",
        "reported_mae",
        "reported_rmse",
        "reported_r2",
        "dataset_split",
        "evaluation_protocol",
        "source_doi",
    ]
].copy()

table_ii_b.columns = [
    "Study",
    "Model",
    "Reported MAE",
    "Reported RMSE",
    "Reported R²",
    "Dataset / Split",
    "Evaluation Protocol",
    "DOI",
]

# ------------------------------------------------------------
# SAVE BOTH PANELS
# ------------------------------------------------------------

TABLE_II_A_PATH = (
    RESULTS_DIR / "table_ii_a_internal_progression.csv"
)

TABLE_II_B_PATH = (
    RESULTS_DIR / "table_ii_b_external_literature.csv"
)

table_ii_a.to_csv(
    TABLE_II_A_PATH,
    index=False,
)

table_ii_b.to_csv(
    TABLE_II_B_PATH,
    index=False,
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 95)
print("TABLE II-A — LOCALLY REPRODUCED MODEL PROGRESSION")
print("=" * 95)

print(
    table_ii_a.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

print("\n" + "=" * 95)
print("TABLE II-B — PUBLISHED LITERATURE")
print("=" * 95)

print(
    table_ii_b[
        [
            "Study",
            "Model",
            "Reported MAE",
            "Reported RMSE",
            "Reported R²",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)

print("\nSaved:")
print(TABLE_II_A_PATH)
print(TABLE_II_B_PATH)

print()
print(
    "NOTE: Panel A contains locally reproduced results. "
    "Panel B contains literature-reported results obtained "
    "under differing preprocessing, sampling, architectures, "
    "and evaluation protocols."
)

# 21...

In [ ]:
# ============================================================
# SECTION 21.1 — INSPECT FIGURE 3 SOURCE DATA
# ============================================================

import pandas as pd

training_history = pd.read_csv(
    RESULTS_DIR / "training_history.csv"
)

reliability_traces = pd.read_csv(
    RESULTS_DIR / "reliability_traces.csv"
)

print("=" * 80)
print("TRAINING HISTORY")
print("=" * 80)
print("Columns:", training_history.columns.tolist())
print()
print(training_history.to_string(index=False))

print("\n" + "=" * 80)
print("RELIABILITY TRACES")
print("=" * 80)
print("Columns:", reliability_traces.columns.tolist())
print()
print(reliability_traces.head(15).to_string(index=False))

print("\nUnique videos:")
print(reliability_traces["video_id"].unique())

In [ ]:
# ============================================================
# SECTION 21.2 — FIGURE 3: TRAINING + RELIABILITY TRACES
# ============================================================

import matplotlib.pyplot as plt

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(
    2, 2,
    figsize=(10, 7),
    constrained_layout=True
)

# ------------------------------------------------------------
# A — Training / validation convergence
# ------------------------------------------------------------

ax = axes[0, 0]

ax.plot(
    training_history["epoch"],
    training_history["train_rmse"],
    marker="o",
    label="Train RMSE"
)

ax.plot(
    training_history["epoch"],
    training_history["val_rmse"],
    marker="s",
    label="Validation RMSE"
)

ax.plot(
    training_history["epoch"],
    training_history["val_mae"],
    marker="^",
    label="Validation MAE"
)

ax.set_xlabel("Epoch")
ax.set_ylabel("EF error (percentage points)")
ax.set_xticks(training_history["epoch"])
ax.grid(alpha=0.25)
ax.legend(fontsize=8)

ax.text(
    -0.14, 1.08,
    "(a)",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold"
)

# ------------------------------------------------------------
# B–D — Representative reliability traces
# ------------------------------------------------------------

groups = ["Low", "Medium", "High"]
trace_axes = [
    axes[0, 1],
    axes[1, 0],
    axes[1, 1],
]

for ax, group in zip(trace_axes, groups):

    df = reliability_traces[
        reliability_traces["reliability_group"] == group
    ].copy()

    video_id = df["video_id"].iloc[0]

    for feature in ["Q", "M", "B", "C"]:
        ax.plot(
            df["time_seconds"],
            df[feature],
            linewidth=1.6,
            label=feature
        )

    ax.set_title(
        f"{group}-reliability representative",
        fontsize=10
    )

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Reliability proxy")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.25)

    ax.text(
        0.02, 0.04,
        f"Video: {video_id}",
        transform=ax.transAxes,
        fontsize=7
    )

# Panel labels
for label, ax in zip(
    ["(b)", "(c)", "(d)"],
    trace_axes
):
    ax.text(
        -0.14, 1.08,
        label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold"
    )

# Single legend for Q/M/B/C panels
handles, labels = trace_axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center right",
    bbox_to_anchor=(1.08, 0.50),
    ncol=1,
    fontsize=8
)

fig.suptitle(
    "Training Convergence and Representative Reliability-Proxy Dynamics",
    fontsize=13
)

# ------------------------------------------------------------
# SAVE HIGH-RESOLUTION FIGURE
# ------------------------------------------------------------

FIG3_PNG = FIGURES_DIR / "figure_3_training_reliability.png"
FIG3_PDF = FIGURES_DIR / "figure_3_training_reliability.pdf"

fig.savefig(
    FIG3_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG3_PDF,
    bbox_inches="tight"
)

plt.show()

print("=" * 70)
print("FIGURE 3 SAVED")
print("=" * 70)
print(FIG3_PNG)
print(FIG3_PDF)

In [ ]:
# ============================================================
# SECTION 21.3 — FIGURE 4: TEST PREDICTIONS + ERROR COMPARISON
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

predictions_test = pd.read_csv(
    RESULTS_DIR / "predictions_test.csv"
)

transformer_test = predictions_test[
    predictions_test["model_name"] == "Temporal Transformer"
].copy()

dde_test = predictions_test[
    predictions_test["model_name"]
    == "Full DDE + reliability + adaptive fusion"
].copy()

fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 4.2),
    constrained_layout=True
)

# ------------------------------------------------------------
# A — Actual vs predicted EF
# ------------------------------------------------------------

ax = axes[0]

ax.scatter(
    transformer_test["ground_truth_ef"],
    transformer_test["predicted_ef"],
    s=12,
    alpha=0.35,
    label="Temporal Transformer",
)

ax.scatter(
    dde_test["ground_truth_ef"],
    dde_test["predicted_ef"],
    s=12,
    alpha=0.35,
    marker="x",
    label="Full DDE",
)

all_ef = np.concatenate([
    transformer_test["ground_truth_ef"].to_numpy(),
    transformer_test["predicted_ef"].to_numpy(),
    dde_test["predicted_ef"].to_numpy(),
])

limit_min = np.floor(all_ef.min() / 5) * 5
limit_max = np.ceil(all_ef.max() / 5) * 5

ax.plot(
    [limit_min, limit_max],
    [limit_min, limit_max],
    linestyle="--",
    linewidth=1.2,
    label="Ideal prediction",
)

ax.set_xlim(limit_min, limit_max)
ax.set_ylim(limit_min, limit_max)

ax.set_xlabel("Ground-truth EF (%)")
ax.set_ylabel("Predicted EF (%)")
ax.set_title("Actual vs. Predicted EF")
ax.grid(alpha=0.2)
ax.legend(fontsize=8)

ax.text(
    -0.13, 1.06,
    "(a)",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold",
)

# ------------------------------------------------------------
# B — Absolute-error distributions
# ------------------------------------------------------------

ax = axes[1]

error_data = [
    transformer_test["absolute_error"].to_numpy(),
    dde_test["absolute_error"].to_numpy(),
]

ax.boxplot(
    error_data,
    tick_labels=[
        "Transformer",
        "Full DDE",
    ],
    showfliers=False,
)

ax.set_ylabel("Absolute EF error (percentage points)")
ax.set_title("TEST Absolute-Error Distribution")
ax.grid(axis="y", alpha=0.2)

transformer_mae = transformer_test["absolute_error"].mean()
dde_mae = dde_test["absolute_error"].mean()

ax.text(
    1,
    transformer_mae,
    f"MAE = {transformer_mae:.3f}",
    ha="center",
    va="bottom",
    fontsize=8,
)

ax.text(
    2,
    dde_mae,
    f"MAE = {dde_mae:.3f}",
    ha="center",
    va="bottom",
    fontsize=8,
)

ax.text(
    -0.13, 1.06,
    "(b)",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold",
)

fig.suptitle(
    "Official TEST-Set EF Prediction Performance",
    fontsize=13,
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

FIG4_PNG = (
    FIGURES_DIR
    / "figure_4_test_prediction_performance.png"
)

FIG4_PDF = (
    FIGURES_DIR
    / "figure_4_test_prediction_performance.pdf"
)

fig.savefig(
    FIG4_PNG,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    FIG4_PDF,
    bbox_inches="tight",
)

plt.show()

print("=" * 70)
print("FIGURE 4 SAVED")
print("=" * 70)
print(FIG4_PNG)
print(FIG4_PDF)

In [ ]:
# ============================================================
# SECTION 21.4 — FIGURE 5: RELIABILITY GROUPS + FUSION GATE
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

reliability_stratified = pd.read_csv(
    RESULTS_DIR / "reliability_stratified_metrics.csv"
)

delay_and_gate = pd.read_csv(
    RESULTS_DIR / "delay_and_gate.csv"
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 4.2),
    constrained_layout=True
)

# ------------------------------------------------------------
# A — MAE by reliability tertile
# ------------------------------------------------------------

ax = axes[0]

groups = ["Low", "Medium", "High"]
x = np.arange(len(groups))
width = 0.35

transformer_mae = []
dde_mae = []

for group in groups:

    transformer_mae.append(
        reliability_stratified[
            (reliability_stratified["reliability_group"] == group)
            &
            (reliability_stratified["model"] == "Temporal Transformer")
        ]["mae"].iloc[0]
    )

    dde_mae.append(
        reliability_stratified[
            (reliability_stratified["reliability_group"] == group)
            &
            (reliability_stratified["model"] == "Residual DDE")
        ]["mae"].iloc[0]
    )

ax.bar(
    x - width / 2,
    transformer_mae,
    width,
    label="Temporal Transformer"
)

ax.bar(
    x + width / 2,
    dde_mae,
    width,
    label="Full DDE"
)

ax.set_xticks(x)
ax.set_xticklabels(groups)

ax.set_xlabel("Relative reliability tertile")
ax.set_ylabel("Validation MAE (percentage points)")
ax.set_title("Performance by Reliability Group")

ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.2)

ax.text(
    -0.13, 1.06,
    "(a)",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold"
)

# ------------------------------------------------------------
# B — Fusion gate vs overall reliability proxy
# ------------------------------------------------------------

ax = axes[1]

x_rel = delay_and_gate["reliability_mean"].to_numpy()
y_gate = delay_and_gate["fusion_gate"].to_numpy()

ax.scatter(
    x_rel,
    y_gate,
    s=14,
    alpha=0.35
)

# Linear trend
coef = np.polyfit(
    x_rel,
    y_gate,
    1
)

x_line = np.linspace(
    x_rel.min(),
    x_rel.max(),
    100
)

y_line = (
    coef[0] * x_line
    + coef[1]
)

ax.plot(
    x_line,
    y_line,
    linestyle="--",
    linewidth=1.5
)

pearson_r = np.corrcoef(
    x_rel,
    y_gate
)[0, 1]

ax.set_xlabel("Mean reliability-proxy score")
ax.set_ylabel("Fusion gate")
ax.set_title("Adaptive Fusion Gate vs. Reliability")

ax.grid(alpha=0.2)

ax.text(
    0.04,
    0.94,
    f"Pearson r = {pearson_r:.3f}",
    transform=ax.transAxes,
    va="top",
    fontsize=9
)

ax.text(
    -0.13, 1.06,
    "(b)",
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold"
)

fig.suptitle(
    "Reliability-Stratified Performance and Adaptive Fusion Behavior",
    fontsize=13
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

FIG5_PNG = (
    FIGURES_DIR
    / "figure_5_reliability_gate.png"
)

FIG5_PDF = (
    FIGURES_DIR
    / "figure_5_reliability_gate.pdf"
)

fig.savefig(
    FIG5_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG5_PDF,
    bbox_inches="tight"
)

plt.show()

print("=" * 70)
print("FIGURE 5 SAVED")
print("=" * 70)
print(FIG5_PNG)
print(FIG5_PDF)

In [ ]:
# ============================================================
# SECTION 21.5 — FIGURE 2: NEURAL DDE MECHANISM
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 4.5))

ax.set_xlim(0, 11)
ax.set_ylim(0, 5)
ax.axis("off")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def box(x, y, w, h, text):

    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.03",
        linewidth=1.4,
        fill=False
    )

    ax.add_patch(patch)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=10
    )


def arrow(x1, y1, x2, y2, text=None):

    a = FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        arrowstyle="->",
        mutation_scale=14,
        linewidth=1.3
    )

    ax.add_patch(a)

    if text is not None:

        ax.text(
            (x1 + x2) / 2,
            (y1 + y2) / 2 + 0.16,
            text,
            ha="center",
            fontsize=9
        )


# ------------------------------------------------------------
# INPUT BOXES
# ------------------------------------------------------------

box(
    0.4, 3.3,
    1.8, 0.8,
    "Current latent\n$z(t)$"
)

box(
    0.4, 1.7,
    1.8, 0.8,
    "Delayed latent\n$z(t-\\tau)$"
)

box(
    3.0, 3.3,
    1.9, 0.8,
    "Transformer\nfeature $h(t)$"
)

box(
    3.0, 1.7,
    1.9, 0.8,
    "Reliability proxies\n$Q,M,B,C$"
)


# ------------------------------------------------------------
# DDE DYNAMICS
# ------------------------------------------------------------

box(
    5.8, 2.35,
    2.0, 1.1,
    "Reliability-conditioned\nDDE dynamics\n$F_\\theta$"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

box(
    8.7, 2.35,
    1.8, 1.1,
    "State derivative\n$\\dot{z}(t)$"
)


# ------------------------------------------------------------
# CONNECTIONS
# ------------------------------------------------------------

# Current latent -> DDE
arrow(
    2.2, 3.7,
    5.8, 3.05
)

# Transformer feature -> DDE
arrow(
    4.9, 3.7,
    5.8, 3.15
)

# Reliability proxies -> DDE
arrow(
    4.9, 2.1,
    5.8, 2.72
)

# ------------------------------------------------------------
# Delayed-state path
# Route BELOW reliability box so lines do not overlap it
# ------------------------------------------------------------

ax.plot(
    [2.2, 2.55, 5.05],
    [2.1, 1.30, 1.30],
    linewidth=1.3
)

arrow(
    5.05, 1.30,
    5.8, 2.48
)

ax.text(
    3.55,
    1.08,
    r"delay $\tau$",
    fontsize=9,
    ha="center"
)


# DDE -> state derivative
arrow(
    7.8, 2.9,
    8.7, 2.9
)


# ------------------------------------------------------------
# EULER UPDATE EQUATION
# ------------------------------------------------------------

ax.text(
    5.5,
    0.72,
    r"$z(t+\Delta t)=z(t)+\Delta t\,"
    r"F_\theta(z(t),z(t-\tau),h(t),r(t))$",
    ha="center",
    fontsize=11
)


# ------------------------------------------------------------
# INTERPOLATION EXPLANATION
# ------------------------------------------------------------

ax.text(
    5.5,
    0.28,
    "Delayed state obtained by linear interpolation "
    "of previously computed latent states",
    ha="center",
    fontsize=9
)


# ------------------------------------------------------------
# TITLE
# ------------------------------------------------------------

ax.set_title(
    "Reliability-Conditioned Neural Delay Differential Equation",
    fontsize=13
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

FIG2_PNG = (
    FIGURES_DIR
    / "figure_2_neural_dde_mechanism.png"
)

FIG2_PDF = (
    FIGURES_DIR
    / "figure_2_neural_dde_mechanism.pdf"
)

fig.savefig(
    FIG2_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG2_PDF,
    bbox_inches="tight"
)

plt.show()

print("=" * 70)
print("FIGURE 2 SAVED")
print("=" * 70)
print(FIG2_PNG)
print(FIG2_PDF)

In [ ]:
# ============================================================
# SECTION 21.6 — FIGURE 1: COMPLETE MODEL ARCHITECTURE
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(
    figsize=(15, 6.2)
)

ax.set_xlim(0, 15)
ax.set_ylim(0, 7)
ax.axis("off")


# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------

def box(
    x, y, w, h,
    text,
    fontsize=10,
    linewidth=1.4,
    linestyle="-"
):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.035",
        linewidth=linewidth,
        linestyle=linestyle,
        fill=False
    )

    ax.add_patch(patch)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize
    )


def arrow(
    x1, y1,
    x2, y2,
    text=None,
    text_offset=0.16
):
    patch = FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        arrowstyle="->",
        mutation_scale=14,
        linewidth=1.3
    )

    ax.add_patch(patch)

    if text is not None:
        ax.text(
            (x1 + x2) / 2,
            (y1 + y2) / 2 + text_offset,
            text,
            ha="center",
            va="center",
            fontsize=8.5
        )


# ============================================================
# TOP ROW — SHARED SPATIOTEMPORAL BACKBONE
# ============================================================

box(
    0.3, 4.85,
    1.5, 1.0,
    "EchoNet-Dynamic\nA4C video\n$V$"
)

box(
    2.25, 4.85,
    1.8, 1.0,
    "ROI extraction\n+ 32 ordered\nframes"
)

box(
    4.55, 4.85,
    1.65, 1.0,
    "ConvNeXt\nspatial encoder"
)

box(
    6.75, 4.85,
    1.9, 1.0,
    "Temporal\nTransformer"
)

box(
    9.15, 4.85,
    1.75, 1.0,
    "Transformer\nEF head"
)

# Backbone arrows
arrow(
    1.8, 5.35,
    2.25, 5.35
)

arrow(
    4.05, 5.35,
    4.55, 5.35
)

arrow(
    6.2, 5.35,
    6.75, 5.35,
    "768-D/frame"
)

arrow(
    8.65, 5.35,
    9.15, 5.35
)


# ============================================================
# BASELINE EF OUTPUT
# ============================================================

box(
    11.4, 4.85,
    1.55, 1.0,
    "Baseline EF\n$\\hat{y}_{T}$"
)

arrow(
    10.9, 5.35,
    11.4, 5.35
)


# ============================================================
# LOWER BRANCH — RELIABILITY EXTRACTION
# ============================================================

box(
    4.65, 2.55,
    2.05, 1.15,
    "Reliability-proxy\nextraction\n$Q,M,B,C$"
)

# branch from ConvNeXt / temporal representation
arrow(
    5.35, 4.85,
    5.65, 3.70
)

ax.text(
    4.55,
    3.93,
    "frame / temporal\ninformation",
    fontsize=8,
    ha="center"
)


# ============================================================
# DELAY ESTIMATION
# ============================================================

box(
    7.25, 2.55,
    1.75, 1.15,
    "Cycle-based\nper-video delay\n$\\tau$ (s)"
)

arrow(
    6.70, 3.12,
    7.25, 3.12
)


# ============================================================
# NEURAL DDE
# ============================================================

box(
    9.55, 2.35,
    2.15, 1.55,
    "Reliability-conditioned\nNeural DDE\n\n"
    "$F_\\theta(z(t),z(t-\\tau),$\n"
    "$h(t),r(t))$",
    fontsize=9.5
)

# Transformer temporal features -> DDE
arrow(
    7.75, 4.85,
    10.05, 3.90,
    "$h(t)$"
)

# Reliability -> DDE
arrow(
    6.70, 2.95,
    9.55, 2.95,
    "$r(t)$"
)

# Delay -> DDE
arrow(
    9.00, 3.12,
    9.55, 3.12,
    "$\\tau$",
    text_offset=0.22
)


# ============================================================
# DDE RESIDUAL CORRECTION
# ============================================================

box(
    12.15, 2.55,
    1.7, 1.15,
    "DDE residual\ncorrection\n$\\Delta \\hat{y}$"
)

arrow(
    11.70, 3.12,
    12.15, 3.12
)


# ============================================================
# ADAPTIVE FUSION
# ============================================================

box(
    11.30, 0.45,
    2.25, 1.20,
    "Adaptive residual fusion\n\n"
    "$\\hat{y}=\\hat{y}_{T}"
    "+g\\,\\Delta\\hat{y}$"
)

# Baseline EF -> fusion
ax.plot(
    [12.18, 12.18, 12.02],
    [4.85, 1.95, 1.65],
    linewidth=1.3
)

arrow(
    12.02, 1.65,
    12.02, 1.55
)

# DDE correction -> fusion
arrow(
    13.00, 2.55,
    12.85, 1.65
)


# ============================================================
# FINAL EF OUTPUT
# ============================================================

box(
    13.95, 0.55,
    0.85, 1.0,
    "Final\nEF (%)"
)

arrow(
    13.55, 1.05,
    13.95, 1.05
)


# ============================================================
# GATE ANNOTATION
# ============================================================

ax.text(
    10.10,
    1.13,
    "learned gate $g$",
    fontsize=8.5,
    ha="center"
)

arrow(
    10.55, 1.10,
    11.30, 1.10
)


# ============================================================
# FROZEN / TRAINABLE ANNOTATIONS
# ============================================================

ax.text(
    7.55,
    6.35,
    "Frozen during DDE training",
    fontsize=9,
    ha="center",
    fontweight="bold"
)

ax.plot(
    [4.45, 10.95],
    [6.15, 6.15],
    linewidth=1.1,
    linestyle="--"
)

ax.text(
    10.85,
    4.15,
    "Trainable DDE\nand fusion branch",
    fontsize=8.5,
    ha="center",
    fontweight="bold"
)


# ============================================================
# TITLE
# ============================================================

ax.set_title(
    "Reliability-Aware Neural DDE Framework for "
    "Ejection Fraction Estimation",
    fontsize=14,
    pad=14
)


# ============================================================
# FOOTNOTE
# ============================================================

ax.text(
    7.5,
    0.05,
    "Q: image-quality proxy; M: motion-consistency proxy; "
    "B: edge/boundary-consistency proxy; "
    "C: cardiac-cycle-consistency proxy.",
    ha="center",
    fontsize=8
)


# ============================================================
# SAVE
# ============================================================

FIG1_PNG = (
    FIGURES_DIR
    / "figure_1_overall_architecture.png"
)

FIG1_PDF = (
    FIGURES_DIR
    / "figure_1_overall_architecture.pdf"
)

fig.savefig(
    FIG1_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG1_PDF,
    bbox_inches="tight"
)

plt.show()

print("=" * 70)
print("FIGURE 1 SAVED")
print("=" * 70)
print(FIG1_PNG)
print(FIG1_PDF)

# 22...

In [ ]:
# ============================================================
# SECTION 22.1 — PAIRED BOOTSTRAP TEST ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

predictions_test = pd.read_csv(
    RESULTS_DIR / "predictions_test.csv"
)

# ------------------------------------------------------------
# Align Transformer and DDE predictions by video
# ------------------------------------------------------------

transformer = (
    predictions_test[
        predictions_test["model_name"] == "Temporal Transformer"
    ][
        ["video_id", "ground_truth_ef", "predicted_ef"]
    ]
    .rename(
        columns={
            "predicted_ef": "transformer_pred"
        }
    )
)

dde = (
    predictions_test[
        predictions_test["model_name"]
        == "Full DDE + reliability + adaptive fusion"
    ][
        ["video_id", "predicted_ef"]
    ]
    .rename(
        columns={
            "predicted_ef": "dde_pred"
        }
    )
)

paired = transformer.merge(
    dde,
    on="video_id",
    how="inner"
)

assert len(paired) == 1277

y = paired["ground_truth_ef"].to_numpy()
t = paired["transformer_pred"].to_numpy()
d = paired["dde_pred"].to_numpy()

# ------------------------------------------------------------
# Metric function
# Positive "improvement" means DDE is better
# ------------------------------------------------------------

def compute_metrics(y, t, d):

    t_err = t - y
    d_err = d - y

    t_mae = np.mean(np.abs(t_err))
    d_mae = np.mean(np.abs(d_err))

    t_rmse = np.sqrt(np.mean(t_err ** 2))
    d_rmse = np.sqrt(np.mean(d_err ** 2))

    t_bias = np.mean(t_err)
    d_bias = np.mean(d_err)

    return {
        "mae_improvement":
            t_mae - d_mae,

        "rmse_improvement":
            t_rmse - d_rmse,

        "signed_bias_reduction":
            t_bias - d_bias,

        "absolute_bias_reduction":
            abs(t_bias) - abs(d_bias),
    }


observed = compute_metrics(
    y, t, d
)

# ------------------------------------------------------------
# Paired bootstrap
# ------------------------------------------------------------

N_BOOT = 10000
rng = np.random.default_rng(42)

bootstrap_results = {
    key: []
    for key in observed
}

n = len(y)

for _ in range(N_BOOT):

    idx = rng.integers(
        0,
        n,
        size=n
    )

    result = compute_metrics(
        y[idx],
        t[idx],
        d[idx]
    )

    for key, value in result.items():
        bootstrap_results[key].append(value)

# ------------------------------------------------------------
# Summarize 95% confidence intervals
# ------------------------------------------------------------

summary_rows = []

for metric, observed_value in observed.items():

    values = np.asarray(
        bootstrap_results[metric]
    )

    lower = np.percentile(
        values,
        2.5
    )

    upper = np.percentile(
        values,
        97.5
    )

    probability_positive = np.mean(
        values > 0
    )

    summary_rows.append({
        "metric": metric,
        "observed_improvement": observed_value,
        "ci_95_lower": lower,
        "ci_95_upper": upper,
        "bootstrap_probability_improvement":
            probability_positive,
    })

bootstrap_summary = pd.DataFrame(
    summary_rows
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

BOOTSTRAP_PATH = (
    RESULTS_DIR
    / "test_paired_bootstrap_summary.csv"
)

bootstrap_summary.to_csv(
    BOOTSTRAP_PATH,
    index=False
)

print("=" * 90)
print("PAIRED BOOTSTRAP — OFFICIAL TEST SET")
print("=" * 90)
print("Videos:", len(paired))
print("Bootstrap samples:", N_BOOT)
print()

print(
    bootstrap_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print()
print("Interpretation:")
print("Positive improvement values favor the Full DDE.")
print(
    "A 95% CI entirely above zero provides evidence "
    "that the corresponding improvement is consistent "
    "across bootstrap resamples."
)

print()
print("Saved to:")
print(BOOTSTRAP_PATH)

In [ ]:
# ============================================================
# SECTION 22.2 — SELECT FAILURE-CASE CANDIDATES
# ============================================================

import pandas as pd
import numpy as np

predictions_test = pd.read_csv(
    RESULTS_DIR / "predictions_test.csv"
)

# ------------------------------------------------------------
# Put Transformer and DDE predictions on one row per video
# ------------------------------------------------------------

transformer = (
    predictions_test[
        predictions_test["model_name"] == "Temporal Transformer"
    ][
        [
            "video_id",
            "ground_truth_ef",
            "predicted_ef",
            "absolute_error",
        ]
    ]
    .rename(
        columns={
            "predicted_ef": "transformer_pred",
            "absolute_error": "transformer_abs_error",
        }
    )
)

dde = (
    predictions_test[
        predictions_test["model_name"]
        == "Full DDE + reliability + adaptive fusion"
    ][
        [
            "video_id",
            "predicted_ef",
            "absolute_error",
        ]
    ]
    .rename(
        columns={
            "predicted_ef": "dde_pred",
            "absolute_error": "dde_abs_error",
        }
    )
)

failure_cases = transformer.merge(
    dde,
    on="video_id",
    how="inner"
)

assert len(failure_cases) == 1277

# Positive = DDE improved relative to Transformer
failure_cases["error_improvement"] = (
    failure_cases["transformer_abs_error"]
    - failure_cases["dde_abs_error"]
)

# ------------------------------------------------------------
# Select representative cases
# ------------------------------------------------------------

top_dde_errors = (
    failure_cases
    .nlargest(10, "dde_abs_error")
    .copy()
)

best_dde_improvements = (
    failure_cases
    .nlargest(5, "error_improvement")
    .copy()
)

worst_dde_worsenings = (
    failure_cases
    .nsmallest(5, "error_improvement")
    .copy()
)

top_dde_errors["case_type"] = "High DDE error"
best_dde_improvements["case_type"] = "DDE improved most"
worst_dde_worsenings["case_type"] = "DDE worsened most"

selected_failure_cases = pd.concat(
    [
        top_dde_errors,
        best_dde_improvements,
        worst_dde_worsenings,
    ],
    ignore_index=True,
)

selected_failure_cases = (
    selected_failure_cases
    .drop_duplicates(subset=["video_id", "case_type"])
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FAILURE_CASES_PATH = (
    RESULTS_DIR / "selected_failure_cases.csv"
)

selected_failure_cases.to_csv(
    FAILURE_CASES_PATH,
    index=False
)

print("=" * 100)
print("SELECTED FAILURE / CONTRAST CASES")
print("=" * 100)

print(
    selected_failure_cases[
        [
            "case_type",
            "video_id",
            "ground_truth_ef",
            "transformer_pred",
            "dde_pred",
            "transformer_abs_error",
            "dde_abs_error",
            "error_improvement",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print()
print("Saved to:")
print(FAILURE_CASES_PATH)

In [ ]:
# ============================================================
# SECTION 22.3 — FAILURE-CASE MODEL DIAGNOSTICS
# ============================================================

import numpy as np
import pandas as pd
import torch

selected_failure_cases = pd.read_csv(
    RESULTS_DIR / "selected_failure_cases.csv"
)

# ------------------------------------------------------------
# Map TEST video IDs -> dataset indices
# ------------------------------------------------------------

test_id_to_index = {}

for idx, filename in enumerate(
    test_dataset.metadata["FileName"].astype(str)
):
    video_id = filename.replace(".avi", "")
    test_id_to_index[video_id] = idx

diagnostic_rows = []

diagnostic_model.eval()

# ------------------------------------------------------------
# Run only selected failure / contrast videos
# ------------------------------------------------------------

with torch.no_grad():

    for _, case in selected_failure_cases.iterrows():

        video_id = str(case["video_id"])
        dataset_idx = test_id_to_index[video_id]

        sample = test_dataset[dataset_idx]

        video = sample["video"].unsqueeze(0).to(device)
        times = sample["sample_times"].unsqueeze(0).to(device)

        output = diagnostic_model(
            video,
            times
        )

        reliability = (
            output["reliability"][0]
            .detach()
            .cpu()
            .numpy()
        )

        Q_mean = float(reliability[:, 0].mean())
        M_mean = float(reliability[:, 1].mean())
        B_mean = float(reliability[:, 2].mean())
        C_mean = float(reliability[:, 3].mean())

        reliability_mean = float(
            reliability.mean()
        )

        delay_seconds = float(
            output["delay_seconds"]
            .detach()
            .cpu()
            .reshape(-1)[0]
        )

        fusion_gate = float(
            output["fusion_gate"]
            .detach()
            .cpu()
            .reshape(-1)[0]
        )

        ef_correction = float(
            output["correction"]
            .detach()
            .cpu()
            .reshape(-1)[0]
        )

        diagnostic_rows.append({
            "case_type":
                case["case_type"],

            "video_id":
                video_id,

            "ground_truth_ef":
                case["ground_truth_ef"],

            "transformer_pred":
                case["transformer_pred"],

            "dde_pred":
                case["dde_pred"],

            "transformer_abs_error":
                case["transformer_abs_error"],

            "dde_abs_error":
                case["dde_abs_error"],

            "error_improvement":
                case["error_improvement"],

            "Q_mean":
                Q_mean,

            "M_mean":
                M_mean,

            "B_mean":
                B_mean,

            "C_mean":
                C_mean,

            "reliability_mean":
                reliability_mean,

            "delay_seconds":
                delay_seconds,

            "fusion_gate":
                fusion_gate,

            "ef_correction":
                ef_correction,
        })

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

failure_diagnostics = pd.DataFrame(
    diagnostic_rows
)

FAILURE_DIAGNOSTICS_PATH = (
    RESULTS_DIR
    / "failure_case_diagnostics.csv"
)

failure_diagnostics.to_csv(
    FAILURE_DIAGNOSTICS_PATH,
    index=False
)

print("=" * 120)
print("FAILURE-CASE MODEL DIAGNOSTICS")
print("=" * 120)

print(
    failure_diagnostics[
        [
            "case_type",
            "video_id",
            "dde_abs_error",
            "error_improvement",
            "Q_mean",
            "M_mean",
            "B_mean",
            "C_mean",
            "reliability_mean",
            "delay_seconds",
            "fusion_gate",
            "ef_correction",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

print()
print("Saved to:")
print(FAILURE_DIAGNOSTICS_PATH)

In [ ]:
# ============================================================
# SECTION 22.4A — INSPECT ROI + DATASET INTERFACE
# ============================================================

sample = test_dataset[0]

print("=" * 70)
print("TEST DATASET SAMPLE KEYS")
print("=" * 70)

for key, value in sample.items():
    if hasattr(value, "shape"):
        print(f"{key:20s} shape={tuple(value.shape)}")
    else:
        print(f"{key:20s} {type(value).__name__}: {value}")

print("\n" + "=" * 70)
print("ROI PROCESSOR")
print("=" * 70)

print("Class:")
print(type(roi_detector))

print("\nPublic attributes / methods:")
print([
    name
    for name in dir(roi_detector)
    if not name.startswith("_")
])

In [ ]:
# ============================================================
# SECTION 22.4B — INSPECT ROI METHOD STRUCTURE
# ============================================================

import dis

print("ROI object dictionary:")
print(roi_detector.__dict__)

print("\nMotionBasedCardiacROI methods:")
print(MotionBasedCardiacROI.__dict__.keys())

print("\n__call__ bytecode:")
dis.dis(MotionBasedCardiacROI.__call__)

In [ ]:
# ============================================================
# SECTION 22.4C — VISUALIZE TOP HIGH-ERROR ROI CASES
# ============================================================

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ------------------------------------------------------------
# Select six largest DDE errors
# ------------------------------------------------------------

cases_to_view = (
    failure_diagnostics[
        failure_diagnostics["case_type"] == "High DDE error"
    ]
    .nlargest(6, "dde_abs_error")
    .reset_index(drop=True)
)

fig, axes = plt.subplots(
    len(cases_to_view),
    2,
    figsize=(9, 3.2 * len(cases_to_view))
)

for row_idx, case in cases_to_view.iterrows():

    video_id = str(case["video_id"])
    dataset_idx = test_id_to_index[video_id]

    sample = test_dataset[dataset_idx]

    # ROI is stored as (x1, y1, x2, y2)
    bbox = (
        sample["roi_bbox"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    x1, y1, x2, y2 = bbox

    # Use the middle sampled frame
    frame_indices = (
        sample["frame_indices"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    middle_idx = int(
        frame_indices[len(frame_indices) // 2]
    )

    # --------------------------------------------------------
    # Read original frame from AVI
    # --------------------------------------------------------

    video_path = VIDEO_DIR / f"{video_id}.avi"

    cap = cv2.VideoCapture(str(video_path))
    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        middle_idx
    )

    ok, frame_bgr = cap.read()
    cap.release()

    if not ok:
        raise RuntimeError(
            f"Could not read frame {middle_idx} "
            f"from {video_id}"
        )

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    cropped_rgb = frame_rgb[
        y1:y2,
        x1:x2
    ]

    # --------------------------------------------------------
    # Left — original frame + ROI
    # --------------------------------------------------------

    ax = axes[row_idx, 0]

    ax.imshow(frame_rgb)

    rect = Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)

    ax.set_title(
        f"{video_id}\n"
        f"GT={case['ground_truth_ef']:.1f}% | "
        f"DDE={case['dde_pred']:.1f}% | "
        f"Error={case['dde_abs_error']:.1f}"
    )

    ax.axis("off")

    # --------------------------------------------------------
    # Right — resulting ROI crop
    # --------------------------------------------------------

    ax = axes[row_idx, 1]

    ax.imshow(cropped_rgb)

    ax.set_title(
        "Motion-based ROI crop\n"
        f"Q={case['Q_mean']:.2f}, "
        f"M={case['M_mean']:.2f}, "
        f"B={case['B_mean']:.2f}, "
        f"C={case['C_mean']:.2f}"
    )

    ax.axis("off")


axes[0, 0].text(
    -0.10,
    1.18,
    "Original frame + detected ROI",
    transform=axes[0, 0].transAxes,
    fontsize=12,
    fontweight="bold"
)

axes[0, 1].text(
    -0.02,
    1.18,
    "ROI supplied to model",
    transform=axes[0, 1].transAxes,
    fontsize=12,
    fontweight="bold"
)

fig.suptitle(
    "Visual Inspection of Highest-Error TEST Cases",
    fontsize=14
)

plt.tight_layout()

FAILURE_FIG_PATH = (
    FIGURES_DIR
    / "failure_case_roi_inspection.png"
)

fig.savefig(
    FAILURE_FIG_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(FAILURE_FIG_PATH)

In [ ]:
# ============================================================
# SECTION 22.4D — SAVE HIGH-ERROR ROI MANUAL REVIEW
# ============================================================

import pandas as pd

roi_high_error_review = pd.DataFrame([
    {
        "video_id": "0X29044407543B4ED0",
        "roi_status": "partially_adequate",
        "comment": "ROI somewhat tight but main cardiac region remains visible."
    },
    {
        "video_id": "0X5BFC2EC0D445EA65",
        "roi_status": "adequate",
        "comment": "ROI broadly preserves the visible cardiac region."
    },
    {
        "video_id": "0X211D307253ACBEE7",
        "roi_status": "adequate",
        "comment": "ROI broadly preserves the visible cardiac region."
    },
    {
        "video_id": "0XB72CF150368BAB2",
        "roi_status": "adequate",
        "comment": "ROI broadly preserves the visible cardiac region."
    },
    {
        "video_id": "0X1E433E7966FD7332",
        "roi_status": "failure",
        "comment": "ROI is too narrow/off-center and excludes substantial cardiac anatomy."
    },
    {
        "video_id": "0XABBDDF3F072EB3F",
        "roi_status": "failure",
        "comment": "ROI is too narrow and excludes substantial cardiac anatomy."
    },
])

ROI_HIGH_ERROR_REVIEW_PATH = (
    RESULTS_DIR / "roi_high_error_manual_review.csv"
)

roi_high_error_review.to_csv(
    ROI_HIGH_ERROR_REVIEW_PATH,
    index=False
)

print(roi_high_error_review.to_string(index=False))

print("\nSaved to:")
print(ROI_HIGH_ERROR_REVIEW_PATH)

In [ ]:
# ============================================================
# SECTION 22.5A — SAMPLE 100 VAL VIDEOS FOR ROI AUDIT
# ============================================================

import numpy as np
import pandas as pd

ROI_AUDIT_N = 100
ROI_AUDIT_SEED = 42

rng = np.random.default_rng(
    ROI_AUDIT_SEED
)

all_val_indices = np.arange(
    len(val_dataset)
)

audit_indices = rng.choice(
    all_val_indices,
    size=ROI_AUDIT_N,
    replace=False
)

audit_rows = []

for audit_number, dataset_idx in enumerate(
    audit_indices,
    start=1
):

    sample = val_dataset[int(dataset_idx)]

    bbox = (
        sample["roi_bbox"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    x1, y1, x2, y2 = bbox

    audit_rows.append({
        "audit_number":
            audit_number,

        "dataset_index":
            int(dataset_idx),

        "video_id":
            str(sample["filename"]),

        "x1": int(x1),
        "y1": int(y1),
        "x2": int(x2),
        "y2": int(y2),

        # Filled manually during review
        "roi_status": "",
        "comment": "",
    })

roi_audit_manifest = pd.DataFrame(
    audit_rows
)

ROI_AUDIT_MANIFEST_PATH = (
    RESULTS_DIR
    / "roi_audit_100_val_manifest.csv"
)

roi_audit_manifest.to_csv(
    ROI_AUDIT_MANIFEST_PATH,
    index=False
)

print("=" * 70)
print("VAL ROI AUDIT SAMPLE CREATED")
print("=" * 70)
print("Videos selected:", len(roi_audit_manifest))
print("Random seed:", ROI_AUDIT_SEED)
print()
print(
    roi_audit_manifest[
        [
            "audit_number",
            "dataset_index",
            "video_id",
        ]
    ].head(20).to_string(index=False)
)

print()
print("Saved to:")
print(ROI_AUDIT_MANIFEST_PATH)

In [ ]:
# ============================================================
# SECTION 22.5B — ROI AUDIT VISUALIZATION: BATCH 1 OF 5
# ============================================================

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

# First 20 videos
batch = roi_audit_manifest.iloc[0:20].copy()

fig, axes = plt.subplots(
    5,
    4,
    figsize=(16, 18)
)

axes = axes.flatten()

for ax, (_, row) in zip(
    axes,
    batch.iterrows()
):

    dataset_idx = int(row["dataset_index"])
    audit_number = int(row["audit_number"])
    video_id = str(row["video_id"])

    sample = val_dataset[dataset_idx]

    # ROI bbox: (x1, y1, x2, y2)
    x1, y1, x2, y2 = (
        sample["roi_bbox"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_indices = (
        sample["frame_indices"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    # Middle sampled frame
    frame_idx = int(
        frame_indices[len(frame_indices) // 2]
    )

    video_path = VIDEO_DIR / f"{video_id}.avi"

    cap = cv2.VideoCapture(str(video_path))
    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_idx
    )

    ok, frame_bgr = cap.read()
    cap.release()

    if not ok:
        ax.text(
            0.5,
            0.5,
            "FRAME READ FAILED",
            ha="center",
            va="center"
        )
        ax.axis("off")
        continue

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(frame_rgb)

    rect = Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)

    ax.set_title(
        f"#{audit_number}  {video_id}",
        fontsize=9
    )

    ax.axis("off")


fig.suptitle(
    "Random VAL ROI Audit — Batch 1 of 5 "
    "(Videos 1–20)",
    fontsize=15
)

plt.tight_layout()

ROI_BATCH1_PATH = (
    FIGURES_DIR
    / "roi_audit_batch_01.png"
)

fig.savefig(
    ROI_BATCH1_PATH,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(ROI_BATCH1_PATH)

In [ ]:
# ============================================================
# SECTION 22.5C — SAVE ROI AUDIT BATCH 1 LABELS
# ============================================================

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

# Make manual-review columns explicitly text
roi_audit_manifest["roi_status"] = (
    roi_audit_manifest["roi_status"]
    .fillna("")
    .astype("object")
)

roi_audit_manifest["comment"] = (
    roi_audit_manifest["comment"]
    .fillna("")
    .astype("object")
)

batch1_labels = {
    1:  ("adequate", ""),
    2:  ("adequate", ""),
    3:  ("partially_adequate", "ROI appears somewhat tight/off-center."),
    4:  ("adequate", ""),
    5:  ("adequate", ""),
    6:  ("adequate", ""),
    7:  ("adequate", ""),
    8:  ("partially_adequate", "ROI appears somewhat tight."),
    9:  ("adequate", ""),
    10: ("adequate", ""),
    11: ("partially_adequate", "ROI excludes some peripheral cardiac anatomy."),
    12: ("adequate", ""),
    13: ("partially_adequate", "ROI is noticeably asymmetric/tight."),
    14: ("partially_adequate", "ROI appears vertically tight."),
    15: ("adequate", ""),
    16: ("failure", "ROI substantially excludes visible cardiac anatomy."),
    17: ("adequate", ""),
    18: ("failure", "ROI substantially excludes visible cardiac anatomy."),
    19: ("adequate", ""),
    20: ("adequate", ""),
}

for audit_number, (status, comment) in batch1_labels.items():

    mask = (
        roi_audit_manifest["audit_number"]
        == audit_number
    )

    roi_audit_manifest.loc[
        mask, "roi_status"
    ] = status

    roi_audit_manifest.loc[
        mask, "comment"
    ] = comment

roi_audit_manifest.to_csv(
    ROI_AUDIT_MANIFEST_PATH,
    index=False
)

print(
    roi_audit_manifest.iloc[:20][
        [
            "audit_number",
            "video_id",
            "roi_status",
            "comment",
        ]
    ].to_string(index=False)
)

print("\nBatch 1 counts:")
print(
    roi_audit_manifest.iloc[:20][
        "roi_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# SECTION 22.5D — ROI AUDIT VISUALIZATION: BATCH 2 OF 5
# ============================================================

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

# Videos 21–40
batch = roi_audit_manifest.iloc[20:40].copy()

fig, axes = plt.subplots(
    5,
    4,
    figsize=(16, 18)
)

axes = axes.flatten()

for ax, (_, row) in zip(
    axes,
    batch.iterrows()
):

    dataset_idx = int(row["dataset_index"])
    audit_number = int(row["audit_number"])
    video_id = str(row["video_id"])

    sample = val_dataset[dataset_idx]

    x1, y1, x2, y2 = (
        sample["roi_bbox"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_indices = (
        sample["frame_indices"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_idx = int(
        frame_indices[len(frame_indices) // 2]
    )

    video_path = VIDEO_DIR / f"{video_id}.avi"

    cap = cv2.VideoCapture(str(video_path))
    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_idx
    )

    ok, frame_bgr = cap.read()
    cap.release()

    if not ok:
        ax.text(
            0.5,
            0.5,
            "FRAME READ FAILED",
            ha="center",
            va="center"
        )
        ax.axis("off")
        continue

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(frame_rgb)

    rect = Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)

    ax.set_title(
        f"#{audit_number}  {video_id}",
        fontsize=9
    )

    ax.axis("off")


fig.suptitle(
    "Random VAL ROI Audit — Batch 2 of 5 "
    "(Videos 21–40)",
    fontsize=15
)

plt.tight_layout()

ROI_BATCH2_PATH = (
    FIGURES_DIR
    / "roi_audit_batch_02.png"
)

fig.savefig(
    ROI_BATCH2_PATH,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(ROI_BATCH2_PATH)

In [ ]:
# ============================================================
# SECTION 22.5E — SAVE ROI AUDIT BATCH 2 LABELS
# ============================================================

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

roi_audit_manifest["roi_status"] = (
    roi_audit_manifest["roi_status"]
    .fillna("")
    .astype("object")
)

roi_audit_manifest["comment"] = (
    roi_audit_manifest["comment"]
    .fillna("")
    .astype("object")
)

batch2_labels = {
    21: ("adequate", ""),
    22: ("adequate", ""),
    23: ("partially_adequate", "ROI appears slightly tight/off-center."),
    24: ("adequate", ""),
    25: ("adequate", ""),
    26: ("adequate", ""),
    27: ("adequate", ""),
    28: ("partially_adequate", "ROI is usable but somewhat tight/unusual in appearance."),
    29: ("adequate", ""),
    30: ("adequate", ""),
    31: ("partially_adequate", "ROI appears somewhat tight around the visible cardiac region."),
    32: ("adequate", ""),
    33: ("adequate", ""),
    34: ("adequate", ""),
    35: ("partially_adequate", "ROI appears somewhat tight and low-contrast."),
    36: ("adequate", ""),
    37: ("adequate", ""),
    38: ("adequate", ""),
    39: ("adequate", ""),
    40: ("adequate", ""),
}

for audit_number, (status, comment) in batch2_labels.items():

    mask = (
        roi_audit_manifest["audit_number"]
        == audit_number
    )

    roi_audit_manifest.loc[
        mask, "roi_status"
    ] = status

    roi_audit_manifest.loc[
        mask, "comment"
    ] = comment

roi_audit_manifest.to_csv(
    ROI_AUDIT_MANIFEST_PATH,
    index=False
)

print(
    roi_audit_manifest.iloc[20:40][
        [
            "audit_number",
            "video_id",
            "roi_status",
            "comment",
        ]
    ].to_string(index=False)
)

print("\nBatch 2 counts:")
print(
    roi_audit_manifest.iloc[20:40][
        "roi_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# SECTION 22.5F — ROI AUDIT VISUALIZATION: BATCH 3 OF 5
# ============================================================

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

# Videos 41–60
batch = roi_audit_manifest.iloc[40:60].copy()

fig, axes = plt.subplots(
    5,
    4,
    figsize=(16, 18)
)

axes = axes.flatten()

for ax, (_, row) in zip(
    axes,
    batch.iterrows()
):

    dataset_idx = int(row["dataset_index"])
    audit_number = int(row["audit_number"])
    video_id = str(row["video_id"])

    sample = val_dataset[dataset_idx]

    x1, y1, x2, y2 = (
        sample["roi_bbox"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_indices = (
        sample["frame_indices"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_idx = int(
        frame_indices[len(frame_indices) // 2]
    )

    video_path = VIDEO_DIR / f"{video_id}.avi"

    cap = cv2.VideoCapture(str(video_path))
    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_idx
    )

    ok, frame_bgr = cap.read()
    cap.release()

    if not ok:
        ax.text(
            0.5,
            0.5,
            "FRAME READ FAILED",
            ha="center",
            va="center"
        )
        ax.axis("off")
        continue

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(frame_rgb)

    rect = Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)

    ax.set_title(
        f"#{audit_number}  {video_id}",
        fontsize=9
    )

    ax.axis("off")


fig.suptitle(
    "Random VAL ROI Audit — Batch 3 of 5 "
    "(Videos 41–60)",
    fontsize=15
)

plt.tight_layout()

ROI_BATCH3_PATH = (
    FIGURES_DIR
    / "roi_audit_batch_03.png"
)

fig.savefig(
    ROI_BATCH3_PATH,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(ROI_BATCH3_PATH)

In [ ]:
# ============================================================
# SECTION 22.5G — SAVE ROI AUDIT BATCH 3 LABELS
# ============================================================

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

roi_audit_manifest["roi_status"] = (
    roi_audit_manifest["roi_status"]
    .fillna("")
    .astype("object")
)

roi_audit_manifest["comment"] = (
    roi_audit_manifest["comment"]
    .fillna("")
    .astype("object")
)

batch3_labels = {
    41: ("adequate", ""),
    42: ("partially_adequate", "ROI excludes some peripheral anatomy on the right."),
    43: ("partially_adequate", "ROI appears vertically tight near the apex."),
    44: ("adequate", ""),
    45: ("adequate", ""),
    46: ("partially_adequate", "ROI appears somewhat vertically tight."),
    47: ("adequate", ""),
    48: ("partially_adequate", "ROI excludes some lateral cardiac anatomy."),
    49: ("adequate", ""),
    50: ("adequate", ""),
    51: ("adequate", ""),
    52: ("partially_adequate", "ROI is somewhat narrow and excludes lateral anatomy."),
    53: ("adequate", ""),
    54: ("partially_adequate", "ROI is narrow but retains the main visible cardiac region."),
    55: ("adequate", ""),
    56: ("adequate", ""),
    57: ("adequate", ""),
    58: ("adequate", ""),
    59: ("adequate", ""),
    60: ("partially_adequate", "ROI is slightly tight near the superior/right margin."),
}

for audit_number, (status, comment) in batch3_labels.items():

    mask = (
        roi_audit_manifest["audit_number"]
        == audit_number
    )

    roi_audit_manifest.loc[
        mask, "roi_status"
    ] = status

    roi_audit_manifest.loc[
        mask, "comment"
    ] = comment

roi_audit_manifest.to_csv(
    ROI_AUDIT_MANIFEST_PATH,
    index=False
)

print("\nBatch 3 counts:")
print(
    roi_audit_manifest.iloc[40:60][
        "roi_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# SECTION 22.5H — ROI AUDIT VISUALIZATION: BATCH 4 OF 5
# ============================================================

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

# Videos 61–80
batch = roi_audit_manifest.iloc[60:80].copy()

fig, axes = plt.subplots(
    5,
    4,
    figsize=(16, 18)
)

axes = axes.flatten()

for ax, (_, row) in zip(
    axes,
    batch.iterrows()
):

    dataset_idx = int(row["dataset_index"])
    audit_number = int(row["audit_number"])
    video_id = str(row["video_id"])

    sample = val_dataset[dataset_idx]

    x1, y1, x2, y2 = (
        sample["roi_bbox"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_indices = (
        sample["frame_indices"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_idx = int(
        frame_indices[len(frame_indices) // 2]
    )

    video_path = VIDEO_DIR / f"{video_id}.avi"

    cap = cv2.VideoCapture(str(video_path))
    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_idx
    )

    ok, frame_bgr = cap.read()
    cap.release()

    if not ok:
        ax.text(
            0.5,
            0.5,
            "FRAME READ FAILED",
            ha="center",
            va="center"
        )
        ax.axis("off")
        continue

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(frame_rgb)

    rect = Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)

    ax.set_title(
        f"#{audit_number}  {video_id}",
        fontsize=9
    )

    ax.axis("off")


fig.suptitle(
    "Random VAL ROI Audit — Batch 4 of 5 "
    "(Videos 61–80)",
    fontsize=15
)

plt.tight_layout()

ROI_BATCH4_PATH = (
    FIGURES_DIR
    / "roi_audit_batch_04.png"
)

fig.savefig(
    ROI_BATCH4_PATH,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(ROI_BATCH4_PATH)

In [ ]:
# ============================================================
# SECTION 22.5I — SAVE ROI AUDIT BATCH 4 LABELS
# ============================================================

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

roi_audit_manifest["roi_status"] = (
    roi_audit_manifest["roi_status"]
    .fillna("")
    .astype("object")
)

roi_audit_manifest["comment"] = (
    roi_audit_manifest["comment"]
    .fillna("")
    .astype("object")
)

batch4_labels = {
    61: ("partially_adequate", "ROI is vertically tight and excludes part of the apex."),
    62: ("adequate", ""),
    63: ("partially_adequate", "ROI is mildly tight near the apex."),
    64: ("adequate", ""),
    65: ("partially_adequate", "ROI is somewhat asymmetric/tight."),
    66: ("partially_adequate", "ROI excludes some superior cardiac anatomy."),
    67: ("partially_adequate", "ROI is slightly tight on the right side."),
    68: ("partially_adequate", "ROI excludes some left-lateral anatomy."),
    69: ("adequate", ""),
    70: ("partially_adequate", "ROI is vertically tight near the apex."),
    71: ("adequate", ""),
    72: ("adequate", ""),
    73: ("partially_adequate", "ROI is slightly tight laterally."),
    74: ("adequate", ""),
    75: ("partially_adequate", "ROI is somewhat narrow around the cardiac region."),
    76: ("adequate", ""),
    77: ("partially_adequate", "ROI excludes some superior anatomy."),
    78: ("partially_adequate", "ROI is mildly vertically tight."),
    79: ("partially_adequate", "ROI is somewhat tight on the left side."),
    80: ("adequate", ""),
}

for audit_number, (status, comment) in batch4_labels.items():

    mask = (
        roi_audit_manifest["audit_number"]
        == audit_number
    )

    roi_audit_manifest.loc[
        mask, "roi_status"
    ] = status

    roi_audit_manifest.loc[
        mask, "comment"
    ] = comment

roi_audit_manifest.to_csv(
    ROI_AUDIT_MANIFEST_PATH,
    index=False
)

print("\nBatch 4 counts:")
print(
    roi_audit_manifest.iloc[60:80][
        "roi_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# SECTION 22.5J — ROI AUDIT VISUALIZATION: BATCH 5 OF 5
# ============================================================

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

# Videos 81–100
batch = roi_audit_manifest.iloc[80:100].copy()

fig, axes = plt.subplots(
    5,
    4,
    figsize=(16, 18)
)

axes = axes.flatten()

for ax, (_, row) in zip(
    axes,
    batch.iterrows()
):

    dataset_idx = int(row["dataset_index"])
    audit_number = int(row["audit_number"])
    video_id = str(row["video_id"])

    sample = val_dataset[dataset_idx]

    x1, y1, x2, y2 = (
        sample["roi_bbox"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_indices = (
        sample["frame_indices"]
        .detach()
        .cpu()
        .numpy()
        .astype(int)
    )

    frame_idx = int(
        frame_indices[len(frame_indices) // 2]
    )

    video_path = VIDEO_DIR / f"{video_id}.avi"

    cap = cv2.VideoCapture(str(video_path))

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_idx
    )

    ok, frame_bgr = cap.read()
    cap.release()

    if not ok:

        ax.text(
            0.5,
            0.5,
            "FRAME READ FAILED",
            ha="center",
            va="center"
        )

        ax.axis("off")
        continue

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(frame_rgb)

    rect = Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=2
    )

    ax.add_patch(rect)

    ax.set_title(
        f"#{audit_number}  {video_id}",
        fontsize=9
    )

    ax.axis("off")


fig.suptitle(
    "Random VAL ROI Audit — Batch 5 of 5 "
    "(Videos 81–100)",
    fontsize=15
)

plt.tight_layout()

ROI_BATCH5_PATH = (
    FIGURES_DIR
    / "roi_audit_batch_05.png"
)

fig.savefig(
    ROI_BATCH5_PATH,
    dpi=250,
    bbox_inches="tight"
)

plt.show()

print("Saved to:")
print(ROI_BATCH5_PATH)

In [ ]:
# ============================================================
# SECTION 22.5K — SAVE ROI AUDIT BATCH 5 LABELS
# ============================================================

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

roi_audit_manifest["roi_status"] = (
    roi_audit_manifest["roi_status"]
    .fillna("")
    .astype("object")
)

roi_audit_manifest["comment"] = (
    roi_audit_manifest["comment"]
    .fillna("")
    .astype("object")
)

batch5_labels = {
    81: ("failure", "ROI excludes a substantial superior/apical portion of the cardiac anatomy."),
    82: ("partially_adequate", "ROI appears somewhat narrow/laterally restricted."),
    83: ("partially_adequate", "ROI is somewhat tight near the superior portion."),
    84: ("adequate", ""),
    85: ("adequate", ""),
    86: ("adequate", ""),
    87: ("adequate", ""),
    88: ("failure", "ROI excludes a substantial superior/apical portion of the cardiac anatomy."),
    89: ("partially_adequate", "ROI is somewhat tight laterally."),
    90: ("partially_adequate", "ROI is mildly vertically tight."),
    91: ("partially_adequate", "ROI excludes some lateral cardiac anatomy."),
    92: ("adequate", ""),
    93: ("partially_adequate", "ROI excludes some left-lateral anatomy."),
    94: ("adequate", ""),
    95: ("partially_adequate", "ROI is somewhat narrow around the cardiac region."),
    96: ("failure", "ROI excludes a substantial superior portion of the visible cardiac anatomy."),
    97: ("adequate", ""),
    98: ("adequate", ""),
    99: ("partially_adequate", "ROI is somewhat tight on the right side."),
    100: ("partially_adequate", "ROI is mildly vertically tight."),
}

for audit_number, (status, comment) in batch5_labels.items():

    mask = (
        roi_audit_manifest["audit_number"]
        == audit_number
    )

    roi_audit_manifest.loc[
        mask, "roi_status"
    ] = status

    roi_audit_manifest.loc[
        mask, "comment"
    ] = comment

roi_audit_manifest.to_csv(
    ROI_AUDIT_MANIFEST_PATH,
    index=False
)

print("\nBatch 5 counts:")
print(
    roi_audit_manifest.iloc[80:100][
        "roi_status"
    ].value_counts()
)

In [ ]:
# ============================================================
# SECTION 22.5L — FINAL 100-VIDEO ROI AUDIT SUMMARY
# ============================================================

import pandas as pd

roi_audit_manifest = pd.read_csv(
    RESULTS_DIR / "roi_audit_100_val_manifest.csv"
)

# Verify every video received a label
assert len(roi_audit_manifest) == 100
assert roi_audit_manifest["roi_status"].notna().all()
assert (roi_audit_manifest["roi_status"] != "").all()

# ------------------------------------------------------------
# Counts + percentages
# ------------------------------------------------------------

summary = (
    roi_audit_manifest["roi_status"]
    .value_counts()
    .reindex(
        [
            "adequate",
            "partially_adequate",
            "failure",
        ],
        fill_value=0
    )
    .rename_axis("roi_status")
    .reset_index(name="count")
)

summary["percentage"] = (
    100
    * summary["count"]
    / len(roi_audit_manifest)
)

ROI_AUDIT_SUMMARY_PATH = (
    RESULTS_DIR
    / "roi_audit_100_val_summary.csv"
)

summary.to_csv(
    ROI_AUDIT_SUMMARY_PATH,
    index=False
)

print("=" * 70)
print("FINAL 100-VIDEO VAL ROI AUDIT")
print("=" * 70)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.1f}"
    )
)

print()
print(
    "Strict ROI failure rate:",
    f"{summary.loc[summary['roi_status']=='failure', 'percentage'].iloc[0]:.1f}%"
)

print(
    "Adequate or partially adequate:",
    f"{summary.loc[summary['roi_status']!='failure', 'percentage'].sum():.1f}%"
)

print()
print(
    "NOTE: Manual visual audit of one representative "
    "middle frame per randomly selected VAL video."
)

print("\nSaved to:")
print(ROI_AUDIT_SUMMARY_PATH)

In [ ]:
# ============================================================
# SECTION 22.6 — DDE DERIVATIVE-MAGNITUDE DIAGNOSTIC
# ============================================================

import json
import numpy as np
import torch

diagnostic_model.eval()

DERIVATIVE_DIAGNOSTIC_N = 16

derivative_norms = []
latent_norms = []
nonfinite_count = 0

with torch.no_grad():

    for idx in range(DERIVATIVE_DIAGNOSTIC_N):

        sample = val_dataset[idx]

        video = (
            sample["video"]
            .unsqueeze(0)
            .to(device)
        )

        times = (
            sample["sample_times"]
            .unsqueeze(0)
            .to(device)
        )

        output = diagnostic_model(
            video,
            times
        )

        # [1, T, latent_dim]
        z = output["dde_sequence"]

        # ----------------------------------------------------
        # Latent-state norms
        # ----------------------------------------------------

        z_norm = torch.linalg.vector_norm(
            z,
            dim=-1
        )

        latent_norms.extend(
            z_norm
            .detach()
            .cpu()
            .reshape(-1)
            .numpy()
        )

        # ----------------------------------------------------
        # Effective derivative from explicit Euler trajectory
        # dz/dt ≈ (z[t+1] - z[t]) / dt
        # ----------------------------------------------------

        dt = (
            times[:, 1:]
            - times[:, :-1]
        )

        dz = (
            z[:, 1:, :]
            - z[:, :-1, :]
        )

        derivative = (
            dz
            / dt.unsqueeze(-1)
        )

        derivative_norm = torch.linalg.vector_norm(
            derivative,
            dim=-1
        )

        derivative_norms.extend(
            derivative_norm
            .detach()
            .cpu()
            .reshape(-1)
            .numpy()
        )

        nonfinite_count += int(
            (~torch.isfinite(derivative))
            .sum()
            .item()
        )

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

derivative_norms = np.asarray(
    derivative_norms
)

latent_norms = np.asarray(
    latent_norms
)

diagnostic_summary = {
    "videos_checked":
        DERIVATIVE_DIAGNOSTIC_N,

    "derivative_norm_min":
        float(derivative_norms.min()),

    "derivative_norm_mean":
        float(derivative_norms.mean()),

    "derivative_norm_median":
        float(np.median(derivative_norms)),

    "derivative_norm_p95":
        float(np.percentile(
            derivative_norms,
            95
        )),

    "derivative_norm_max":
        float(derivative_norms.max()),

    "latent_norm_min":
        float(latent_norms.min()),

    "latent_norm_mean":
        float(latent_norms.mean()),

    "latent_norm_max":
        float(latent_norms.max()),

    "nonfinite_derivative_values":
        nonfinite_count,
}

DERIVATIVE_DIAGNOSTIC_PATH = (
    RESULTS_DIR
    / "dde_derivative_diagnostics.json"
)

with open(
    DERIVATIVE_DIAGNOSTIC_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        diagnostic_summary,
        f,
        indent=2
    )

print("=" * 75)
print("DDE DERIVATIVE-MAGNITUDE DIAGNOSTIC")
print("=" * 75)

for key, value in diagnostic_summary.items():

    if isinstance(value, float):
        print(
            f"{key:35s} {value:.6f}"
        )
    else:
        print(
            f"{key:35s} {value}"
        )

print()
print(
    "Status:",
    "PASSED"
    if nonfinite_count == 0
    else "CHECK REQUIRED"
)

print()
print("Saved to:")
print(DERIVATIVE_DIAGNOSTIC_PATH)

In [ ]:
# ============================================================
# SECTION 22.7 — TEST CORRECTION / FAILURE-MODE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

predictions_test = pd.read_csv(
    RESULTS_DIR / "predictions_test.csv"
)

# ------------------------------------------------------------
# Align Transformer and DDE predictions
# ------------------------------------------------------------

transformer = (
    predictions_test[
        predictions_test["model_name"]
        == "Temporal Transformer"
    ][
        [
            "video_id",
            "ground_truth_ef",
            "predicted_ef",
            "absolute_error",
        ]
    ]
    .rename(
        columns={
            "predicted_ef": "transformer_pred",
            "absolute_error": "transformer_abs_error",
        }
    )
)

dde = (
    predictions_test[
        predictions_test["model_name"]
        == "Full DDE + reliability + adaptive fusion"
    ][
        [
            "video_id",
            "predicted_ef",
            "absolute_error",
        ]
    ]
    .rename(
        columns={
            "predicted_ef": "dde_pred",
            "absolute_error": "dde_abs_error",
        }
    )
)

correction_analysis = transformer.merge(
    dde,
    on="video_id"
)

assert len(correction_analysis) == 1277

# ------------------------------------------------------------
# Derived quantities
# ------------------------------------------------------------

correction_analysis["dde_correction"] = (
    correction_analysis["dde_pred"]
    - correction_analysis["transformer_pred"]
)

correction_analysis["transformer_signed_error"] = (
    correction_analysis["transformer_pred"]
    - correction_analysis["ground_truth_ef"]
)

correction_analysis["dde_signed_error"] = (
    correction_analysis["dde_pred"]
    - correction_analysis["ground_truth_ef"]
)

# Positive = DDE improved absolute error
correction_analysis["error_improvement"] = (
    correction_analysis["transformer_abs_error"]
    - correction_analysis["dde_abs_error"]
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

correction = correction_analysis[
    "dde_correction"
].to_numpy()

improvement = correction_analysis[
    "error_improvement"
].to_numpy()

baseline_error = correction_analysis[
    "transformer_signed_error"
].to_numpy()

ground_truth = correction_analysis[
    "ground_truth_ef"
].to_numpy()

summary = {
    "n_test_videos":
        len(correction_analysis),

    "mean_dde_correction":
        float(correction.mean()),

    "median_dde_correction":
        float(np.median(correction)),

    "min_dde_correction":
        float(correction.min()),

    "max_dde_correction":
        float(correction.max()),

    "percent_negative_corrections":
        float(100 * np.mean(correction < 0)),

    "percent_positive_corrections":
        float(100 * np.mean(correction > 0)),

    "percent_videos_dde_improved":
        float(100 * np.mean(improvement > 0)),

    "percent_videos_dde_worsened":
        float(100 * np.mean(improvement < 0)),

    "percent_videos_equal":
        float(100 * np.mean(improvement == 0)),

    "corr_correction_vs_baseline_signed_error":
        float(np.corrcoef(
            correction,
            baseline_error
        )[0, 1]),

    "corr_correction_vs_ground_truth_ef":
        float(np.corrcoef(
            correction,
            ground_truth
        )[0, 1]),

    "corr_correction_vs_error_improvement":
        float(np.corrcoef(
            correction,
            improvement
        )[0, 1]),
}

summary_df = pd.DataFrame(
    [summary]
)

CORRECTION_SUMMARY_PATH = (
    RESULTS_DIR
    / "test_dde_correction_analysis.csv"
)

summary_df.to_csv(
    CORRECTION_SUMMARY_PATH,
    index=False
)

print("=" * 80)
print("TEST DDE CORRECTION / FAILURE-MODE ANALYSIS")
print("=" * 80)

for key, value in summary.items():

    if isinstance(value, float):
        print(f"{key:45s} {value:.4f}")
    else:
        print(f"{key:45s} {value}")

print()
print("Saved to:")
print(CORRECTION_SUMMARY_PATH)

# 23...

23.1 Group A: Deep learning for echocardiography and EF estimation

1. D. Ouyang et al., “Video-based AI for beat-to-beat assessment of cardiac function,” Nature, 2020.
   DOI: 10.1038/s41586-020-2145-8.
2. R. Muhtaseb and M. Yaqub, “EchoCoTr: Estimation of the Left Ventricular Ejection Fraction from Spatiotemporal Echocardiography,” MICCAI 2022.
   DOI: 10.1007/978-3-031-16440-8_36.
3. F. A. Maani, N. Saeed, A. Matsun, and M. Yaqub, “CoReEcho: Continuous Representation Learning for 2D+time Echocardiography Analysis,” MICCAI 2024.
   DOI: 10.1007/978-3-031-72083-3_55.
4. W. Ali et al., “EFNet: Estimation of left ventricular ejection fraction from cardiac ultrasound videos using deep learning,” PeerJ Computer Science, 2025.
   DOI: 10.7717/peerj-cs.2506.
5. T. Akan et al., “ViViEchoformer: Deep Video Regressor Predicting Ejection Fraction,” Journal of Imaging Informatics in Medicine, 2025.
   DOI: 10.1007/s10278-024-01336-y.


23.2 Group B: Temporal modeling of cardiac ultrasound

1. H. Reynaud et al., “Ultrasound Video Transformers for Cardiac Ejection Fraction Estimation,” MICCAI, 2021.
   DOI: 10.1007/978-3-030-87231-1_48.
2. R. Muhtaseb and M. Yaqub, “EchoCoTr: Estimation of the Left Ventricular Ejection Fraction from Spatiotemporal Echocardiography,” MICCAI, 2022.
3. J. Lin et al., “Dynamic-Guided Spatiotemporal Attention for Echocardiography Video Segmentation,” IEEE Transactions on Medical Imaging, 2024.
   DOI: 10.1109/TMI.2024.3403687.
4. SemanticST, “Semantics-enhanced Spatio-Temporal Modeling for Ejection Fraction Estimation in Echocardiography,” IEEE Journal of Biomedical and Health Informatics, 2026.
   DOI: 10.1109/JBHI.2026.3703643.


23.3 Group C: Reliability, uncertainty, and quality-aware echocardiography

1. Z. Liao et al., “On Modelling Label Uncertainty in Deep Neural Networks: Automatic Estimation of Intra-Observer Variability in 2D Echocardiography Quality Assessment,” IEEE Transactions on Medical Imaging, 2020.
   DOI: 10.1109/TMI.2019.2959209.
2. “Automated assessment of transthoracic echocardiogram image quality using deep neural networks,” Intelligent Medicine, 2023.
   DOI: 10.1016/j.imed.2022.08.001.
3. G. Van De Vyver et al., “Regional Image Quality Scoring for 2-D Echocardiography Using Deep Learning,” Ultrasound in Medicine & Biology, 2025.
   DOI: 10.1016/j.ultrasmedbio.2024.12.008.


23.4 Group D: Neural ODE/DDE and continuous-time learning

1. R. T. Q. Chen, Y. Rubanova, J. Bettencourt, and D. Duvenaud, “Neural Ordinary Differential Equations,” NeurIPS, 2018.
2. Y. Rubanova, R. T. Q. Chen, and D. Duvenaud, “Latent Ordinary Differential Equations for Irregularly-Sampled Time Series,” NeurIPS, 2019.
3. Q. Zhu, Y. Guo, and W. Lin, “Neural Delay Differential Equations,” ICLR, 2021.
4. S. Anumasa and P. K. Srijith, “Delay Differential Neural Networks,” ICMLT, 2021.


23.5 Final reference set

1. D. Ouyang et al., “Video-based AI for beat-to-beat assessment of cardiac function,” Nature, vol. 580, no. 7802, pp. 252–256, 2020, doi: 10.1038/s41586-020-2145-8. PubMed
2. H. Reynaud, A. Vlontzos, B. Hou, A. Beqiri, P. Leeson, and B. Kainz, “Ultrasound Video Transformers for Cardiac Ejection Fraction Estimation,” in MICCAI 2021, pp. 495–505, 2021, doi: 10.1007/978-3-030-87231-1_48. Springer
3. R. Muhtaseb and M. Yaqub, “EchoCoTr: Estimation of the Left Ventricular Ejection Fraction from Spatiotemporal Echocardiography,” in MICCAI 2022, pp. 370–379, 2022, doi: 10.1007/978-3-031-16440-8_36. Springer
4. Y. Zeng et al., “MAEF-Net: Multi-attention efficient feature fusion network for left ventricular segmentation and quantitative analysis in two-dimensional echocardiography,” Ultrasonics, vol. 127, Art. no. 106855, 2023, doi: 10.1016/j.ultras.2022.106855. PubMed
5. F. A. Maani, N. Saeed, A. Matsun, and M. Yaqub, “CoReEcho: Continuous Representation Learning for 2D+time Echocardiography Analysis,” in MICCAI 2024, pp. 591–601, 2024, doi: 10.1007/978-3-031-72083-3_55. MICCAI Society
6. T. Akan et al., “ViViEchoformer: Deep Video Regressor Predicting Ejection Fraction,” Journal of Imaging Informatics in Medicine, vol. 38, pp. 2041–2052, 2025, doi: 10.1007/s10278-024-01336-y. PubMed
7. W. Ali, W. Alsabban, M. Shahbaz, A. Al-Laith, and B. Almogadwy, “EFNet: Estimation of left ventricular ejection fraction from cardiac ultrasound videos using deep learning,” PeerJ Computer Science, vol. 11, e2506, 2025, doi: 10.7717/peerj-cs.2506. PubMed
8. Y. Zhou, J. Tian, M. Kang, and K. Kang, “Echo-SMADS: A hierarchical planning model for predicting ejection fraction using echocardiography,” Computer Methods and Programs in Biomedicine, vol. 281, Art. no. 109377, 2026, doi: 10.1016/j.cmpb.2026.109377. ScienceDirect
9. J. Lin, W. Xie, L. Kang, and H. Wu, “Dynamic-Guided Spatiotemporal Attention for Echocardiography Video Segmentation,” IEEE Transactions on Medical Imaging, vol. 43, no. 11, pp. 3843–3855, 2024, doi: 10.1109/TMI.2024.3403687. PubMed
10. D. Zheng et al., “SemanticST: Semantics-enhanced Spatio-Temporal Modeling for Ejection Fraction Estimation in Echocardiography,” IEEE Journal of Biomedical and Health Informatics, 2026, doi: 10.1109/JBHI.2026.3703643.
11. Z. Liao et al., “On Modelling Label Uncertainty in Deep Neural Networks: Automatic Estimation of Intra-Observer Variability in 2D Echocardiography Quality Assessment,” IEEE Transactions on Medical Imaging, vol. 39, no. 6, pp. 1868–1883, 2020, doi: 10.1109/TMI.2019.2959209. PubMed
12. R. B. Labs, A. Vrettos, J. Loo, and M. Zolgharni, “Automated assessment of transthoracic echocardiogram image quality using deep neural networks,” Intelligent Medicine, vol. 3, no. 3, pp. 191–199, 2023, doi: 10.1016/j.imed.2022.08.001. ScienceDirect
13. G. Van De Vyver et al., “Regional Image Quality Scoring for 2-D Echocardiography Using Deep Learning,” Ultrasound in Medicine & Biology, vol. 51, no. 4, pp. 638–649, 2025, doi: 10.1016/j.ultrasmedbio.2024.12.008. ScienceDirect
14. R. T. Q. Chen, Y. Rubanova, J. Bettencourt, and D. K. Duvenaud, “Neural Ordinary Differential Equations,” in Advances in Neural Information Processing Systems 31, 2018. NeurIPS Proceedings
15. Y. Rubanova, R. T. Q. Chen, and D. K. Duvenaud, “Latent Ordinary Differential Equations for Irregularly-Sampled Time Series,” in Advances in Neural Information Processing Systems 32, 2019. NeurIPS Proceedings
16. Q. Zhu, Y. Guo, and W. Lin, “Neural Delay Differential Equations,” in International Conference on Learning Representations (ICLR), 2021. DOI
17. S. Anumasa and P. K. Srijith, “Delay Differential Neural Networks,” in Proc. 6th International Conference on Machine Learning Technologies (ICMLT), pp. 117–121, 2021, doi: 10.1145/3468891.3468908. ResearchGate


Final Abstract

Automated estimation of left ventricular ejection fraction (EF) from echocardiography requires modeling cardiac motion across time while accounting for variability in image quality, motion consistency, structural visibility, and temporal sampling. Existing convolutional and Transformer-based approaches capture spatial and temporal patterns, but they do not explicitly combine observation reliability with delayed latent cardiac dynamics. We propose a reliability-aware Neural Delay Differential Equation (Neural DDE) framework for video-based EF estimation. The pipeline samples 32 temporally ordered frames, applies motion-based cardiac region-of-interest extraction, encodes spatial features using ConvNeXt-Tiny, and models inter-frame dependencies using a Temporal Transformer. A four-component reliability representation capturing relative image quality, motion consistency, structural-boundary consistency, and cardiac-cycle consistency conditions a physical-time Neural DDE whose dynamics depend on both current and delayed latent states. The resulting DDE correction is adaptively fused with the Transformer EF estimate. On EchoNet-Dynamic, validation MAE improved from 8.495 with a frozen ConvNeXt baseline to 7.386 after partial fine-tuning and 6.467 with the Temporal Transformer. On the untouched official test set, the full DDE achieved MAE 6.501, RMSE 8.898, and bias 0.876, compared with 6.511, 8.990, and 1.729 for the Transformer. Paired bootstrap analysis indicated no clear MAE improvement but supported consistent reductions in RMSE and systematic bias. Ablation results showed only modest incremental predictive benefit from the individual reliability, delay, and fusion components, motivating further refinement of reliability-conditioned delayed dynamics.

In [ ]:
# ============================================================
# DDE ABLATION RUNTIME RECOVERY
# ============================================================

from torch.utils.data import DataLoader, Subset

E2E_RUN_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "e2e_reproducibility_run"
)

RESULTS_DIR = PROJECT_ROOT / "results"
ABLATION_RESULTS_PATH = RESULTS_DIR / "ablation_results.csv"

DDE_BATCH_SIZE = 8
DDE_MAX_EPOCHS = 10
DDE_PATIENCE = 3
DDE_LR = 1e-4
RESIDUAL_LR = 3e-4
DDE_WEIGHT_DECAY = 1e-4

TRANSFORMER_BASELINE_MAE = 6.467345480593095
SEED = 42
USE_AMP = True

DDE_EXCLUDED_SHORT_VIDEOS = {
    "0X174A9943AB689618",
    "0X2A55659AE64722AA",
    "0X6A46321A41C98FC7",
}

dde_train_indices = [
    i
    for i, filename in enumerate(
        train_dataset.metadata["FileName"].astype(str)
    )
    if filename.replace(".avi", "")
    not in DDE_EXCLUDED_SHORT_VIDEOS
]

dde_train_dataset = Subset(
    train_dataset,
    dde_train_indices,
)

dde_val_loader = DataLoader(
    val_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("DDE TRAIN:", len(dde_train_dataset))
print("DDE VAL:", len(val_dataset))
print("VAL batches:", len(dde_val_loader))

assert len(dde_train_dataset) == 7462
assert len(val_dataset) == 1288

In [ ]:
# ============================================================
# RECOVER DDE TRAINING DATASET
# ============================================================

from torch.utils.data import Subset

DDE_EXCLUDED_SHORT_VIDEOS = {
    "0X174A9943AB689618",  # 28 frames
    "0X2A55659AE64722AA",  # 29 frames
    "0X6A46321A41C98FC7",  # 29 frames
}

dde_train_indices = [
    i
    for i, filename in enumerate(
        train_dataset.metadata["FileName"].astype(str)
    )
    if filename.replace(".avi", "")
    not in DDE_EXCLUDED_SHORT_VIDEOS
]

dde_train_dataset = Subset(
    train_dataset,
    dde_train_indices,
)

print("Original TRAIN:", len(train_dataset))
print("DDE TRAIN:", len(dde_train_dataset))
print(
    "Excluded:",
    len(train_dataset) - len(dde_train_dataset),
)

assert len(dde_train_dataset) == 7462

In [ ]:
# ============================================================
# RECOVER DDE TRAINING CONSTANTS
# ============================================================

DDE_BATCH_SIZE = 8
DDE_MAX_EPOCHS = 10
DDE_PATIENCE = 3

DDE_LR = 1e-4
RESIDUAL_LR = 3e-4
DDE_WEIGHT_DECAY = 1e-4

TRANSFORMER_BASELINE_MAE = 6.467345480593095

SEED = 42
USE_AMP = True

print("DDE training constants restored.")

In [ ]:
# ============================================================
# RECOVER DDE VALIDATION LOADER
# ============================================================

from torch.utils.data import DataLoader

DDE_BATCH_SIZE = 8

dde_val_loader = DataLoader(
    val_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("DDE VAL samples:", len(val_dataset))
print("DDE VAL batches:", len(dde_val_loader))

In [ ]:
WITHOUT_M_TRAINED_BEST_PATH = (
    E2E_RUN_DIR / "10_without_m_best_trained.pth"
)

print(WITHOUT_M_TRAINED_BEST_PATH)
print(WITHOUT_M_TRAINED_BEST_PATH.exists())

In [ ]:
# ============================================================
# SECTION 13.7A — TIMESTAMP DIAGNOSTIC
# ============================================================

frame_counts = train_dataset.metadata["NumberOfFrames"].astype(int)

print("=" * 60)
print("TIMESTAMP DIAGNOSTIC")
print("=" * 60)
print("Minimum frames:", frame_counts.min())
print("Videos with < 32 frames:", (frame_counts < NUM_FRAMES).sum())
print("Videos with = 32 frames:", (frame_counts == NUM_FRAMES).sum())
print("Videos with > 32 frames:", (frame_counts > NUM_FRAMES).sum())

bad_examples = []

for i in range(len(train_dataset)):
    sample = train_dataset[i]
    times = sample["sample_times"]

    if not torch.all(times[1:] > times[:-1]):
        bad_examples.append((
            i,
            sample["filename"],
            sample["original_num_frames"],
            sample["frame_indices"].tolist(),
            times.tolist(),
        ))

        if len(bad_examples) == 3:
            break

print("\nFirst problematic examples:")
for i, filename, n_frames, indices, times in bad_examples:
    print(f"\nIndex: {i}")
    print(f"Filename: {filename}")
    print(f"Original frames: {n_frames}")
    print(f"Frame indices: {indices}")
    print(f"Times: {[round(x, 4) for x in times]}")

print("=" * 60)

In [ ]:
# ============================================================
# SECTION 13.7B — FILTER SHORT VIDEOS FOR DDE TRAINING
# ============================================================

train_frames = train_dataset.metadata["NumberOfFrames"].astype(int)
val_frames = val_dataset.metadata["NumberOfFrames"].astype(int)

short_train = train_dataset.metadata[train_frames < NUM_FRAMES]
short_val = val_dataset.metadata[val_frames < NUM_FRAMES]

print("=" * 60)
print("SHORT VIDEO CHECK")
print("=" * 60)
print(f"TRAIN videos < {NUM_FRAMES} frames: {len(short_train)}")
print(f"VAL videos < {NUM_FRAMES} frames: {len(short_val)}")

if len(short_train):
    print("\nTRAIN exclusions:")
    print(short_train[["FileName", "NumberOfFrames", "FPS"]].to_string(index=False))

if len(short_val):
    print("\nVAL exclusions:")
    print(short_val[["FileName", "NumberOfFrames", "FPS"]].to_string(index=False))

# Exclude only videos that cannot provide 32 distinct timestamps
dde_train_indices = train_dataset.metadata.index[
    train_frames >= NUM_FRAMES
].tolist()

dde_train_dataset = Subset(train_dataset, dde_train_indices)

dde_train_loader = DataLoader(
    dde_train_dataset,
    batch_size=DDE_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=USE_AMP,
    generator=dde_train_generator,
)

print()
print(f"Original TRAIN size: {len(train_dataset)}")
print(f"DDE TRAIN size: {len(dde_train_dataset)}")
print(f"DDE TRAIN batches: {len(dde_train_loader)}")
print("=" * 60)